In [ ]:
pip install networkx

In [ ]:
pip install llama-index

In [ ]:
pip install graphrag==1.0.1

In [ ]:
pip install graspologic

In [ ]:
pip install langchain_openai

In [ ]:
pip install nest_asyncio

In [ ]:
pip install lancedb

In [ ]:
pip install sparqlwrapper

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON


In [ ]:
from llama_index.core.graph_stores.types import LabelledNode,Relation, EntityNode
from graspologic.partition import hierarchical_leiden
from llama_index.core import PropertyGraphIndex
from IPython.display import Markdown, display
from langchain_openai import  OpenAI
import os
import openai
import ast
import asyncio
import logging
import re
import time
from collections.abc import AsyncGenerator
from copy import deepcopy
from datetime import datetime, timezone
from typing import Any, Optional
from uuid import uuid4

import networkx as nx
import numpy as np
import pandas as pd
import tiktoken
from SPARQLWrapper import SPARQLWrapper, JSON

from datashaper import AsyncType, NoopVerbCallbacks, VerbCallbacks

from graphrag.config.enums import LLMType
from graphrag.config.models.summarize_descriptions_config import SummarizeDescriptionsConfig
from graphrag.index.graph.extractors.community_reports.schemas import (
    CLAIM_DESCRIPTION,
    CLAIM_DETAILS,
    CLAIM_ID,
    CLAIM_STATUS,
    CLAIM_SUBJECT,
    CLAIM_TYPE,
    COMMUNITY_ID,
    EDGE_DEGREE,
    EDGE_DESCRIPTION,
    EDGE_DETAILS,
    EDGE_ID,
    EDGE_SOURCE,
    EDGE_TARGET,
    NODE_DEGREE,
    NODE_DESCRIPTION,
    NODE_DETAILS,
    NODE_ID,
    NODE_NAME,
)
from graphrag.index.operations.cluster_graph import cluster_graph
from graphrag.index.operations.compute_edge_combined_degree import compute_edge_combined_degree
from graphrag.index.operations.create_graph import create_graph
from graphrag.index.operations.embed_graph import embed_graph
from graphrag.index.operations.layout_graph import layout_graph
from graphrag.index.operations.snapshot import snapshot
from graphrag.index.operations.summarize_communities import (
    prepare_community_reports,
    restore_community_hierarchy,
    summarize_communities,
)
from graphrag.model.community_report import CommunityReport
from graphrag.model.covariate import Covariate
from graphrag.model.entity import Entity
from graphrag.model.relationship import Relationship
from graphrag.model.text_unit import TextUnit
from graphrag.prompts.query.local_search_system_prompt import LOCAL_SEARCH_SYSTEM_PROMPT
from graphrag.query.context_builder.builders import ContextBuilderResult, LocalContextBuilder
from graphrag.query.context_builder.community_context import build_community_context
from graphrag.query.context_builder.conversation_history import ConversationHistory
from graphrag.query.context_builder.entity_extraction import (
    EntityVectorStoreKey,
    map_query_to_entities,
)
from graphrag.query.context_builder.local_context import (
    build_covariates_context,
    build_entity_context,
    build_relationship_context,
    get_candidate_context,
)
from graphrag.query.context_builder.source_context import (
    build_text_unit_context,
    count_relationships,
)
from graphrag.query.input.retrieval.community_reports import get_candidate_communities
from graphrag.query.input.retrieval.text_units import get_candidate_text_units
from graphrag.query.llm.base import BaseLLM, BaseLLMCallback, BaseTextEmbedding
from graphrag.query.llm.oai.chat_openai import ChatOpenAI
from graphrag.query.llm.oai.embedding import OpenAIEmbedding
from graphrag.query.llm.oai.typing import OpenaiApiType
from graphrag.query.llm.text_utils import num_tokens
from graphrag.query.structured_search.base import BaseSearch, LocalContextBuilder, SearchResult
from graphrag.query.structured_search.local_search.mixed_context import LocalSearchMixedContext
from graphrag.query.structured_search.local_search.search import LocalSearch
from graphrag.storage.pipeline_storage import PipelineStorage
from graphrag.cache.pipeline_cache import PipelineCache
from graphrag.cache.noop_pipeline_cache import NoopPipelineCache
from graphrag.vector_stores.base import (
    BaseVectorStore,
    VectorStoreDocument,
    VectorStoreSearchResult,
)
from graphrag.vector_stores.lancedb import LanceDBVectorStore


In [ ]:
import pandas as pd

#triples_df = pd.read_csv("$data")
triples_df
triples_df = triples_df .rename(columns={
    'subject': 'Subject',
    'predicate': 'Predicate',
    'object': 'Object'
})
triples_df = triples_df .rename(columns={
    'subject_def': 'Subject_description',
    'object_def': 'Object_description'

})

In [ ]:
import pandas as pd
import networkx as nx
from uuid import uuid4

def prepare_graph_from_triples_with_descriptions(triples_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, nx.DiGraph]:
    """
    Prepare nodes, edges, and a directed graph from triples DataFrame.
    Use the Subject_description and Object_description columns directly.
    """
    required_columns = {"Subject", "Predicate", "Object", "Subject_description", "Object_description"}
    if not required_columns.issubset(triples_df.columns):
        raise ValueError(f"Input DataFrame must contain columns: {required_columns}")


    triples_df["weight"] = triples_df.groupby(["Subject", "Object", "Predicate"])["Subject"].transform("count")


    unique_nodes = pd.concat([triples_df["Subject"], triples_df["Object"]]).unique()
    nodes = pd.DataFrame(unique_nodes, columns=["name"])
    nodes["id"] = nodes["name"].apply(lambda _: str(uuid4()))
    nodes["type"] = "entity"


    def get_description(node):
        desc_subj = triples_df.loc[triples_df["Subject"] == node, "Subject_description"]
        if not desc_subj.empty and pd.notna(desc_subj.iloc[0]):
            return desc_subj.iloc[0]

        desc_obj = triples_df.loc[triples_df["Object"] == node, "Object_description"]
        if not desc_obj.empty and pd.notna(desc_obj.iloc[0]):
            return desc_obj.iloc[0]
        return ""

    nodes["description"] = nodes["name"].apply(get_description)


    edges = triples_df.copy()
    edges["description"] = edges.apply(
        lambda row: f"{row['Predicate']}; Subject: {row['Subject_description']}; Object: {row['Object_description']}",
        axis=1
    )

    edges = edges.rename(columns={"Subject": "source", "Object": "target"})
    edges = edges[["source", "target", "description"]].drop_duplicates()
    edges["id"] = edges.index.map(lambda _: str(uuid4()))


    graph = nx.from_pandas_edgelist(
        edges,
        source="source",
        target="target",
        edge_attr=["description"],
        create_using=nx.DiGraph()
    )

    return nodes, edges, graph


In [ ]:
nodes, edges, graph = prepare_graph_from_triples_with_descriptions(triples_df)

print("Nodes:")
print(nodes.head())

print("Edges:")
print(edges.head())

print("Graph edges:")
print(list(graph.edges(data=True))[:5])

In [ ]:
import pandas as pd
import networkx as nx
from uuid import uuid4

def prepare_graph_from_triples(triples_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, nx.DiGraph]:
    """
    Prepare nodes, edges, and a directed graph from triples DataFrame.
    Uses Subject_description and Object_description directly.
    """

    required_columns = {"Subject", "Predicate", "Object", "Subject_description", "Object_description"}
    if not required_columns.issubset(triples_df.columns):
        raise ValueError(f"Input DataFrame must contain columns: {required_columns}")


    triples_df["weight"] = triples_df.groupby(["Subject", "Object", "Predicate"])["Subject"].transform("count")


    unique_nodes = pd.concat([triples_df["Subject"], triples_df["Object"]]).unique()
    nodes = pd.DataFrame(unique_nodes, columns=["name"])
    nodes["id"] = nodes["name"].apply(lambda _: str(uuid4()))
    nodes["type"] = "entity"


    def get_node_description(node):
        desc_subj = triples_df.loc[triples_df["Subject"] == node, "Subject_description"]
        if not desc_subj.empty and pd.notna(desc_subj.iloc[0]):
            return desc_subj.iloc[0]
        desc_obj = triples_df.loc[triples_df["Object"] == node, "Object_description"]
        if not desc_obj.empty and pd.notna(desc_obj.iloc[0]):
            return desc_obj.iloc[0]
        return ""

    nodes["description_summary"] = nodes["name"].apply(get_node_description)


    edges = triples_df.copy()
    edges["description_summary"] = edges.apply(
        lambda row: f"{row['Predicate']}; Subject: {row['Subject_description']}; Object: {row['Object_description']}",
        axis=1
    )

    edges = edges.rename(columns={"Subject": "source", "Object": "target"})
    edges = edges[["source", "target", "description_summary"]].drop_duplicates()
    edges["id"] = edges.index.map(lambda _: str(uuid4()))
    edges["human_readable_id"] = edges.index


    graph = nx.from_pandas_edgelist(
        edges,
        source="source",
        target="target",
        edge_attr=["description_summary"],
        create_using=nx.DiGraph()
    )


    degrees_df = pd.DataFrame([
        {"name": node, "in_degree": graph.in_degree(node), "out_degree": graph.out_degree(node)}
        for node in graph.nodes
    ])
    nodes = nodes.merge(degrees_df, on="name").rename(columns={"name": "title"}).reset_index(drop=True)
    nodes["human_readable_id"] = nodes.index
    nodes["id"] = nodes["human_readable_id"].apply(lambda _: str(uuid4()))

    return nodes, edges, graph


In [ ]:
nodes, edges, graph = prepare_graph_from_triples(triples_df)
print("Nodes:")
print(nodes.head())

print("\nEdges:")
print(edges.head())

print("\nGraph edges:")
print(list(graph.edges(data=True))[:5])

In [ ]:
    base_entity_nodes, base_relationship_edges, graph = prepare_graph_from_triples(triples_df)

    print("\nPrepared Nodes:")
    print(base_entity_nodes)

    print("\nPrepared Edges:")
    print(base_relationship_edges)

    print("\nGraph Edges:")
    print(list(graph.edges(data=True)))


In [ ]:
base_entity_nodes["degree"] = base_entity_nodes["in_degree"] + base_entity_nodes["out_degree"]

print(base_entity_nodes)

In [ ]:
def create_final_nodes(
    base_entity_nodes: pd.DataFrame,
    base_relationship_edges: pd.DataFrame,
    base_communities: pd.DataFrame,
    callbacks: VerbCallbacks,
    layout_strategy: dict[str, Any],
    embedding_strategy: dict[str, Any] | None = None,
) -> pd.DataFrame:
    """All the steps to transform final nodes."""
    graph = create_graph(base_relationship_edges)
    graph_embeddings = None
    if embedding_strategy:
        graph_embeddings = embed_graph(
            graph,
            embedding_strategy,
        )
    layout = layout_graph(
        graph,
        callbacks,
        layout_strategy,
        embeddings=graph_embeddings,
    )
    nodes = base_entity_nodes.merge(
        layout, left_on="title", right_on="label", how="left"
    )

    joined = nodes.merge(base_communities, on="title", how="left")
    joined["level"] = joined["level"].fillna(0).astype(int)
    joined["community"] = joined["community"].fillna(-1).astype(int)

    return joined.loc[
        :,
        [
            "id",
            "human_readable_id",
            "title",
            "community",
            "level",
            "degree",
            "x",
            "y",
        ],
    ]

In [ ]:
async def compute_communities(
    base_relationship_edges: pd.DataFrame,
    storage: PipelineStorage,
    clustering_strategy: dict[str, Any],
    snapshot_transient_enabled: bool = False,
) -> pd.DataFrame:
    """
    Compute communities based on the graph and clustering strategy.
    """
    graph = create_graph(base_relationship_edges)

    communities = cluster_graph(
        graph,
        strategy=clustering_strategy,
    )

    base_communities = pd.DataFrame(
        communities, columns=pd.Index(["level", "community", "parent", "title"])
    ).explode("title")
    base_communities["community"] = base_communities["community"].astype(int)

    if snapshot_transient_enabled:
        await snapshot(
            base_communities,
            name="base_communities",
            storage=storage,
            formats=["parquet"],
        )

    return base_communities

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
clustering_strategy = {
        "algorithm": "leiden",
        "params": {
            "resolution": 0.1,
        },
}

storage = None

base_communities = asyncio.run(
        compute_communities(
            base_relationship_edges=edges,
            storage=storage,
            clustering_strategy=clustering_strategy,
            snapshot_transient_enabled=False,
        )
)

print("\nComputed Communities:")
print(base_communities)

In [ ]:
import os
import openai
os.environ["OPENAI_API_KEY"] = ""


In [ ]:
final_nodes = create_final_nodes(
        base_entity_nodes=base_entity_nodes,
        base_relationship_edges=base_relationship_edges,
        base_communities=base_communities,
        callbacks=NoopVerbCallbacks(),
        layout_strategy={"type": "zero"},
        embedding_strategy=None,
)

In [ ]:
def create_final_entities(base_entity_nodes: pd.DataFrame):
    """"""
    return base_entity_nodes.loc[
        :,
        [
            "id",
            "human_readable_id",
            "title",
            "type",
            "description",
        ],
    ]


In [ ]:
base_entity_nodes["description"]=base_entity_nodes["description_summary"]
final_entities= create_final_entities(base_entity_nodes)
base_relationship_edges["description"]=base_relationship_edges["description_summary"]
base_relationship_edges["description"]=base_relationship_edges["description_summary"]

In [ ]:
final_nodes = final_nodes.merge(
    final_entities[['id', 'description']],
    on='id',
    how='left'
)


In [ ]:
def create_final_relationships(
    base_relationship_edges: pd.DataFrame,
    base_entity_nodes: pd.DataFrame,
) -> pd.DataFrame:
    """All the steps to transform final relationships."""
    relationships = base_relationship_edges
    relationships["combined_degree"] = compute_edge_combined_degree(
        relationships,
        base_entity_nodes,
        node_name_column="title",
        node_degree_column="degree",
        edge_source_column="source",
        edge_target_column="target",
    )

    return relationships.loc[
        :,
        [
            "id",
            "human_readable_id",
            "source",
            "target",
            "description",
            "weight",
            "combined_degree",
        ],
    ]


In [ ]:
base_relationship_edges["weight"] = 1

In [ ]:
final_relationships = create_final_relationships(base_relationship_edges, base_entity_nodes)

In [ ]:
def create_final_nodes(
    base_entity_nodes: pd.DataFrame,
    base_relationship_edges: pd.DataFrame,
    base_communities: pd.DataFrame,
    callbacks: VerbCallbacks,
    layout_strategy: dict[str, Any],
    embedding_strategy: dict[str, Any] | None = None,
) -> pd.DataFrame:
    """All the steps to transform final nodes."""
    graph = create_graph(base_relationship_edges)
    graph_embeddings = None
    if embedding_strategy:
        graph_embeddings = embed_graph(
            graph,
            embedding_strategy,
        )
    layout = layout_graph(
        graph,
        callbacks,
        layout_strategy,
        embeddings=graph_embeddings,
    )
    nodes = base_entity_nodes.merge(
        layout, left_on="title", right_on="label", how="left"
    )

    joined = nodes.merge(base_communities, on="title", how="left")
    joined["level"] = joined["level"].fillna(0).astype(int)
    joined["community"] = joined["community"].fillna(-1).astype(int)

    return joined.loc[
        :,
        [
            "id",
            "human_readable_id",
            "title",
            "community",
            "level",
            "degree",
            "x",
            "y",
        ],
    ]

In [ ]:
base_entity_nodes["title"] = base_entity_nodes["title"].str.strip().str.lower()
base_communities["title"] = base_communities["title"].str.strip().str.lower()

entity_ids = base_communities.merge(base_entity_nodes, on="title", how="inner")

print("\nMerged Entity IDs (after normalization):")
print(entity_ids)


In [ ]:
final_nodes = create_final_nodes(
        base_entity_nodes=base_entity_nodes,
        base_relationship_edges=base_relationship_edges,
        base_communities=base_communities,
        callbacks=NoopVerbCallbacks(),
        layout_strategy={"type": "zero"},
        embedding_strategy=None,
)

In [ ]:
final_nodes = final_nodes.merge(
    final_entities[['id', 'description']],
    on='id',
    how='left'
)


In [ ]:
def _prep_nodes(input: pd.DataFrame) -> pd.DataFrame:
    """"""

    input = input.loc[input.loc[:, COMMUNITY_ID] != -1]


    input.loc[:, NODE_DESCRIPTION] = input.loc[:, NODE_DESCRIPTION].fillna(
        "No Description"
    )


    input.loc[:, NODE_DETAILS] = input.loc[
        :, [NODE_ID, NODE_NAME, NODE_DESCRIPTION, NODE_DEGREE]
    ].to_dict(orient="records")

    return input


def _prep_edges(input: pd.DataFrame) -> pd.DataFrame:
    input.fillna(value={NODE_DESCRIPTION: "No Description"}, inplace=True)
    input.loc[:, EDGE_DETAILS] = input.loc[
        :, [EDGE_ID, EDGE_SOURCE, EDGE_TARGET, EDGE_DESCRIPTION, EDGE_DEGREE]
    ].to_dict(orient="records")

    return input

In [ ]:
com_nodes = _prep_nodes(final_nodes)

In [ ]:
rel2 = _prep_edges(final_relationships)
print(rel2)

In [ ]:
local_contexts = prepare_community_reports(
        com_nodes,
        rel2,
        None,
        NoopVerbCallbacks(),
        1000
)
print(local_contexts)

In [ ]:
community_hierarchy = restore_community_hierarchy(com_nodes)

In [ ]:
MOCK_LLM_CONFIG = {
    "type":  LLMType.OpenAIChat,
    "parse_json": True,
    "model": "gpt-3.5-turbo",
}


summarize_config = SummarizeDescriptionsConfig(
    strategy={
        "type": "graph_intelligence",
        "llm": MOCK_LLM_CONFIG,
        "max_summary_length": 2000,
    }
)


In [ ]:
resolved = summarize_config.resolved_strategy(root_dir="path/to/root")
print(resolved)

In [ ]:
async def create_final_community_reports(
    nodes_input: pd.DataFrame,
    edges_input: pd.DataFrame,
    entities: pd.DataFrame,
    communities: pd.DataFrame,
    claims_input: pd.DataFrame | None,
    callbacks: VerbCallbacks,
    cache: None,
    summarization_strategy: dict,
    async_mode: AsyncType = AsyncType.AsyncIO,
    num_threads: int = 4,
) :
    """All the steps to transform community reports."""
    community_reports = community_reports2


    community_reports["human_readable_id"] = community_reports["community"]
    community_reports["id"] = [uuid4().hex for _ in range(len(community_reports))]
    print(community_reports)

    merged = community_reports.merge(
        communities.loc[:, ["community", "parent", "size", "period"]],
        on="community",
        how="left",
        copy=False,
    )
    return merged.loc[
        :,
        [
            "id",
            "human_readable_id",
            "community",
            "parent",
            "level",
            "title",
            "summary",
            "full_content",
            "rank",
            "rank_explanation",
            "findings",
            "full_content_json",
            "period",
            "size",
        ],
    ]


final_repORT = asyncio.run(
    create_final_community_reports(
        final_nodes,
        final_relationships,
        final_entities,
        output,
        None,
        NoopVerbCallbacks(),
        None,
        resolved,
        async_mode="asyncio",
        num_threads=4,
    )
)



In [ ]:
final_report.rename(columns={'community': 'community_id'}, inplace=True)

In [ ]:
def create_community_reports(df: pd.DataFrame) -> list:
    """
    """
    community_reports = []
    for _, d in df.iterrows():

        community_report = CommunityReport(
            id=d['id'],
               short_id=d['human_readable_id'],
            title=d['title'],
            community_id=d['community_id'],
            summary=d['summary'],
            full_content=d['full_content'],
            rank=d['rank'],
            size=d.get('size', None),
            period=d.get('period', None)
        )
        community_reports.append(community_report)
    return community_reports

In [ ]:
com_report = create_community_reports(final_report)

In [ ]:

def create_rel_reports(df: pd.DataFrame) -> list:
    """
    """
    rel_reports = []
    for _, d in df.iterrows():

        rel_report = Relationship(
             id=d['id'],
               short_id=d['human_readable_id'],
            source=d['source'],
            target=d['target'],
            description=d.get('description'),
            weight=d.get('weight'),
        )
        rel_reports.append(rel_report)
    return rel_reports

com_relations = create_rel_reports(final_relationships)

In [ ]:

def create_community_entities(df: pd.DataFrame) -> list:
    """
    """
    community_en = []
    for _, d in df.iterrows():
        en = Entity(
            id=d['id'],
               short_id=d['human_readable_id'],
            title=d['title'],
            type=d['type'],
            description = d['description']
        )
        community_en.append(en)
    return community_en


In [ ]:
com_entities = create_community_entities(final_entities)

In [ ]:
LANCEDB_URI = "./lancedb_store"
description_embedding_store = LanceDBVectorStore(collection_name="entity_descriptions")
description_embedding_store.connect(db_uri=LANCEDB_URI)

In [ ]:

entity_text_embeddings = {
    entity.id: text_embedder.embed(entity.title) for entity in com_entities
}
documents = [
    VectorStoreDocument(
        id=entity.title,
        text=entity.title,
        vector=text_embedder.embed(entity.title),
    )
    for entity in com_entities
]
df_documents = pd.DataFrame([{
    "id": doc.id,
    "text": doc.text,
    "vector": json.dumps(doc.vector)
} for doc in documents])


In [ ]:
description_embedding_store.load_documents(df_documents, overwrite=True)

In [ ]:
api_key = os.environ["OPENAI_API_KEY"]
llm_model = "gpt-3.5-turbo"
embedding_model = "text-embedding-ada-002"

llm2 = ChatOpenAI(
    api_key=api_key,
    model="gpt-3.5-turbo",
    api_type=OpenaiApiType.OpenAI,
    max_retries=20,
)


In [ ]:

def stringify_community_report_fields(reports):
    """
    """
    stringified_reports = []
    for report in reports:

        stringified_report = CommunityReport(
            id=str(report.id),
            short_id=str(report.short_id),
            title=str(report.title),
            community_id=str(report.community_id),
            summary=str(report.summary),
            full_content=str(report.full_content),
            rank=report.rank,
            full_content_embedding=str(report.full_content_embedding)
            if report.full_content_embedding is not None
            else "",
            attributes={str(k): str(v) for k, v in report.attributes.items()}
            if report.attributes
            else {},
            size=str(report.size),
            period=str(report.period),
        )
        stringified_reports.append(stringified_report)
    return stringified_reports



com_report_stringified = stringify_community_report_fields(com_report)



In [ ]:
final_relationships = com_relations
final_entities = com_entities

In [ ]:
def stringify_entity_fields(entities):
    """
    """
    stringified_entities = []
    for entity in entities:
        stringified_entity = Entity(
            id=str(entity.id),
            short_id=str(entity.short_id),
            title=str(entity.title),
            type=str(entity.type),
            description=str(entity.description),
            description_embedding=str(entity.description_embedding)
            if entity.description_embedding is not None
            else "",
            name_embedding=str(entity.name_embedding)
            if entity.name_embedding is not None
            else "",
            community_ids=[str(community_id) for community_id in entity.community_ids]
            if entity.community_ids
            else [],
            text_unit_ids=[str(text_unit_id) for text_unit_id in entity.text_unit_ids]
            if entity.text_unit_ids
            else [],
            rank=str(entity.rank),
            attributes={str(k): str(v) for k, v in entity.attributes.items()}
            if entity.attributes
            else {},
        )
        stringified_entities.append(stringified_entity)
    return stringified_entities



entities_stringified = stringify_entity_fields(final_entities)



In [ ]:
def stringify_relationship_fields(relationships):
    """
    """
    stringified_relationships = []
    for relationship in relationships:
        stringified_relationship = Relationship(
            id=str(relationship.id),
            short_id=str(relationship.short_id),
            source=str(relationship.source),
            target=str(relationship.target),
            weight=str(relationship.weight),
            description=str(relationship.description),
            description_embedding=str(relationship.description_embedding)
            if relationship.description_embedding is not None
            else "",
            text_unit_ids=[
                str(text_unit_id) for text_unit_id in relationship.text_unit_ids
            ]
            if relationship.text_unit_ids
            else [],
            rank=str(relationship.rank),
            attributes={str(k): str(v) for k, v in relationship.attributes.items()}
            if relationship.attributes
            else {},
        )
        stringified_relationships.append(stringified_relationship)
    return stringified_relationships

relationships_stringified = stringify_relationship_fields(final_relationships)

In [ ]:
# Copyright (c) 2024 Microsoft Corporation.
# Licensed under the MIT License

log = logging.getLogger(__name__)


class LocalSearchMixedContext(LocalContextBuilder):

    def __init__(
        self,
        entities: list[Entity],
        entity_text_embeddings: BaseVectorStore,
        text_embedder: BaseTextEmbedding,
        text_units: list[TextUnit] | None = None,
        community_reports: list[CommunityReport] | None = None,
        relationships: list[Relationship] | None = None,
        covariates: dict[str, list[Covariate]] | None = None,
        token_encoder: tiktoken.Encoding | None = None,
        embedding_vectorstore_key: str = EntityVectorStoreKey.ID,
    ):
        if community_reports is None:
            community_reports = []
        if relationships is None:
            relationships = []
        if covariates is None:
            covariates = {}
        if text_units is None:
            text_units = []
        self.entities = {entity.id: entity for entity in entities}
        self.community_reports = {
            community.community_id: community for community in community_reports
        }
        self.text_units = {unit.id: unit for unit in text_units}
        self.relationships = {
            relationship.id: relationship for relationship in relationships
        }
        self.covariates = covariates
        self.entity_text_embeddings = entity_text_embeddings
        self.text_embedder = text_embedder
        self.token_encoder = token_encoder
        self.embedding_vectorstore_key = embedding_vectorstore_key

    def filter_by_entity_keys(self, entity_keys: list[int] | list[str]):
        """Filter entity text embeddings by entity keys."""
        self.entity_text_embeddings.filter_by_id(entity_keys)

    def build_context(
        self,
        query: str,
        conversation_history: ConversationHistory | None = None,
        include_entity_names: list[str] | None = None,
        exclude_entity_names: list[str] | None = None,
        conversation_history_max_turns: int | None = 5,
        conversation_history_user_turns_only: bool = True,
        max_tokens: int = 50000,
        text_unit_prop: float = 0.5,
        community_prop: float = 0.25,
        top_k_mapped_entities: int = 50,
        top_k_relationships: int = 50,
        include_community_rank: bool = False,
        include_entity_rank: bool = False,
        rank_description: str = "number of relationships",
        include_relationship_weight: bool = False,
        relationship_ranking_attribute: str = "rank",
        return_candidate_context: bool = False,
        use_community_summary: bool = False,
        min_community_rank: int = 0,
        community_context_name: str = "Reports",
        column_delimiter: str = "|",
        **kwargs: dict[str, Any],
    ) -> ContextBuilderResult:
        """
        Build data context for local search prompt.

        Build a context by combining community reports and entity/relationship/covariate tables, and text units using a predefined ratio set by summary_prop.
        """
        if include_entity_names is None:
            include_entity_names = []
        if exclude_entity_names is None:
            exclude_entity_names = []
        if community_prop + text_unit_prop > 1:
            value_error = (
                "The sum of community_prop and text_unit_prop should not exceed 1."
            )
            raise ValueError(value_error)

        if conversation_history:
            pre_user_questions = "\n".join(
                conversation_history.get_user_turns(conversation_history_max_turns)
            )
            query = f"{query}\n{pre_user_questions}"

        selected_entities = map_query_to_entities(
            query=query,
            text_embedding_vectorstore=self.entity_text_embeddings,
            text_embedder=self.text_embedder,
            all_entities_dict=self.entities,
            embedding_vectorstore_key=self.embedding_vectorstore_key,
            include_entity_names=include_entity_names,
            exclude_entity_names=exclude_entity_names,
            k=5,
            oversample_scaler=2,
        )


        final_context = list[str]()
        final_context_data = dict[str, pd.DataFrame]()

        if conversation_history:

            (
                conversation_history_context,
                conversation_history_context_data,
            ) = conversation_history.build_context(
                include_user_turns_only=conversation_history_user_turns_only,
                max_qa_turns=conversation_history_max_turns,
                column_delimiter=column_delimiter,
                max_tokens=max_tokens,
                recency_bias=False,
            )
            if conversation_history_context.strip() != "":
                final_context.append(conversation_history_context)
                final_context_data = conversation_history_context_data
                max_tokens = max_tokens - num_tokens(
                    conversation_history_context, self.token_encoder
                )


        community_tokens = max(int(max_tokens * community_prop), 0)
        community_context, community_context_data = self._build_community_context(
            selected_entities=selected_entities,
            max_tokens=community_tokens,
            use_community_summary=use_community_summary,
            column_delimiter=column_delimiter,
            include_community_rank=include_community_rank,
            min_community_rank=min_community_rank,
            return_candidate_context=return_candidate_context,
            context_name=community_context_name,
        )
        if community_context.strip() != "":
            final_context.append(community_context)
            final_context_data = {**final_context_data, **community_context_data}


        local_prop = 1 - community_prop - text_unit_prop
        local_tokens = max(int(max_tokens * local_prop), 0)
        local_context, local_context_data = self._build_local_context(
            selected_entities=selected_entities,
            max_tokens=local_tokens,
            include_entity_rank=include_entity_rank,
            rank_description=rank_description,
            include_relationship_weight=include_relationship_weight,
            top_k_relationships=top_k_relationships,
            relationship_ranking_attribute=relationship_ranking_attribute,
            return_candidate_context=return_candidate_context,
            column_delimiter=column_delimiter,
        )
        if local_context.strip() != "":
            final_context.append(str(local_context))
            final_context_data = {**final_context_data, **local_context_data}

        text_unit_tokens = max(int(max_tokens * text_unit_prop), 0)
        text_unit_context, text_unit_context_data = self._build_text_unit_context(
            selected_entities=selected_entities,
            max_tokens=text_unit_tokens,
            return_candidate_context=return_candidate_context,
        )

        if text_unit_context.strip() != "":
            final_context.append(text_unit_context)
            final_context_data = {**final_context_data, **text_unit_context_data}

        return ContextBuilderResult(
            context_chunks="\n\n".join(final_context),
            context_records=final_context_data,
        )

    def _build_community_context(
        self,
        selected_entities: list[Entity],
        max_tokens: int = 4000,
        use_community_summary: bool = False,
        column_delimiter: str = "|",
        include_community_rank: bool = False,
        min_community_rank: int = 0,
        return_candidate_context: bool = False,
        context_name: str = "Reports",
    ) -> tuple[str, dict[str, pd.DataFrame]]:
        """Add community data to the context window until it hits the max_tokens limit."""
        if len(selected_entities) == 0 or len(self.community_reports) == 0:
            return ("", {context_name.lower(): pd.DataFrame()})

        community_matches = {}
        for entity in selected_entities:

            if entity.community_ids:
                for community_id in entity.community_ids:
                    community_matches[community_id] = (
                        community_matches.get(community_id, 0) + 1
                    )
        selected_communities = [
            self.community_reports[community_id]
            for community_id in community_matches
            if community_id in self.community_reports
        ]
        for community in selected_communities:
            if community.attributes is None:
                community.attributes = {}
            community.attributes["matches"] = community_matches[community.community_id]
        selected_communities.sort(
            key=lambda x: (x.attributes["matches"], x.rank),
            reverse=True,
        )
        for community in selected_communities:
            del community.attributes["matches"]

        context_text, context_data = build_community_context(
            community_reports=selected_communities,
            token_encoder=self.token_encoder,
            use_community_summary=use_community_summary,
            column_delimiter=column_delimiter,
            shuffle_data=False,
            include_community_rank=include_community_rank,
            min_community_rank=min_community_rank,
            max_tokens=max_tokens,
            single_batch=True,
            context_name=context_name,
        )
        if isinstance(context_text, list) and len(context_text) > 0:
            context_text = "\n\n".join(context_text)

        if return_candidate_context:
            candidate_context_data = get_candidate_communities(
                selected_entities=selected_entities,
                community_reports=list(self.community_reports.values()),
                use_community_summary=use_community_summary,
                include_community_rank=include_community_rank,
            )
            context_key = context_name.lower()
            if context_key not in context_data:
                context_data[context_key] = candidate_context_data
                context_data[context_key]["in_context"] = False
            else:
                if (
                    "id" in candidate_context_data.columns
                    and "id" in context_data[context_key].columns
                ):
                    candidate_context_data["in_context"] = candidate_context_data[
                        "id"
                    ].isin(
                        context_data[context_key]["id"]
                    )
                    context_data[context_key] = candidate_context_data
                else:
                    context_data[context_key]["in_context"] = True
        return (str(context_text), context_data)

    def _build_text_unit_context(
        self,
        selected_entities: list[Entity],
        max_tokens: int = 18000,
        return_candidate_context: bool = False,
        column_delimiter: str = "|",
        context_name: str = "Sources",
    ) -> tuple[str, dict[str, pd.DataFrame]]:
        """Rank matching text units and add them to the context window until it hits the max_tokens limit."""
        if not selected_entities or not self.text_units:
            return ("", {context_name.lower(): pd.DataFrame()})
        selected_text_units = []
        text_unit_ids_set = set()

        unit_info_list = []
        relationship_values = list(self.relationships.values())
        for index, entity in enumerate(selected_entities):
            entity_relationships = [
                rel
                for rel in relationship_values
                if rel.source == entity.title or rel.target == entity.title
            ]

            for text_id in entity.text_unit_ids or []:
                if text_id not in text_unit_ids_set and text_id in self.text_units:
                    selected_unit = deepcopy(self.text_units[text_id])
                    num_relationships = count_relationships(
                        entity_relationships, selected_unit
                    )
                    text_unit_ids_set.add(text_id)
                    unit_info_list.append((selected_unit, index, num_relationships))


        unit_info_list.sort(key=lambda x: (x[1], -x[2]))

        selected_text_units = [unit[0] for unit in unit_info_list]

        context_text, context_data = build_text_unit_context(
            text_units=selected_text_units,
            token_encoder=self.token_encoder,
            max_tokens=max_tokens,
            shuffle_data=False,
            context_name=context_name,
            column_delimiter=column_delimiter,
        )

        if return_candidate_context:
            candidate_context_data = get_candidate_text_units(
                selected_entities=selected_entities,
                text_units=list(self.text_units.values()),
            )
            context_key = context_name.lower()
            if context_key not in context_data:
                candidate_context_data["in_context"] = False
                context_data[context_key] = candidate_context_data
            else:
                if (
                    "id" in candidate_context_data.columns
                    and "id" in context_data[context_key].columns
                ):
                    candidate_context_data["in_context"] = candidate_context_data[
                        "id"
                    ].isin(context_data[context_key]["id"])
                    context_data[context_key] = candidate_context_data
                else:
                    context_data[context_key]["in_context"] = True

        return (str(context_text), context_data)




    def _build_local_context(
        self,
        selected_entities: list[Entity],
        max_tokens: int = 8000,
        include_entity_rank: bool = False,
        rank_description: str = "relationship count",
        include_relationship_weight: bool = False,
        top_k_relationships: int = 15,
        relationship_ranking_attribute: str = "rank",
        return_candidate_context: bool = False,
        column_delimiter: str = "|",
    ) -> tuple[str, dict[str, pd.DataFrame]]:
        """Build data context for local search prompt combining entity/relationship/covariate tables."""

        entity_context, entity_context_data = build_entity_context(
            selected_entities=selected_entities,
            token_encoder=self.token_encoder,
            max_tokens=max_tokens,
            column_delimiter=column_delimiter,
            include_entity_rank=include_entity_rank,
            rank_description=rank_description,
            context_name="Entities",
        )
        entity_tokens = num_tokens(entity_context, self.token_encoder)

        added_entities = []
        final_context = []
        final_context_data = {}

        for entity in selected_entities:
            current_context = []
            current_context_data = {}
            added_entities.append(entity)


            (
                relationship_context,
                relationship_context_data,
            ) = build_relationship_context(
                selected_entities=added_entities,
                relationships=list(self.relationships.values()),
                token_encoder=self.token_encoder,
                max_tokens=max_tokens,
                column_delimiter=column_delimiter,
                top_k_relationships=top_k_relationships,
                include_relationship_weight=include_relationship_weight,
                relationship_ranking_attribute=relationship_ranking_attribute,
                context_name="Relationships",
            )

            current_context.append(relationship_context)
            current_context_data["relationships"] = relationship_context_data
            total_tokens = entity_tokens + num_tokens(
                relationship_context, self.token_encoder
            )

            for covariate in self.covariates:
                covariate_context, covariate_context_data = build_covariates_context(
                    selected_entities=added_entities,
                    covariates=self.covariates[covariate],
                    token_encoder=self.token_encoder,
                    max_tokens=max_tokens,
                    column_delimiter=column_delimiter,
                    context_name=covariate,
                )
                total_tokens += num_tokens(covariate_context, self.token_encoder)
                current_context.append(covariate_context)
                current_context_data[covariate.lower()] = covariate_context_data
            final_context = current_context
            final_context_data = current_context_data


        final_context_text = entity_context + "\n\n" + "\n\n".join(final_context)
        final_context_data["entities"] = entity_context_data

        if return_candidate_context:

            candidate_context_data = get_candidate_context(
                selected_entities=selected_entities,
                entities=list(self.entities.values()),
                relationships=list(self.relationships.values()),
                covariates=self.covariates,
                include_entity_rank=include_entity_rank,
                entity_rank_description=rank_description,
                include_relationship_weight=include_relationship_weight,
            )
            for key in candidate_context_data:
                candidate_df = candidate_context_data[key]
                if key not in final_context_data:
                    final_context_data[key] = candidate_df
                    final_context_data[key]["in_context"] = False
                else:
                    in_context_df = final_context_data[key]

                    if "id" in in_context_df.columns and "id" in candidate_df.columns:
                        candidate_df["in_context"] = candidate_df[
                            "id"
                        ].isin(
                            in_context_df["id"]
                        )
                        final_context_data[key] = candidate_df
                    else:
                        final_context_data[key]["in_context"] = True
        else:
            for key in final_context_data:
                final_context_data[key]["in_context"] = True
        return (final_context_text, final_context_data)




In [ ]:
api_key = os.environ["OPENAI_API_KEY"]
embedding_model = "text-embedding-ada-002"


text_embedder = OpenAIEmbedding(
    api_key=api_key,
    api_base=None,
    api_type=OpenaiApiType.OpenAI,
    model=embedding_model,
    deployment_name=embedding_model,
    max_retries=20,
)

In [ ]:
token_encoder = tiktoken.get_encoding("cl100k_base")

In [ ]:

context_builder = LocalSearchMixedContext(
    community_reports=com_report_stringified,
    text_units=None,
    entities=entities_stringified,
    relationships=relationships_stringified,
    covariates=None,
    entity_text_embeddings=description_embedding_store,
    embedding_vectorstore_key=EntityVectorStoreKey.TITLE,
    text_embedder=text_embedder,
    token_encoder=token_encoder,

)

In [ ]:
# Copyright (c) 2024 Microsoft Corporation.
# Licensed under the MIT License
"""LocalSearch implementation with customizable search prompt."""

import logging
import time
from collections.abc import AsyncGenerator
from typing import Any, Optional

import tiktoken

from graphrag.prompts.query.local_search_system_prompt import LOCAL_SEARCH_SYSTEM_PROMPT
from graphrag.query.context_builder.builders import LocalContextBuilder
from graphrag.query.context_builder.conversation_history import ConversationHistory
from graphrag.query.llm.base import BaseLLM, BaseLLMCallback
from graphrag.query.llm.text_utils import num_tokens
from graphrag.query.structured_search.base import BaseSearch, SearchResult

DEFAULT_LLM_PARAMS = {
    "max_tokens": 1500,
    "temperature": 0.0,
}

log = logging.getLogger(__name__)


class LocalSearchForGPT(BaseSearch[LocalContextBuilder]):
    """Search orchestration for local search mode with external search prompt support."""

    def __init__(
        self,
        llm: BaseLLM,
        context_builder: LocalContextBuilder,
        token_encoder: Optional[tiktoken.Encoding] = None,
        system_prompt: Optional[str] = None,
        response_type: str = "triples",
        callbacks: Optional[list[BaseLLMCallback]] = None,
        llm_params: dict[str, Any] = DEFAULT_LLM_PARAMS,
        context_builder_params: Optional[dict] = None,
    ):
        super().__init__(
            llm=llm,
            context_builder=context_builder,
            token_encoder=token_encoder,
            llm_params=llm_params,
            context_builder_params=context_builder_params or {},
        )
        self.system_prompt = system_prompt or LOCAL_SEARCH_SYSTEM_PROMPT
        self.callbacks = callbacks
        self.response_type = response_type

    async def astream_search(
        self,
        query: str,
        conversation_history: Optional[ConversationHistory] = None,
        search_prompt: Optional[str] = None,
    ) -> AsyncGenerator[str, None]:
        """Asynchronous generator for streaming RDF triples with customizable search prompt."""
        start_time = time.time()

        context_result = self.context_builder.build_context(
            query=query,
            conversation_history=conversation_history,
            **self.context_builder_params,
        )

        context_data = context_result.context_chunks


        search_prompt = search_prompt or (
            f"### Context ###\n"
            f"{context_data}\n\n"
            f"### Query ###\n"
            f"{query}\n\n"
            f"### Instructions ###\n"
            f"Extract RDF triples that **directly match** the query. "
            f"Each triple must follow the format: `<subject> <predicate> <object> .` "
            f"Do not infer additional triples beyond what is explicitly stated."
        )

        search_messages = [
            {"role": "system", "content": "You are an assistant skilled in generating RDF triples."},
            {"role": "user", "content": search_prompt},
        ]

        yield context_result.context_records
        async for response in self.llm.astream_generate(
            messages=search_messages,
            callbacks=self.callbacks,
            **self.llm_params,
        ):
            yield response

    def search(
        self,
        query: str,
        conversation_history: Optional[ConversationHistory] = None,
        search_prompt: Optional[str] = None,
        **kwargs,
    ) -> SearchResult:
        """Synchronous search function with customizable search prompt."""
        start_time = time.time()
        llm_calls, prompt_tokens, output_tokens = {}, {}, {}

        context_result = self.context_builder.build_context(
            query=query,
            conversation_history=conversation_history,
            **kwargs,
            **self.context_builder_params,
        )

        context_data = context_result.context_chunks


        search_prompt = search_prompt or (
            f"### Context ###\n"
            f"{context_data}\n\n"
            f"### Query ###\n"
            f"{query}\n\n"
            f"### Instructions ###\n"
            f"Extract RDF triples that **directly match** the query. "
            f"Each triple must follow the format: `<subject> <predicate> <object> .` "
            f"Do not infer additional triples beyond what is explicitly stated."
        )

        search_messages = [
            {"role": "system", "content": "You are an assistant skilled in generating RDF triples."},
            {"role": "user", "content": search_prompt},
        ]

        try:
            print(context_data)
            response = self.llm.generate(
                messages=search_messages,
                streaming=False,
                callbacks=self.callbacks,
                **self.llm_params,
            )

            llm_calls["response"] = 1
            prompt_tokens["response"] = num_tokens(search_prompt, self.token_encoder)
            output_tokens["response"] = num_tokens(response, self.token_encoder)

            return SearchResult(
                response=response,
                context_data=context_result.context_records,
                context_text=context_result.context_chunks,
                completion_time=time.time() - start_time,
                llm_calls=sum(llm_calls.values()),
                prompt_tokens=sum(prompt_tokens.values()),
                output_tokens=sum(output_tokens.values()),
                llm_calls_categories=llm_calls,
                prompt_tokens_categories=prompt_tokens,
                output_tokens_categories=output_tokens,
            )

        except Exception as e:
            log.exception("Exception in search: %s", e)
            return SearchResult(
                response="",
                context_data=context_result.context_records,
                context_text=context_result.context_chunks,
                completion_time=time.time() - start_time,
                llm_calls=1,
                prompt_tokens=num_tokens(search_prompt, self.token_encoder),
                output_tokens=0,
            )

    async def asearch(
        self,
        query: str,
        conversation_history: Optional[ConversationHistory] = None,
        search_prompt: Optional[str] = None,
        **kwargs,
    ) -> SearchResult:
        """Asynchronous search function with customizable search prompt."""
        start_time = time.time()
        llm_calls, prompt_tokens, output_tokens = {}, {}, {}

        context_result = context_builder.build_context(query=query)
        context_data = context_result.context_chunks


        search_prompt = (

            f"{search_prompt or ''}"
                 f"Query: "
            f"{query}\n\n"
            f"Context:\n"
            f"{context_data}\n\n"
        )

        search_prompt = search_prompt or ""

        search_messages = [
            {"role": "system", "content": "You are an assistant skilled in generating RDF triples."},
            {"role": "user", "content": search_prompt},
        ]

        try:
            response = await self.llm.agenerate(
                messages=search_messages,
                streaming=False,
                callbacks=self.callbacks,
                **self.llm_params,
            )
            llm_calls["response"] = 1
            prompt_tokens["response"] = num_tokens(search_prompt, self.token_encoder)
            output_tokens["response"] = num_tokens(response, self.token_encoder)

            return SearchResult(
                response=response,
                context_data=context_result.context_records,
                context_text=context_result.context_chunks,
                completion_time=time.time() - start_time,
                llm_calls=sum(llm_calls.values()),
                prompt_tokens=sum(prompt_tokens.values()),
                output_tokens=sum(output_tokens.values()),
                llm_calls_categories=llm_calls,
                prompt_tokens_categories=prompt_tokens,
                output_tokens_categories=output_tokens,
            )

        except Exception as e:
            log.exception("Exception in asearch: %s", e)
            return SearchResult(
                response="",
                context_data=context_result.context_records,
                context_text=context_result.context_chunks,
                completion_time=time.time() - start_time,
                llm_calls=1,
                prompt_tokens=num_tokens(search_prompt, self.token_encoder),
                output_tokens=0,
            )


In [ ]:
p1 = """You are a Knowledge Graph Expert. You will be provided with a community summary report and a query, both written in natural language. Your task is to extract relevant knowledge in the form of RDF triples to accurately answer the query based on the information contained in the community reports. Provide the extracted RDF triples in the format: 'subject predicate object', always in the same correct order: first-subject, second - predicate, third - object. Return structured list without additional explanations or descriptions, avoid numbering the elements and do not include additional special characters or additional text.
Special case: If the query starts with '*Is it true that*', after  returning the triples also return the according boolean anwer in this format : Final True or False. Final True or Final False"""


In [ ]:
p2 = """You are a Knowledge Graph Expert. You will be provided with a community summary report and a query, both written in natural language. Your task is to extract relevant knowledge in the form of RDF triples to accurately answer the query based on the information contained in the community reports. If the query includes the predicate, return only those triples that match that exact predicate. Return the subject or object that match the query and keep the exact format specified in the query without modifying the names and notation. Provide the extracted RDF triples in the format: 'subject predicate object', always in the same, correct order: first-subject, second- predicate, third - object. Return structured list without additional explanations or descriptions, avoid numbering the elements and do not include additional special characters or additional text.Special case: If the query starts with '*Is it true that*', after  returning the triples also return the according boolean anwer in this format : Final True or False. Final True or Final False"""

In [ ]:
p3 = """
You are a Knowledge Graph Expert. You will be provided with a community summary report and a query, both written in natural language. Your task is to extract relevant knowledge in the form of RDF triples to accurately answer the query based on the information contained in the community reports. If the query includes the predicate, return only those triples that match that exact predicate. Return the subject or object that match the query and keep the exact format specified in the query without modifying the names and notation. Provide the extracted RDF triples in the format: 'subject predicate object', always in the same, correct order: first-subject, second- predicate, third - object. Return structured list without additional explanations or descriptions, avoid numbering the elements and do not include additional special characters or additional text. Special case: If the query starts with '*Is it true that*', after  returning the triples also return the according boolean anwer in this format : Final True or False. Final True or Final False

The report consists of two sections:
- Entities Section: Contains entities along with their descriptions.
- Relationships Section: Contains relationships between entities, specifying a source entity, target entity, and the predicate that connects them.

Extraction Tasks

1. Identify key elements in the query:
   - Determine if the query specifies a subject, object, or predicate.
   - If an entity is missing, infer it based on available predicates.
   Determine if the query is a boolean question expecting TRUE or FALSE.

2. Search the Entities Section:
   - If an entity is mentioned, find relevant triples where it appears as a subject or object.
   - Match predicates exactly when provided.

3. Search the Relationships Section:
   - Locate relationships that involve the mentioned entities.
   - Extract all predicates that connect two entities when no specific predicate is given.

4. Handle queries with missing information:
   - If only a predicate is provided, retrieve all subjects and objects associated with it.
   - If only entities are given, extract all relationships between them.

5. Format the extracted triples correctly:
   - Each triple should follow the format:
     <subject> <predicate> <object> .

6. Ensure output constraints:
   -Provide the extracted RDF triples as a structured list without additional explanations or descriptions.
   -Each triple should be in the format: 'subject predicate object'. Do not separate into different sections, just list all triples in a single combined list.
   -Avoid numbering the elements.
   -Do not add special characters.
   -Do not generate additional text—only output the extracted triples.
"""


In [ ]:
p4 = """
You are a Knowledge Graph Expert. You will be provided with a community summary report and a query, both written in natural language. Your task is to extract relevant knowledge in the form of RDF triples to accurately answer the query based on the information contained in the community reports. If the query includes the predicate, return only those triples that match that exact predicate. Return the subject or object that match the query and keep the exact format specified in the query without modifying the names and notation. Provide the extracted RDF triples in the format: 'subject predicate object', always in the same, correct order: first-subject, second- predicate, third - object. Return structured list without additional explanations or descriptions, avoid numbering the elements and do not include additional special characters or additional text. Special case: If the query starts with '*Is it true that*', after  returning the triples also return the according boolean anwer in this format : Final True or False. Final True or Final False.
The report consists of two sections:

- Entities Section: Contains entities along with descriptive text about them.- Relationships Section: Contains relationships between entities, specifying a source entity, target entity, and the predicate that connects them.

Extraction Instructions:

1. Analyze the query:
   - Identify the element from the triple that has to be reconstructed
   - Determine if the query is a boolean question expecting TRUE or FALSE.

2. Search for relevant triples:
   - In the Entities Section, extract triples by inferring predicates and objects from entity descriptions when necessary.
   - In the Relationships Section, extract all triples that match the mentioned entities and predicates.
   - If the query specifies a predicate, return only triples that match it exactly.
   - If the query provides only entities, return all triples connecting them.

3. Handle boolean queries:
   - If the query expects a TRUE or FALSE answer, do not return triples; return only the corresponding boolean value.

4. Formatting rules:
   - Return each triple in the format: 'subject predicate object'.
   - List all triples in a single combined list without numbering or extra characters.
   - Do not add explanations, sections, or additional text.


5. Maintain entity names and notation exactly as they appear in the report.

Examples:

Entities:
Entity(id='c2c1faed-3099-4beb-ad87-980842678b4d', title='structure_of_right_superior_intercostal_vein', description='The right superior intercostal vein is a vein that drains blood from the upper intercostal spaces into the azygos vein.')
Entity(id='f7f75f9d-d840-4d04-9978-f71ad1504004', title='structure_of_azygous_vein', description='The structure of the azygous vein is a vein running along the right side of the vertebral column that drains blood from the thoracic wall and upper lumbar region into the superior vena cava.')

Relationships:
tributary_of - source structure_of_right_superior_intercostal_vein target structure_of_azygous_vein

Example Queries and Expected Outputs:


Query boolean: "Is it true that structure_of_right_superior_intercostal_vein tributary of structure_of_azygous_vein?"
Expected Response (boolean):
YES

 Query : "structure_of_right_superior_intercostal_vein is tributary of who?"
Expected Response:
umls:structure_of_right_superior_intercostal_vein umls:tributary_of umls:structure_of_azygous_vein .




"""


In [ ]:
from openai import OpenAI

client = OpenAI()


In [ ]:
SPARQL_ENDPOINT = "https://fa7c6acf0cf6.ngrok-free.app/repositories/6febcuinf"

In [ ]:
llm1 =  ChatOpenAI(
    api_key=api_key,
    model="gpt-3.5-turbo",
    api_type=OpenaiApiType.OpenAI,
    max_retries=20,
)
llm2 =  ChatOpenAI(
    api_key=api_key,
    model="gpt-4o",
    api_type=OpenaiApiType.OpenAI,
    max_retries=20,
)

In [ ]:


local_context_params = {
    "text_unit_prop": 0.2,
    "community_prop": 0.8,
    "conversation_history_max_turns": 5,
    "conversation_history_user_turns_only": True,
    "top_k_mapped_entities": 50,
    "top_k_relationships": 20,
    "include_entity_rank": True,
    "include_relationship_weight": True,
    "include_community_rank": False,
    "return_candidate_context": False,
    "embedding_vectorstore_key": EntityVectorStoreKey.ID,
    "max_tokens": 1000,
}

llm_params = {
    "max_tokens": 1000,
    "temperature": 0.0,
}


In [ ]:
se1 = LocalSearchForGPT(
    llm=llm1,
    context_builder=context_builder,
    token_encoder=token_encoder,
    llm_params=llm_params,
    context_builder_params=local_context_params,
    response_type="triples",
)

In [ ]:
se2 = LocalSearchForGPT(
    llm=llm2,
    context_builder=context_builder,
    token_encoder=token_encoder,
    llm_params=llm_params,
    context_builder_params=local_context_params,
    response_type="triples",
)

In [ ]:
api_key = os.environ["OPENAI_API_KEY"]
embedding_model = "text-embedding-ada-002"


text_embedder = OpenAIEmbedding(
    api_key=api_key,
    api_base=None,
    api_type=OpenaiApiType.OpenAI,
    model=embedding_model,
    deployment_name=embedding_model,
    max_retries=20,
)

In [ ]:
token_encoder = tiktoken.get_encoding("cl100k_base")

In [ ]:

context_builder = LocalSearchMixedContext(
    community_reports=com_report_stringified,
    text_units=None,
    entities=entities_stringified,
    relationships=relationships_stringified,
    covariates=None,
    entity_text_embeddings=description_embedding_store,
    embedding_vectorstore_key=EntityVectorStoreKey.TITLE,
    text_embedder=text_embedder,
    token_encoder=token_encoder,

)

In [ ]:
# Copyright (c) 2024 Microsoft Corporation.
# Licensed under the MIT License
"""LocalSearch implementation with customizable search prompt."""

import logging
import time
from collections.abc import AsyncGenerator
from typing import Any, Optional

import tiktoken

from graphrag.prompts.query.local_search_system_prompt import LOCAL_SEARCH_SYSTEM_PROMPT
from graphrag.query.context_builder.builders import LocalContextBuilder
from graphrag.query.context_builder.conversation_history import ConversationHistory
from graphrag.query.llm.base import BaseLLM, BaseLLMCallback
from graphrag.query.llm.text_utils import num_tokens
from graphrag.query.structured_search.base import BaseSearch, SearchResult

DEFAULT_LLM_PARAMS = {
    "max_tokens": 1500,
    "temperature": 0.0,
}

log = logging.getLogger(__name__)


class LocalSearchForGPT(BaseSearch[LocalContextBuilder]):
    """Search orchestration for local search mode with external search prompt support."""

    def __init__(
        self,
        llm: BaseLLM,
        context_builder: LocalContextBuilder,
        token_encoder: Optional[tiktoken.Encoding] = None,
        system_prompt: Optional[str] = None,
        response_type: str = "triples",
        callbacks: Optional[list[BaseLLMCallback]] = None,
        llm_params: dict[str, Any] = DEFAULT_LLM_PARAMS,
        context_builder_params: Optional[dict] = None,
    ):
        super().__init__(
            llm=llm,
            context_builder=context_builder,
            token_encoder=token_encoder,
            llm_params=llm_params,
            context_builder_params=context_builder_params or {},
        )
        self.system_prompt = system_prompt or LOCAL_SEARCH_SYSTEM_PROMPT
        self.callbacks = callbacks
        self.response_type = response_type

    async def astream_search(
        self,
        query: str,
        conversation_history: Optional[ConversationHistory] = None,
        search_prompt: Optional[str] = None,
    ) -> AsyncGenerator[str, None]:
        """Asynchronous generator for streaming RDF triples with customizable search prompt."""
        start_time = time.time()

        context_result = self.context_builder.build_context(
            query=query,
            conversation_history=conversation_history,
            **self.context_builder_params,
        )

        context_data = context_result.context_chunks


        search_prompt = search_prompt or (
            f"### Context ###\n"
            f"{context_data}\n\n"
            f"### Query ###\n"
            f"{query}\n\n"
            f"### Instructions ###\n"
            f"Extract RDF triples that **directly match** the query. "
            f"Each triple must follow the format: `<subject> <predicate> <object> .` "
            f"Do not infer additional triples beyond what is explicitly stated."
        )

        search_messages = [
            {"role": "system", "content": "You are an assistant skilled in generating RDF triples."},
            {"role": "user", "content": search_prompt},
        ]

        yield context_result.context_records
        async for response in self.llm.astream_generate(
            messages=search_messages,
            callbacks=self.callbacks,
            **self.llm_params,
        ):
            yield response

    def search(
        self,
        query: str,
        conversation_history: Optional[ConversationHistory] = None,
        search_prompt: Optional[str] = None,
        **kwargs,
    ) -> SearchResult:
        """Synchronous search function with customizable search prompt."""
        start_time = time.time()
        llm_calls, prompt_tokens, output_tokens = {}, {}, {}

        context_result = self.context_builder.build_context(
            query=query,
            conversation_history=conversation_history,
            **kwargs,
            **self.context_builder_params,
        )

        context_data = context_result.context_chunks


        search_prompt = search_prompt or (
            f"### Context ###\n"
            f"{context_data}\n\n"
            f"### Query ###\n"
            f"{query}\n\n"
            f"### Instructions ###\n"
            f"Extract RDF triples that **directly match** the query. "
            f"Each triple must follow the format: `<subject> <predicate> <object> .` "
            f"Do not infer additional triples beyond what is explicitly stated."
        )

        search_messages = [
            {"role": "system", "content": "You are an assistant skilled in generating RDF triples."},
            {"role": "user", "content": search_prompt},
        ]

        try:
            print(context_data)
            response = self.llm.generate(
                messages=search_messages,
                streaming=False,
                callbacks=self.callbacks,
                **self.llm_params,
            )

            llm_calls["response"] = 1
            prompt_tokens["response"] = num_tokens(search_prompt, self.token_encoder)
            output_tokens["response"] = num_tokens(response, self.token_encoder)

            return SearchResult(
                response=response,
                context_data=context_result.context_records,
                context_text=context_result.context_chunks,
                completion_time=time.time() - start_time,
                llm_calls=sum(llm_calls.values()),
                prompt_tokens=sum(prompt_tokens.values()),
                output_tokens=sum(output_tokens.values()),
                llm_calls_categories=llm_calls,
                prompt_tokens_categories=prompt_tokens,
                output_tokens_categories=output_tokens,
            )

        except Exception as e:
            log.exception("Exception in search: %s", e)
            return SearchResult(
                response="",
                context_data=context_result.context_records,
                context_text=context_result.context_chunks,
                completion_time=time.time() - start_time,
                llm_calls=1,
                prompt_tokens=num_tokens(search_prompt, self.token_encoder),
                output_tokens=0,
            )

    async def asearch(
        self,
        query: str,
        conversation_history: Optional[ConversationHistory] = None,
        search_prompt: Optional[str] = None,
        **kwargs,
    ) -> SearchResult:
        """Asynchronous search function with customizable search prompt."""
        start_time = time.time()
        llm_calls, prompt_tokens, output_tokens = {}, {}, {}

        context_result = context_builder.build_context(query=query)
        context_data = context_result.context_chunks


        search_prompt = (

            f"{search_prompt or ''}"
                 f"Query: "
            f"{query}\n\n"
            f"Context:\n"
            f"{context_data}\n\n"
        )

        search_prompt = search_prompt or ""

        search_messages = [
            {"role": "system", "content": "You are an assistant skilled in generating RDF triples."},
            {"role": "user", "content": search_prompt},
        ]

        try:
            response = await self.llm.agenerate(
                messages=search_messages,
                streaming=False,
                callbacks=self.callbacks,
                **self.llm_params,
            )


            llm_calls["response"] = 1
            prompt_tokens["response"] = num_tokens(search_prompt, self.token_encoder)
            output_tokens["response"] = num_tokens(response, self.token_encoder)

            return SearchResult(
                response=response,
                context_data=context_result.context_records,
                context_text=context_result.context_chunks,
                completion_time=time.time() - start_time,
                llm_calls=sum(llm_calls.values()),
                prompt_tokens=sum(prompt_tokens.values()),
                output_tokens=sum(output_tokens.values()),
                llm_calls_categories=llm_calls,
                prompt_tokens_categories=prompt_tokens,
                output_tokens_categories=output_tokens,
            )

        except Exception as e:
            log.exception("Exception in asearch: %s", e)
            return SearchResult(
                response="",
                context_data=context_result.context_records,
                context_text=context_result.context_chunks,
                completion_time=time.time() - start_time,
                llm_calls=1,
                prompt_tokens=num_tokens(search_prompt, self.token_encoder),
                output_tokens=0,
            )


In [ ]:
import re
import ast

def strip_prefix(token):
    if not token or not isinstance(token, str):
        return token
    if '#' in token:
        token = token.split('#')[-1]
    if ':' in token:
        token = token.split(':')[-1]
    for p in ("umls", "rdf", "rdfs"):
        if token.startswith(p):
            token = token[len(p):]
            break

    return token


    return token
def extract_triplessparql(triples_list):
    if not triples_list or not isinstance(triples_list, list):
        return []

    clean = []

    for line in triples_list:
        if not isinstance(line, str):
            continue

        parts = line.strip().split()
        if len(parts) != 3:
            continue

        s, p, o = parts

        s = strip_prefix(s)
        p = strip_prefix(p)
        o = strip_prefix(o)

        clean.append(f"{s} {p} {o}")

    return clean


PREFIXES = ("umls", "rdf", "rdfs")

def strip_prefixes(text):

    if not text or not isinstance(text, str):
        return text

    for p in PREFIXES:
        text = re.sub(rf'\b{p}:\s*', '', text)
        text = re.sub(rf'\b{p}(?=[A-Z_])', '', text)

    return text


def extract_triplesf(response_text):
    if not response_text or not isinstance(response_text, str):
        return []
    response_text = re.sub(r'^\s*\d+\.\s*', '', response_text, flags=re.MULTILINE)
    response_text = re.sub(r'-', '', response_text)
    response_text = re.sub(r'[\[\]\(\)\.\,\–\—]', ' ', response_text)
    response_text = re.sub(r'\s+', ' ', response_text).strip()
    pattern = r'\b([a-zA-Z0-9_]+)\s+([a-zA-Z0-9_]+)\s+([a-zA-Z0-9_]+)\b'
    matches = re.findall(pattern, response_text)

    return [f"{s} {p} {o}" for s, p, o in matches if s and p and o]


def extract_triplesfsl(response_text):
    if not response_text or not isinstance(response_text, str):
        return []

    response_text = re.sub(r'^\s*\d+\.\s*', '', response_text, flags=re.MULTILINE)
    response_text = re.sub(r'-', '', response_text)
    response_text = re.sub(r'[\[\]\(\)\.\,\–\—]', ' ', response_text)
    response_text = re.sub(r'\s+', ' ', response_text).strip()

    component = r'(?:[a-zA-Z_][a-zA-Z0-9_]*:[a-zA-Z0-9_]+|[a-zA-Z0-9_]+)'
    pattern = rf'\b({component})\s+({component})\s+({component})\b'

    matches = re.findall(pattern, response_text)

    return [f"{s} {p} {o}" for s, p, o in matches]

def extract_triples(data):
    """Extracts triples from the SPARQL query results."""
    triples = []
    if not data or 'results' not in data or 'bindings' not in data['results']:
        return triples

    for binding in data['results']['bindings']:
        sub = binding['subject']['value'].split("/")[-1]
        pred = binding['predicate']['value'].split("/")[-1]
        obj = binding['object']['value'].split("/")[-1]
        triples.append(f"{sub} {pred} {obj}")

    return triples






def extract_array_from_string(text):
    """Extracts an array of triples from a given text string, handling various formats."""
    if not text or not isinstance(text, str):
        return []
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if match:
        array_str = match.group(0)
        try:
            triples_list = ast.literal_eval(array_str)
            if isinstance(triples_list, list):
                return [clean_triple(triple) for triple in triples_list]
        except (SyntaxError, ValueError):
            pass
    text = re.sub(r'\d+\.\s*', '', text)
    text = re.sub(r'^-\s*', '', text, flags=re.MULTILINE)
    text = text.strip()
    triples = text.split("\n")

    return [clean_triple(triple) for triple in triples if len(triple.split()) >= 3]


def clean_triple(triple):
    """Cleans and standardizes a triple by removing special characters and normalizing spaces."""
    triple = triple.strip()
    triple = re.sub(r'[^a-zA-Z0-9_ ]', '', triple)
    return " ".join(triple.split())


def extract_triples_from_turtle(response_text):
    """
    Extracts valid triples (subject predicate object) from a Turtle-style response.
    - Removes Markdown (` ```turtle ... ``` `)
    - Cleans up extra spaces, backticks, and symbols
    - Extracts valid triples only
    - Ensures no trailing spaces
    """
    if not response_text or not isinstance(response_text, str):
        return []


    response_text = re.sub(r'```turtle\s*|\s*```', '', response_text, flags=re.DOTALL).strip()


    triples = []
    for line in response_text.split("\n"):
        clean_line = line.strip().rstrip('.')
        clean_line = re.sub(r'[`]', '', clean_line)
        clean_line = " ".join(clean_line.split())
        words = clean_line.split()

        if len(words) == 3:
            triples.append(clean_line)

    return triples


def find_common_and_extra_elements(benchmark_triples, model_response_text):
    """
    Compares benchmark triples (unchanged) with extracted & normalized model response triples.
    """
    benchmark_set = set(benchmark_triples)
    print(model_response_text)
    model_response_set = set(model_response_text)

    print("\n=== Benchmark Triples ===")
    print(benchmark_set)
    print("\n=== LLM Triples ===")
    print(model_response_set)


    common_elements = list(benchmark_set & model_response_set)
    extra_elements = list(model_response_set - benchmark_set)

    total_benchmark = len(benchmark_set)
    total_penalty = len(extra_elements)


    accuracy = len(common_elements) / max(len(benchmark_set), len(model_response_set)) if max(len(benchmark_set), len(model_response_set)) > 0 else 0

    return common_elements, extra_elements, accuracy

def find_common_and_extra_elements2(benchmark_triples, model_response_triples):

    normalized_benchmark = set(benchmark_triples)
    normalized_model_response = set(model_response_triples)

    print("\n=== Benchmark Triples (Normalized) ===")
    print(normalized_benchmark)
    print("\n=== LLM Triples (Normalized) ===")
    print(normalized_model_response)

    common_elements = list(normalized_benchmark & normalized_model_response)
    extra_elements = list(normalized_model_response - normalized_benchmark)


    precision = len(common_elements) / len(normalized_model_response) if len(normalized_model_response) > 0 else 0
    recall = len(common_elements) / len(normalized_benchmark) if len(normalized_benchmark) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0


    return common_elements, extra_elements, precision, recall, f1


def find_common_and_extra_elements_subjectobj_only_strings(benchmark_strings, model_strings):

    benchmark_triples = [s.split() for s in benchmark_strings]
    model_triples     = [s.split() for s in model_strings]

    benchmark_pairs = set((t[0], t[2]) for t in benchmark_triples)
    model_pairs     = set((t[0], t[2]) for t in model_triples)

    common = list(benchmark_pairs & model_pairs)
    extra  = list(model_pairs - benchmark_pairs)

    precision = len(common) / len(model_pairs) if model_pairs else 0
    recall    = len(common) / len(benchmark_pairs) if benchmark_pairs else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0

    return common, extra, precision, recall, f1




def find_common_and_extra_elements_subjectobj_any_order(benchmark_strings, model_strings):
    """
    Compara doar subiect + obiect din liste de stringuri 'S P O',
    ignorand ordinea (a,b) ~ (b,a).
    """

    benchmark_triples = [s.split() for s in benchmark_strings]
    model_triples     = [s.split() for s in model_strings]


    benchmark_pairs = set(frozenset([t[0], t[2]]) for t in benchmark_triples)
    model_pairs     = set(frozenset([t[0], t[2]]) for t in model_triples)


    common = list(benchmark_pairs & model_pairs)
    extra  = list(model_pairs - benchmark_pairs)


    precision = len(common) / len(model_pairs) if model_pairs else 0
    recall    = len(common) / len(benchmark_pairs) if benchmark_pairs else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0

    return common, extra, precision, recall, f1


def find_common_and_extra_elements_subject_only_strings(benchmark_strings, model_strings):
    """
    Compara doar subiect + obiect din liste de stringuri 'S P O'.
    """

    benchmark_triples = [s.split() for s in benchmark_strings]
    model_triples     = [s.split() for s in model_strings]


    benchmark_pairs = set((t[0]) for t in benchmark_triples)
    model_pairs     = set((t[0]) for t in model_triples)


    common = list(benchmark_pairs & model_pairs)
    extra  = list(model_pairs - benchmark_pairs)


    precision = len(common) / len(model_pairs) if model_pairs else 0
    recall    = len(common) / len(benchmark_pairs) if benchmark_pairs else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0

    return common, extra, precision, recall, f1

def find_common_and_extra_elements_object_only_strings(benchmark_strings, model_strings):
    """
    Compara doar subiect + obiect din liste de stringuri 'S P O'.
    """

    benchmark_triples = [s.split() for s in benchmark_strings]
    model_triples     = [s.split() for s in model_strings]


    benchmark_pairs = set((t[2]) for t in benchmark_triples)
    model_pairs     = set((t[2]) for t in model_triples)


    common = list(benchmark_pairs & model_pairs)
    extra  = list(model_pairs - benchmark_pairs)


    precision = len(common) / len(model_pairs) if model_pairs else 0
    recall    = len(common) / len(benchmark_pairs) if benchmark_pairs else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0

    return common, extra, precision, recall, f1

def normalize_entity(x: str) -> str:
    if ":" in x:
        x = x.split(":")[-1]
    if "#" in x:
        x = x.split("#")[-1]
    return x


def find_common_and_extra_elements_object_only_strings(benchmark_strings, model_strings):
    """
    Compara doar obiectul din liste de stringuri 'S P O'.
    """
    benchmark_triples = [s.split() for s in benchmark_strings]
    model_triples     = [s.split() for s in model_strings]


    benchmark_objs = set(normalize_entity(t[2]) for t in benchmark_triples if len(t) >= 3)
    model_objs     = set(normalize_entity(t[2]) for t in model_triples if len(t) >= 3)

    common = list(benchmark_objs & model_objs)
    extra  = list(model_objs - benchmark_objs)

    precision = len(common) / len(model_objs) if model_objs else 0
    recall    = len(common) / len(benchmark_objs) if benchmark_objs else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0

    return common, extra, precision, recall, f1

def find_common_and_extra_elements_subjectobj_only_strings(benchmark_strings, model_strings):
    """
    Compara doar subiect + obiect din liste de stringuri 'S P O'.
    """

    benchmark_triples = [s.split() for s in benchmark_strings]
    model_triples     = [s.split() for s in model_strings]


    benchmark_pairs = set((t[0], t[2]) for t in benchmark_triples)
    model_pairs     = set((t[0], t[2]) for t in model_triples)

    common = list(benchmark_pairs & model_pairs)
    extra  = list(model_pairs - benchmark_pairs)


    precision = len(common) / len(model_pairs) if model_pairs else 0
    recall    = len(common) / len(benchmark_pairs) if benchmark_pairs else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0

    return common, extra, precision, recall, f1


import re

def normalize_to_bool(x):
    if x is None:
        return None

    s = str(x).strip().lower()

    s = re.sub(r"[^a-z0-9]", "", s)

    if s in {"true", "yes", "1"}:
        return True
    if s in {"false", "no", "0"}:
        return False

    return None



In [ ]:
ont = "$ontology"

In [ ]:
async def infer_rdfs_triples_simple_modified(query: str, response_text: str) -> str:
    client = OpenAI()

    reasoning_prompt = f"""Based on the obtained triples ABOX + TBOX -> TBOX {ont} ABOX{response_text}
read the query again: {query} AND use RDFS reasoning, return the new inferred triples as if they were in a GraphDB reasoning RDFS inference tool.

IMPORTANT: Return ONLY the inferred triples in the following format, with NO explanations, NO introductory text, NO additional comments:
<subject> <predicate> <object> .
<subject> <predicate> <object> .
...
"""

    print(reasoning_prompt)

    completion = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are an expert in RDF, GraphDB, and RDFS reasoning. You return ONLY triples in RDF format, with no explanations or additional text."},
            {"role": "user", "content": reasoning_prompt}
        ],
        temperature=0
    )

    answer = completion.choices[0].message.content.strip()
    print(answer)
    return answer


async def infer_rdfs_triples_simple_modifiedr(query: str, response_text: str) -> str:

    client = OpenAI()
    print(response_text)
    print(query)
    reasoning_prompt2 = f"""Based on the obtained triples ABOX + TBOX -> TBOX {ont} ABOX{response_text}
    read the query again: {query} AND use RDFS reasoning, return the new inferred triples as if they were in a GraphDB reasoning RDFS inference tool. Pay attention also to the ontology range and domain properties. Go through subClassOf and subProeprtyOf relations if it s the case to enable inference

    IMPORTANT: Return ONLY the inferred triples in the following format, with NO explanations, NO introductory text, NO additional comments:
    <subject> <predicate> <object> .
    <subject> <predicate> <object> ....IMPORTANT- EXCEPTION : If the query starts with '*Is it true that*', do not return triples but return the according boolean anwer : True or False'
    """

    reasoning_prompt = f"""
    You are an RDF reasoner with RDFS semantics.

    INPUT:
    TBOX (ontology):
    {ont}

    ABOX (asserted triples):
    {response_text}

    QUERY:
    {query}

    === STEP 1: MATERIALIZE RDFS CLOSURE ===

    From the given ABOX and TBOX, explicitly derive ALL triples that can be inferred using ONLY these rules:

    R1. rdfs:subClassOf is transitive
    R2. rdfs:subPropertyOf is transitive
    R3. If (x P y) AND (y rdfs:subClassOf C), infer (x P C)
    R4. If (P rdfs:domain D) AND (x P y), infer (x rdf:type D)
    R5. If (P rdfs:range R) AND (x P y), infer (y rdf:type R)

    Repeat applying the rules until no new triples can be derived.
    The result is the INFERRED GRAPH.
    ased on the obtained triples ABOX + TBOX -> TBOX {ont} ABOX{response_text}
    read the query again: {query} AND use RDFS reasoning, return the new inferred triples as if they were in a GraphDB reasoning RDFS inference tool. Pay attention also to the ontology range and domain properties. Go through subClassOf and subProeprtyOf relations if it s the case to enable inference.

    === STEP 2: ANSWER THE QUERY ===

       IMPORTANT: Return ONLY the inferred triples in the following format, with NO explanations, NO introductory text, NO additional comments:
    <subject> <predicate> <object> .
    <subject> <predicate> <object> ....
    IMPORTANT- EXCEPTION : If the query starts with '*Is it true that*', after  returning the triples also return the according boolean anwer in this format : Final True or False'
    """

    completion = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are an expert in RDF, GraphDB, and RDFS reasoning. You return ONLY triples in RDF format, with no explanations or additional text."},
            {"role": "user", "content": reasoning_prompt}
        ],
        temperature=0
    )

    answer = completion.choices[0].message.content.strip()
    print(answer)
    return answer

In [ ]:
async def search_by_subjects_location_of_clean_triples(
    triple_strings,
    local_search,
    prompt_level
):


    subjects = set(s.split()[0] for s in triple_strings if s.strip())

    print(subjects)

    all_triples = []
    for subject in subjects:
        query = f"{subject} {predicate}"

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        print(response_text)


        clean_text = strip_prefixes(response_text)
        print(clean_text)

        triples = extract_triplesfsl(clean_text)
        print(triples)


        all_triples.extend(triples)


    all_triples = list(set(all_triples))

    return all_triples


In [ ]:
async def search_by_subjects_location_of_clean_triples2(
    triple_strings,
    local_search,
    prompt_level,
    predicate
):

    EXCLUDED_SUBJECTS = {"type", "subClassOf"}
    subjects = set()

    for s in triple_strings:
        if not s.strip():
            continue

        for token in s.split():
            if token == predicate:
                continue
            if token in EXCLUDED_SUBJECTS:
                continue

            subjects.add(token)

    print(subjects)

    all_triples = []


    for subject in subjects:
        query = f"{subject} {predicate} ?"

        print(query)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response

        print(response_text)

        clean_text = strip_prefixes(response_text)

        print(clean_text)

        triples = extract_triplesfsl(clean_text)

        print(triples)

        all_triples.extend(triples)


    all_triples = list(set(all_triples))

    return all_triples


In [ ]:
async def search_by_subjects_location_of_clean_triples2(
    triple_strings,
    local_search,
    prompt_level,
    predicate
):


    EXCLUDED_SUBJECTS = {"type", "subClassOf"}

    subjects = set()


    for s in triple_strings:
        if not s.strip():
            continue

        tokens = s.split()


        if predicate not in tokens:
            continue

        pred_index = tokens.index(predicate)


        if pred_index == 0:
            continue

        subject = tokens[pred_index - 1]

        if subject in EXCLUDED_SUBJECTS:
            continue

        subjects.add(subject)


    print(subjects)

    all_triples = []


    for subject in subjects:
        query = f"{subject} {predicate} ?"

        print(query)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response

        print(response_text)

        clean_text = strip_prefixes(response_text)
        triples = extract_triplesfsl(clean_text)
        all_triples.extend(triples)

    return list(set(all_triples))


In [ ]:
async def search_by_subjects_location_of_clean_triples2(
    triple_strings,
    local_search,
    prompt_level,
    predicate
):

    subjects = set()

    for s in triple_strings:
        if not s.strip():
            continue

        for token in s.split():
            if token != predicate:
                subjects.add(token)


    print(subjects)

    all_triples = []


    for subject in subjects:
        query = f"{subject} {predicate} ?"

        print(query)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response


        print(response_text)


        clean_text = strip_prefixes(response_text)

        print(clean_text)

        triples = extract_triplesfsl(clean_text)

        print(triples)


        all_triples.extend(triples)


    all_triples = list(set(all_triples))

    return all_triples


In [ ]:
async def search_by_subjects_location_of_clean_triplesfinal(
    triple_strings,
    local_search,
    prompt_level,
    predicate
):
    EXCLUDED = {"type", "subClassOf", predicate}

    subjects = set()
    triples_with_predicate = []


    for s in triple_strings:
        if not s.strip():
            continue
        if predicate in s.split():
            triples_with_predicate.append(s)


    source_triples = (
        triples_with_predicate
        if triples_with_predicate
        else triple_strings
    )


    for s in source_triples:
        for token in s.split():
            if token in EXCLUDED:
                continue
            subjects.add(token)


    print(subjects)

    all_triples = []


    for subject in subjects:
        query = f"{subject} {predicate} ?"
        print("query:", query)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        clean_text = strip_prefixes(response.response)
        triples = extract_triplesfsl(clean_text)
        all_triples.extend(triples)

    return list(set(all_triples))


In [ ]:
def find_common_and_extra_objects(benchmark_strings, model_strings):


    benchmark_subjects = set(s.split()[2] for s in benchmark_strings)
    model_subjects     = set(s.split()[2] for s in model_strings)


    common = list(benchmark_subjects & model_subjects)
    extra  = list(model_subjects - benchmark_subjects)


    precision = len(common) / len(model_subjects) if model_subjects else 0
    recall    = len(common) / len(benchmark_subjects) if benchmark_subjects else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0

    return common, extra, precision, recall, f1


In [ ]:
def find_common_and_extra_subjects(benchmark_strings, model_strings):

    benchmark_subjects = set(s.split()[0] for s in benchmark_strings)
    model_subjects     = set(s.split()[0] for s in model_strings)


    common = list(benchmark_subjects & model_subjects)
    extra  = list(model_subjects - benchmark_subjects)


    precision = len(common) / len(model_subjects) if model_subjects else 0
    recall    = len(common) / len(benchmark_subjects) if benchmark_subjects else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0

    return common, extra, precision, recall, f1


In [ ]:
import re

def extract_bool_from_llm(text):
    if text is None:
        return None

    s = str(text).lower()


    matches = re.findall(r"\b(true|false)\b", s)

    if not matches:
        return None


    last = matches[-1]

    return last == "true"


In [ ]:

import re

def extract_triplesf7(response_text):
    if not response_text or not isinstance(response_text, str):
        return []


    response_text = re.sub(r'^\s*\d+\.\s*', '', response_text, flags=re.MULTILINE)
    response_text = re.sub(r'-', '', response_text)
    response_text = re.sub(r'[\[\]\(\)\,\–\—]', ' ', response_text)
    response_text = re.sub(r'\s+', ' ', response_text).strip()


    component = r'(?:<[^>]+>|[a-zA-Z_][a-zA-Z0-9_]*:[a-zA-Z0-9_]+|[a-zA-Z0-9_]+)'
    pattern = rf'({component})\s+({component})\s+({component})'

    matches = re.findall(pattern, response_text)

    cleaned = []
    for s, p, o in matches:

        for var in [s, p, o]:
            if var.startswith('<') and var.endswith('>'):
                var = var[1:-1].split('#')[-1].split('/')[-1]
        s_clean = s[1:-1].split('#')[-1] if s.startswith('<') else s.split(':')[-1]
        p_clean = p[1:-1].split('#')[-1] if p.startswith('<') else p.split(':')[-1]
        o_clean = o[1:-1].split('#')[-1] if o.startswith('<') else o.split(':')[-1]

        cleaned.append(f"{s_clean} {p_clean} {o_clean}")

    return cleaned


In [ ]:
import re

def extract_triplesfsl(response_text):
    if not response_text or not isinstance(response_text, str):
        return []

    response_text = re.sub(r'^\s*\d+\.\s*', '', response_text, flags=re.MULTILINE)
    response_text = re.sub(r'-', '', response_text)
    response_text = re.sub(r'[\[\]\(\)\.\,\–\—]', ' ', response_text)
    response_text = re.sub(r'\s+', ' ', response_text).strip()

    component = r'(?:[a-zA-Z_][a-zA-Z0-9_]*:[a-zA-Z0-9_]+|[a-zA-Z0-9_]+)'
    pattern = rf'\b({component})\s+({component})\s+({component})\b'

    matches = re.findall(pattern, response_text)

    return [f"{s} {p} {o}" for s, p, o in matches]


In [ ]:
def find_common_and_extra_elements_subjectobj_any_order(benchmark_strings, model_strings):

    benchmark_triples = [s.split() for s in benchmark_strings]
    model_triples     = [s.split() for s in model_strings]

    benchmark_pairs = set(frozenset([t[0], t[2]]) for t in benchmark_triples)
    model_pairs     = set(frozenset([t[0], t[2]]) for t in model_triples)


    common = [list(pair) for pair in (benchmark_pairs & model_pairs)]
    extra  = [list(pair) for pair in (model_pairs - benchmark_pairs)]

    precision = len(common) / len(model_pairs) if model_pairs else 0
    recall    = len(common) / len(benchmark_pairs) if benchmark_pairs else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0

    return common, extra, precision, recall, f1


In [ ]:
import re

def extract_triplesf6(response_text):
    if not response_text or not isinstance(response_text, str):
        return []


    response_text = re.sub(r'^\s*\d+\.\s*', '', response_text, flags=re.MULTILINE)
    response_text = re.sub(r'-', '', response_text)
    response_text = re.sub(r'[\[\]\(\)\,\–\—]', ' ', response_text)
    response_text = re.sub(r'\s+', ' ', response_text).strip()


    component = r'(?:<[^>]+>|[a-zA-Z_][a-zA-Z0-9_]*:[a-zA-Z0-9_]+|[a-zA-Z0-9_]+)'
    pattern = rf'({component})\s+({component})\s+({component})'

    matches = re.findall(pattern, response_text)

    cleaned = []
    for s, p, o in matches:

        for var in [s, p, o]:
            if var.startswith('<') and var.endswith('>'):
                var = var[1:-1].split('#')[-1].split('/')[-1]
        s_clean = s[1:-1].split('#')[-1] if s.startswith('<') else s.split(':')[-1]
        p_clean = p[1:-1].split('#')[-1] if p.startswith('<') else p.split(':')[-1]
        o_clean = o[1:-1].split('#')[-1] if o.startswith('<') else o.split(':')[-1]

        cleaned.append(f"{s_clean} {p_clean} {o_clean}")

    return cleaned


In [ ]:
#SELECT REASONING

In [ ]:
queries_type_1_r = [
    {"query": "abnormal_toe_morphology type?", "subject": "abnormal_toe_morphology", "predicate": "type"},
    {"query": "exudate_in_the_colonic_lumen type?", "subject": "exudate_in_the_colonic_lumen", "predicate": "type"},
    {"query": "cerebellar_cysts type?", "subject": "cerebellar_cysts", "predicate": "type"},
    {"query": "sulphadiazine type?", "subject": "sulphadiazine", "predicate": "type"},
    {"query": "colonic_stents type?", "subject": "colonic_stents", "predicate": "type"},
    {"query": "epidermolysis_bullosa_pretibial type?", "subject": "epidermolysis_bullosa_pretibial", "predicate": "type"},
    {"query": "muscular_dystrophy_congenital_1c type?", "subject": "muscular_dystrophy_congenital_1c", "predicate": "type"},
    {"query": "nilotinib type?", "subject": "nilotinib", "predicate": "type"},
    {"query": "ascorbic_acid_collagen_hydrolyzed type?", "subject": "ascorbic_acid_collagen_hydrolyzed", "predicate": "type"},
    {"query": "locomotion_involved_in_locomotory_behavior type?", "subject": "locomotion_involved_in_locomotory_behavior", "predicate": "type"},
    {"query": "urogenital_system_development type?", "subject": "urogenital_system_development", "predicate": "type"},
    {"query": "hydrocerin_topical_cream type?", "subject": "hydrocerin_topical_cream", "predicate": "type"},
    {"query": "superior_frontal_sulcus_human_only type?", "subject": "superior_frontal_sulcus_human_only", "predicate": "type"},
    {"query": "polyvinyl_alcohol type?", "subject": "polyvinyl_alcohol", "predicate": "type"},
    {"query": "sulcal_segment_of_inferior_frontal_gyrus type?", "subject": "sulcal_segment_of_inferior_frontal_gyrus", "predicate": "type"},
    {"query": "carrington_incontinence_skin_kit type?", "subject": "carrington_incontinence_skin_kit", "predicate": "type"},
    {"query": "cleansers_skin type?", "subject": "cleansers_skin", "predicate": "type"},
    {"query": "durapatite type?", "subject": "durapatite", "predicate": "type"},
    {"query": "science_of_anatomy type?", "subject": "science_of_anatomy", "predicate": "type"},
    {"query": "escherichia_coli type?", "subject": "escherichia_coli", "predicate": "type"},
    {"query": "lactobacillus_cap_oral_14_lactobacillus_cap_vag_14 type?", "subject": "lactobacillus_cap_oral_14_lactobacillus_cap_vag_14", "predicate": "type"},
    {"query": "quality_of_preparation type?", "subject": "quality_of_preparation", "predicate": "type"},
    {"query": "polypectomy type?", "subject": "polypectomy", "predicate": "type"},
    {"query": "colonic_parasites_diagnosis type?", "subject": "colonic_parasites_diagnosis", "predicate": "type"}
]

In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type18(subject, predicate):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT ?subject ?predicate ?object
    WHERE {{
      umls:{subject} rdf:{predicate} ?object.
      BIND(rdf:{predicate} AS ?predicate)
      BIND(umls:{subject} AS ?subject)
    }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()

        print(ret)
        print()
        extr = extract_triples(ret)

        print(extr)

        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type18(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']



        subject = query_data['subject']
        predicate = query_data['predicate']

        ground_truth_triples = await query_sparql_type18(subject, predicate)
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=query, response_text=response_text)
        print(response_text2)
        clean_text = strip_prefixes(response_text2)
        model_response_triples = extract_triplesf(clean_text)
        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(ground_truth_triples, model_response_triples)
        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)

        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")

    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results


    return accuracy_results, mean_metrics, response_results





In [ ]:
async def compute_accuracy_for_query_type18(queries, prompt_level, local_search, query_name, model_name, prompt_name):
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    all_outputs = []

    for i, query_data in enumerate(queries):
        query = query_data['query']
        subject = query_data['subject']
        predicate = query_data['predicate']

        ground_truth_triples = await query_sparql_type18(subject, predicate)

        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response


        print(response_text)

        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=query, response_text=response_text)


        print(response_text2)

        clean_text = strip_prefixes(response_text2)
        model_response_triples = extract_triplesf(clean_text)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(
            ground_truth_triples, model_response_triples
        )

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)

        mean_metrics = {
            'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
            'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
            'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
        }

        safe_query_name = query_name.replace(" ", "_")
        safe_model_name = model_name.replace(" ", "_")
        safe_prompt_name = prompt_name.replace(" ", "_")

        safe_subject = subject.replace(":", "_")
        safe_predicate = predicate.replace(":", "_")

        filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}-sub{safe_subject}-pred{safe_predicate}-{i}.json"
        filepath = os.path.join(output_folder, filename)

        output_data = {
            "query": query,
            "query_name": query_name,
            "model_name": model_name,
            "prompt_name": prompt_name,
            "subject": subject,
            "predicate": predicate,
            "benchmark_triples": list(ground_truth_triples),
            "llm_triples": list(model_response_triples),
            "common_triples": common,
            "extra_triples": extra,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,
            "accuracy_results": accuracy_results,
            "mean_metrics": mean_metrics,
            "responses": response_results
        }

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(output_data, f, indent=4, ensure_ascii=False)

        print(f"Results saved to {filepath}")

        all_outputs.append(output_data)

        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print("--------")

    return accuracy_results, mean_metrics, response_results, all_outputs

In [ ]:
async def compute_accuracy_for_query_type18(queries, prompt_level, local_search, query_name, model_name, prompt_name):

    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    all_query_outputs = []

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']
        predicate = query_data['predicate']

        ground_truth_triples = await query_sparql_type18(subject, predicate)

        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response

        print(response_text)

        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=query, response_text=response_text)

        print(response_text2)

        clean_text = strip_prefixes(response_text2)
        model_response_triples = extract_triplesf(clean_text)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(
            ground_truth_triples, model_response_triples
        )


        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        query_output_data = {
            "query": query,
            "subject": subject,
            "predicate": predicate,
            "benchmark_triples": list(ground_truth_triples),
            "llm_triples": list(model_response_triples),
            "common_triples": common,
            "extra_triples": extra,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,
            "response": response_text
        }

        all_query_outputs.append(query_output_data)

        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "mean_metrics": mean_metrics,
        "accuracy_results": accuracy_results,
        "all_results": all_query_outputs
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")

    return accuracy_results, mean_metrics, response_results

In [ ]:
queries_type_2_r = [
    {"query": "which entities have type anatomical_abnormality?", "object": "anatomical_abnormality"},
    {"query": "which entities have type body_substance?", "object": "body_substance"},
    {"query": "which entities have type acquired_abnormality?", "object": "acquired_abnormality"},
    {"query": "which entities have type pharmacologic_substance?", "object": "pharmacologic_substance"},
    {"query": "which entities have type medical_device?", "object": "medical_device"},
    {"query": "which entities have type congenital_abnormality?", "object": "congenital_abnormality"},
    {"query": "which entities have type disease_or_syndrome?", "object": "disease_or_syndrome"},
    {"query": "which entities have type amino_acid_peptide_or_protein?", "object": "amino_acid_peptide_or_protein"},
    {"query": "which entities have type pharmacologic_substance?", "object": "pharmacologic_substance"},
    {"query": "which entities have type biologic_function?", "object": "biologic_function"},
    {"query": "which entities have type organ_or_tissue_function?", "object": "organ_or_tissue_function"},
    {"query": "which entities have type clinical_drug?", "object": "clinical_drug"},
    {"query": "which entities have type body_space_or_junction?", "object": "body_space_or_junction"},
    {"query": "which entities have type biomedical_or_dental_material?", "object": "biomedical_or_dental_material"},
    {"query": "which entities have type mls:body_location_or_region?", "object": "mls:body_location_or_region"},
    {"query": "which entities have type drug_delivery_device?", "object": "drug_delivery_device"},
    {"query": "which entities have type biomedical_or_dental_material?", "object": "biomedical_or_dental_material"},
    {"query": "which entities have type biomedical_occupation_or_discipline?", "object": "biomedical_occupation_or_discipline"},
    {"query": "which entities have type neoplastic_process?", "object": "neoplastic_process"},
    {"query": "which entities have type organism?", "object": "organism"},
    {"query": "which entities have type manufactured_object?", "object": "manufactured_object"},
    {"query": "which entities have type idea_or_concept?", "object": "idea_or_concept"},
    {"query": "which entities have type therapeutic_or_preventive_procedure?", "object": "therapeutic_or_preventive_procedure"},
    {"query": "which entities have type eukaryote?", "object": "eukaryote"}
]

In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type19(objectf, predicate):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT ?subject ?predicate ?object
    WHERE {{
       ?subject rdf:type umls:{objectf}
      BIND(rdf:type AS ?predicate)
      BIND(umls:{objectf} AS ?object)
    }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print(ret)
        print()
        extr = extract_triples(ret)

        print(extr)
        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type19(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']



        subject = query_data['object']
        predicate = "type"

        ground_truth_triples = await query_sparql_type19(subject, predicate)
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modified(query=query, response_text=response_text)
        print(response_text2)
        clean_text = strip_prefixes(response_text2)
        model_response_triples = extract_triplesf(clean_text)
        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(ground_truth_triples, model_response_triples)
        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }


    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results


    return accuracy_results, mean_metrics, response_results





In [ ]:
async def compute_accuracy_for_query_type19(queries, prompt_level, local_search, query_name, model_name, prompt_name):

    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    all_query_outputs = []

    for query_data in queries:
        query = query_data['query']

        subject = query_data['object']
        predicate = "type"

        ground_truth_triples = await query_sparql_type19(subject, predicate)

        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response


        print(response_text)

        response_text2 = await infer_rdfs_triples_simple_modified(query=query, response_text=response_text)

        print(response_text2)

        clean_text = strip_prefixes(response_text2)
        model_response_triples = extract_triplesf(clean_text)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(
            ground_truth_triples, model_response_triples
        )


        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        query_output_data = {
            "query": query,
            "subject": subject,
            "predicate": predicate,
            "benchmark_triples": list(ground_truth_triples),
            "llm_triples": list(model_response_triples),
            "common_triples": common,
            "extra_triples": extra,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,
            "response": response_text
        }

        all_query_outputs.append(query_output_data)

        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "mean_metrics": mean_metrics,
        "accuracy_results": accuracy_results,
        "all_results": all_query_outputs
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")

    return accuracy_results, mean_metrics, response_results

In [ ]:
queries_type_3_r = [
    {"query": "What more specific categories fall under anatomical_structure?", "object": "anatomical_structure"},
    {"query": "What more specific categories fall under substance?", "object": "substance"},
    {"query": "What more specific categories fall under anatomical_abnormality?", "object": "anatomical_abnormality"},
    {"query": "What more specific categories fall under finding?", "object": "finding"},
    {"query": "What more specific categories fall under physical_object?", "object": "physical_object"},
    {"query": "What more specific categories fall under disease_or_syndrome?", "object": "disease_or_syndrome"},
    {"query": "What more specific categories fall under chemical_viewed_functionally?", "object": "chemical_viewed_functionally"},
    {"query": "What more specific categories fall under event?", "object": "event"},
    {"query": "What more specific categories fall under biologic_function?", "object": "biologic_function"},
    {"query": "What more specific categories fall under body_location_or_region?", "object": "body_location_or_region"},
    {"query": "What more specific categories fall under manufactured_object?", "object": "manufactured_object"},
    {"query": "What more specific categories fall under occupation_or_discipline?", "object": "occupation_or_discipline"},
    {"query": "What more specific categories fall under pathologic_function?", "object": "pathologic_function"},
    {"query": "What more specific categories fall under behavior?", "object": "behavior"},
    {"query": "What more specific categories fall under group?", "object": "group"},
    {"query": "What more specific categories fall under health_care_activity?", "object": "health_care_activity"},
    {"query": "What more specific categories fall under organism?", "object": "organism"},
     {"query": "What more specific categories fall under phenomenon_or_process?", "object": "phenomenon_or_process"},
     {"query": "What more specific categories fall under organization?", "object": "organization"},
      {"query": "What more specific categories fall under molecular_function?", "object": "molecular_function"},
          {"query": "What more specific categories fall under vertebrate?", "object": "vertebrate"},
       {"query": "What more specific categories fall under mammal?", "object": "mammal"},
           {"query": "What more specific categories fall under animal?", "object": "animal"},
               {"query": "What more specific categories fall under intellectual_product?", "object": "intellectual_product"},
               {"query": "What more specific categories fall under conceptual_entity?", "object": "conceptual_entity"},
]

In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type20(objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT  ?subject ?predicate ?object
    WHERE {{
      ?subject rdfs:subClassOf umls:{objectf} .
      BIND(rdfs:subClassOf AS ?predicate)
      BIND(umls:{objectf} AS ?object)
}}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()

        print(ret)
        print()
        extr = extract_triples(ret)

        print(extr)

        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type20(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']



        subject = query_data['object']


        ground_truth_triples = await query_sparql_type20(subject)


        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print("resp etxt")
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=query, response_text=response_text)

        print(response_text2)
        clean_text = strip_prefixes(response_text2)
        model_response_triples = extract_triplesf7(clean_text)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements_subjectobj_any_order(ground_truth_triples, model_response_triples)

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results


    return accuracy_results, mean_metrics, response_results





In [ ]:
async def compute_accuracy_for_query_type20(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):

    accuracy_results = {'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    all_query_outputs = []

    for query_data in queries:
        query = query_data['query']
        objectf = query_data['object']

        ground_truth_triples = await query_sparql_type20(objectf)

        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response


        print(response_text)

        response_text2 = await infer_rdfs_triples_simple_modifiedr(
            query=query,
            response_text=response_text
        )


        print(response_text2)

        clean_text = strip_prefixes(response_text2)
        model_response_triples = extract_triplesf7(clean_text)

        common, extra, precision, recall, f1_score = \
            find_common_and_extra_elements_subjectobj_any_order(
                ground_truth_triples,
                model_response_triples
            )


        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)

        response_results['results'].append({
            "query": query,
            "response": response_text,
            "response_after_inference": response_text2
        })


        all_query_outputs.append({
            "query": query,
            "object": objectf,
            "benchmark_triples": list(ground_truth_triples),
            "llm_triples": list(model_response_triples),
            "common_triples": common,
            "extra_triples": extra,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,
            "response": response_text,
            "response_after_inference": response_text2
        })

        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print("--------")


    mean_metrics = {
        'precision': float(np.mean(accuracy_results['precision'])) if accuracy_results['precision'] else 0,
        'recall': float(np.mean(accuracy_results['recall'])) if accuracy_results['recall'] else 0,
        'f1_score': float(np.mean(accuracy_results['f1_score'])) if accuracy_results['f1_score'] else 0,
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_results": all_query_outputs,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")

    return accuracy_results, mean_metrics, response_results

In [ ]:
queries_type_4_r = [
    {"query": "What broader categories does anatomical_abnormality belong to?", "subject": "anatomical_abnormality"},
    {"query": "What broader categories does body_substance belong to?", "subject": "body_substance"},
    {"query": "What broader categories does acquired_abnormality belong to?", "subject": "acquired_abnormality"},
    {"query": "What broader categories does pharmacologic_substance belong to?", "subject": "pharmacologic_substance"},
    {"query": "What broader categories does medical_device belong to?", "subject": "medical_device"},
    {"query": "What broader categories does congenital_abnormality belong to?", "subject": "congenital_abnormality"},
    {"query": "What broader categories does disease_or_syndrome belong to?", "subject": "disease_or_syndrome"},
    {"query": "What broader categories does amino_acid_peptide_or_protein belong to?", "subject": "amino_acid_peptide_or_protein"},
    {"query": "What broader categories does biologic_function belong to?", "subject": "biologic_function"},
    {"query": "What broader categories does organ_or_tissue_function belong to?", "subject": "organ_or_tissue_function"},
    {"query": "What broader categories does clinical_drug belong to?", "subject": "clinical_drug"},
    {"query": "What broader categories does body_space_or_junction belong to?", "subject": "body_space_or_junction"},
    {"query": "What broader categories does biomedical_or_dental_material belong to?", "subject": "biomedical_or_dental_material"},
    {"query": "What broader categories does body_location_or_region belong to?", "subject": "mls:body_location_or_region"},
    {"query": "What broader categories does drug_delivery_device belong to?", "subject": "drug_delivery_device"},
    {"query": "What broader categories does biomedical_occupation_or_discipline belong to?", "subject": "biomedical_occupation_or_discipline"},
    {"query": "What broader categories does neoplastic_process belong to?", "subject": "neoplastic_process"},
    {"query": "What broader categories does organism belong to?", "subject": "organism"},
    {"query": "What broader categories does manufactured_object belong to?", "subject": "manufactured_object"},
    {"query": "What broader categories does idea_or_concept belong to?", "subject": "idea_or_concept"},
    {"query": "What broader categories does therapeutic_or_preventive_procedure belong to?", "subject": "therapeutic_or_preventive_procedure"},
    {"query": "What broader categories does eukaryote belong to?", "subject": "eukaryote"},
     {"query": "What broader categories does food belong to?", "subject": "food"},
    {"query": "What broader categories does research_device belong to?", "subject": "research_device"},
        {"query": "What broader categories does tissue belong to?", "subject": "tissue"}
]


In [ ]:

import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type21(subject):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>


    SELECT ?subject ?predicate ?object
    WHERE {{
      umls:{subject} rdfs:subClassOf ?object .
      BIND(umls:{subject} AS ?subject)
      BIND(rdfs:subClassOf AS ?predicate)

}}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()

        print(ret)
        print()
        extr = extract_triples(ret)
        print(extr)
        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type21(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']
        ground_truth_triples = await query_sparql_type21(subject)

        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=query, response_text=response_text)

        print(response_text2)
        clean_text = strip_prefixes(response_text2)

        print(clean_text)
        model_response_triples = extract_triplesf6(clean_text)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements_object_only_strings(ground_truth_triples, model_response_triples)

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results


    return accuracy_results, mean_metrics, response_results





In [ ]:
async def compute_accuracy_for_query_type21(queries, prompt_level, local_search, query_name, model_name, prompt_name):
    accuracy_results = {'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    all_query_outputs = []

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']

        ground_truth_triples = await query_sparql_type21(subject)

        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response

        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=query, response_text=response_text)

        clean_text = strip_prefixes(response_text2)
        model_response_triples = extract_triplesf6(clean_text)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements_object_only_strings(
            ground_truth_triples, model_response_triples
        )

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append({
            "query": query,
            "response": response_text,
            "response_after_inference": response_text2
        })

        all_query_outputs.append({
            "query": query,
            "subject": subject,
            "benchmark_triples": list(ground_truth_triples),
            "llm_triples": list(model_response_triples),
            "common_triples": common,
            "extra_triples": extra,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,
            "response": response_text,
            "response_after_inference": response_text2
        })

    mean_metrics = {
        'precision': float(np.mean(accuracy_results['precision'])) if accuracy_results['precision'] else 0,
        'recall': float(np.mean(accuracy_results['recall'])) if accuracy_results['recall'] else 0,
        'f1_score': float(np.mean(accuracy_results['f1_score'])) if accuracy_results['f1_score'] else 0,
    }

    filename = f"query{query_name.replace(' ','_')}-model{model_name.replace(' ','_')}-prompt{prompt_name.replace(' ','_')}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_results": all_query_outputs,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results

In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type22(subject):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT  ?subject ?predicate ?object
    WHERE {{
      umls:{subject} rdfs:subPropertyOf ?object
      BIND(rdfs:subPropertyOf AS ?predicate)
      BIND(umls:{subject} AS ?subject)


}}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print("ret is")
        print(ret)
        print()
        extr = extract_triples(ret)
        print("extr")
        print(extr)
        print("final sp")
        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type22(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']

        llm = query_data['llm']

        subject = query_data['subject']
        predicate = "subPropertyOf"

        ground_truth_triples = await query_sparql_type22(subject)

        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=llm, response_text=response_text)
        print(response_text2)
        clean_text = strip_prefixes(response_text2)
        print(clean_text)
        model_response_triples = extract_triplesf7(clean_text)

        print(model_response_triples)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(ground_truth_triples, model_response_triples)



        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }


    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results


    return accuracy_results, mean_metrics, response_results





In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type23(objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT  ?subject ?predicate ?object
    WHERE {{
      ?subject rdfs:subPropertyOf umls:{objectf}
      BIND(rdfs:subPropertyOf AS ?predicate)
      BIND(umls:{objectf} AS ?object)


}}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print("ret is")
        print(ret)
        print()
        extr = extract_triples(ret)
        print("extr")
        print(extr)
        print("final sp")
        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type23(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']

        llm = query_data['llm']

        subject = query_data['object']
        predicate = "subPropertyOf"

        ground_truth_triples = await query_sparql_type23(subject)
        response = await local_search.asearch(query=llm, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=query, response_text=response_text)
        print(response_text2)
        clean_text = strip_prefixes(response_text2)
        print(clean_text)
        model_response_triples = extract_triplesf7(clean_text)

        print(model_response_triples)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(ground_truth_triples, model_response_triples)



        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results


    return accuracy_results, mean_metrics, response_results





In [ ]:
queries_type_6_r = [
    {"query": "What broader property does `adjacent_to` belong to?", "subject": "adjacent_to", "llm": "subpropertyof adjacent_to?"},
    {"query": "What broader property does `affects` belong to?", "subject": "affects", "llm": "subpropertyof affects?"},
    {"query": "What broader property does `branch_of` belong to?", "subject": "branch_of", "llm": "subpropertyof branch_of?"},
    {"query": "What broader property does `conceptual_part_of` belong to?", "subject": "conceptual_part_of", "llm": "subpropertyof conceptual_part_of?"},
    {"query": "What broader property does `conceptually_related_to` belong to?", "subject": "conceptually_related_to", "llm": "subpropertyof conceptually_related_to?"},
    {"query": "What broader property does `connected_to` belong to?", "subject": "connected_to", "llm": "subpropertyof connected_to?"},
    {"query": "What broader property does `consists_of` belong to?", "subject": "consists_of", "llm": "subpropertyof consists_of?"},
    {"query": "What broader property does `contains` belong to?", "subject": "contains", "llm": "subpropertyof contains?"},
    {"query": "What broader property does `part_of` belong to?", "subject": "part_of", "llm": "subpropertyof part_of?"},
    {"query": "What broader property does `physically_related_to` belong to?", "subject": "physically_related_to", "llm": "subpropertyof physically_related_to?"},
    {"query": "What broader property does `spatially_related_to` belong to?", "subject": "spatially_related_to", "llm": "subpropertyof spatially_related_to?"},
    {"query": "What broader property does `surrounds` belong to?", "subject": "surrounds", "llm": "subpropertyof surrounds?"},
    {"query": "What broader property does `treats` belong to?", "subject": "treats", "llm": "subpropertyof treats?"},
    {"query": "What broader property does `tributary_of` belong to?", "subject": "tributary_of", "llm": "subpropertyof tributary_of?"},
    {"query": "What broader property does `exhibits` belong to?", "subject": "exhibits", "llm": "subpropertyof exhibits?"},
    {"query": "What broader property does `functionally_related_to` belong to?", "subject": "functionally_related_to", "llm": "subpropertyof functionally_related_to?"},
    {"query": "What broader property does `ingredient_of` belong to?", "subject": "ingredient_of", "llm": "subpropertyof ingredient_of?"},
    {"query": "What broader property does `location_of` belong to?", "subject": "location_of", "llm": "subpropertyof location_of?"},
    {"query": "What broader property does `occurs_in` belong to?", "subject": "occurs_in", "llm": "subpropertyof occurs_in?"},
    {"query": "What broader property does `associated_with` belong to?", "subject": "associated_with", "llm": "subpropertyof associated_with?"},
    {"query": "What broader property does `method_of` belong to?", "subject": "method_of", "llm": "subpropertyof method_of?"},
    {"query": "What broader property does `measures` belong to?", "subject": "measures", "llm": "subpropertyof measures?"},
    {"query": "What broader property does `manifestation_of` belong to?", "subject": "manifestation_of", "llm": "subpropertyof manifestation_of?"},
    {"query": "What broader property does `evaluation_of` belong to?", "subject": "evaluation_of", "llm": "subpropertyof evaluation_of?"},
    {"query": "What broader property does `diagnoses` belong to?", "subject": "diagnoses", "llm": "subpropertyof diagnoses?"}
]


In [ ]:
async def compute_accuracy_for_query_type22(queries, prompt_level, local_search, query_name, model_name, prompt_name):
    accuracy_results = {'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    all_query_outputs = []

    for query_data in queries:
        query = query_data['query']
        llm = query_data['llm']
        subject = query_data['subject']
        predicate = "subPropertyOf"

        ground_truth_triples = await query_sparql_type22(subject)

        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response

        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=llm, response_text=response_text)

        clean_text = strip_prefixes(response_text2)
        model_response_triples = extract_triplesf7(clean_text)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(
            ground_truth_triples, model_response_triples
        )

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append({
            "query": query,
            "response": response_text,
            "response_after_inference": response_text2
        })

        all_query_outputs.append({
            "query": query,
            "subject": subject,
            "predicate": predicate,
            "benchmark_triples": list(ground_truth_triples),
            "llm_triples": list(model_response_triples),
            "common_triples": common,
            "extra_triples": extra,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,
            "response": response_text,
            "response_after_inference": response_text2
        })

    mean_metrics = {
        'precision': float(np.mean(accuracy_results['precision'])) if accuracy_results['precision'] else 0,
        'recall': float(np.mean(accuracy_results['recall'])) if accuracy_results['recall'] else 0,
        'f1_score': float(np.mean(accuracy_results['f1_score'])) if accuracy_results['f1_score'] else 0,
    }

    filename = f"query{query_name.replace(' ','_')}-model{model_name.replace(' ','_')}-prompt{prompt_name.replace(' ','_')}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_results": all_query_outputs,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results

In [ ]:
async def compute_accuracy_for_query_type23(queries, prompt_level, local_search, query_name, model_name, prompt_name):
    accuracy_results = {'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    all_query_outputs = []

    for query_data in queries:
        query = query_data['query']
        llm = query_data['llm']
        subject = query_data['object']
        predicate = "subPropertyOf"

        ground_truth_triples = await query_sparql_type23(subject)

        response = await local_search.asearch(query=llm, search_prompt=prompt_level)
        response_text = response.response

        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=query, response_text=response_text)

        clean_text = strip_prefixes(response_text2)
        model_response_triples = extract_triplesf7(clean_text)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(
            ground_truth_triples, model_response_triples
        )

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append({
            "query": query,
            "response": response_text,
            "response_after_inference": response_text2
        })

        all_query_outputs.append({
            "query": query,
            "subject": subject,
            "predicate": predicate,
            "benchmark_triples": list(ground_truth_triples),
            "llm_triples": list(model_response_triples),
            "common_triples": common,
            "extra_triples": extra,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,
            "response": response_text,
            "response_after_inference": response_text2
        })

    mean_metrics = {
        'precision': float(np.mean(accuracy_results['precision'])) if accuracy_results['precision'] else 0,
        'recall': float(np.mean(accuracy_results['recall'])) if accuracy_results['recall'] else 0,
        'f1_score': float(np.mean(accuracy_results['f1_score'])) if accuracy_results['f1_score'] else 0,
    }

    filename = f"query{query_name.replace(' ','_')}-model{model_name.replace(' ','_')}-prompt{prompt_name.replace(' ','_')}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_results": all_query_outputs,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results

In [ ]:
queries_type_6b_r = [
  {
    "query": "Which properties are subproperties of adjacent_to?",
    "object": "adjacent_to",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of adjacent_to?"
  },
  {
    "query": "Which properties are subproperties of affects?",
    "object": "affects",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of affects?"
  },
  {
    "query": "Which properties are subproperties of branch_of?",
    "object": "branch_of",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of branch_of?"
  },
  {
    "query": "Which properties are subproperties of conceptual_part_of?",
    "object": "conceptual_part_of",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of conceptual_part_of?"
  },
  {
    "query": "Which properties are subproperties of conceptually_related_to?",
    "object": "conceptually_related_to",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of conceptually_related_to?"
  },
  {
    "query": "Which properties are subproperties of connected_to?",
    "object": "connected_to",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of connected_to?"
  },
  {
    "query": "Which properties are subproperties of consists_of?",
    "object": "consists_of",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of consists_of?"
  },
  {
    "query": "Which properties are subproperties of contains?",
    "object": "contains",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of contains?"
  },
  {
    "query": "Which properties are subproperties of part_of?",
    "object": "part_of",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of part_of?"
  },
  {
    "query": "Which properties are subproperties of physically_related_to?",
    "object": "physically_related_to",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of physically_related_to?"
  },
  {
    "query": "Which properties are subproperties of spatially_related_to?",
    "object": "spatially_related_to",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of spatially_related_to?"
  },
  {
    "query": "Which properties are subproperties of surrounds?",
    "object": "surrounds",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of surrounds?"
  },
  {
    "query": "Which properties are subproperties of treats?",
    "object": "treats",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of treats?"
  },
  {
    "query": "Which properties are subproperties of tributary_of?",
    "object": "tributary_of",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of tributary_of?"
  },
  {
    "query": "Which properties are subproperties of exhibits?",
    "object": "exhibits",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of exhibits?"
  },
  {
    "query": "Which properties are subproperties of functionally_related_to?",
    "object": "functionally_related_to",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of functionally_related_to?"
  },
  {
    "query": "Which properties are subproperties of ingredient_of?",
    "object": "ingredient_of",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of ingredient_of?"
  },
  {
    "query": "Which properties are subproperties of location_of?",
    "object": "location_of",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of location_of?"
  },
  {
    "query": "Which properties are subproperties of occurs_in?",
    "object": "occurs_in",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of occurs_in?"
  },
  {
    "query": "Which properties are subproperties of associated_with?",
    "object": "associated_with",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of associated_with?"
  },
  {
    "query": "Which properties are subproperties of method_of?",
    "object": "method_of",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of method_of?"
  },
  {
    "query": "Which properties are subproperties of measures?",
    "object": "measures",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of measures?"
  },
  {
    "query": "Which properties are subproperties of manifestation_of?",
    "object": "manifestation_of",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of manifestation_of?"
  },
  {
    "query": "Which properties are subproperties of evaluation_of?",
    "object": "evaluation_of",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of evaluation_of?"
  },
  {
    "query": "Which properties are subproperties of diagnoses?",
    "object": "diagnoses",
    "predicate": "subproperty_of",
    "llm": "entries having subproperty_of diagnoses?"
  }
]


In [ ]:
queries_type_7_r = [
  {
    "query": "entities connected by diagnoses relationship?",
    "predicate": "diagnoses",
    "tipsubiect": "health_care_activity",
    "tipobiect": "disease_or_syndrome",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type health_care_activity that diagnoses instances of type disease_or_syndrome ?"
  },
  {
    "query": "entities connected by diagnoses relationship?",
    "predicate": "diagnoses",
    "tipsubiect": "diagnostic_procedure",
    "tipobiect": "biologic_function",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type diagnostic_procedure that diagnoses instances of type biologic_function ?"
  },
  {
    "query": "entities connected by adjacent_to relationship?",
    "predicate": "adjacent_to",
    "tipsubiect": "anatomical_structure",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type anatomical_structure that adjacent_to instances of type physical_object ?"
  },
  {
    "query": "entities connected by adjacent_to relationship?",
    "predicate": "adjacent_to",
    "tipsubiect": "body_part_organ_or_organ_component",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type body_part_organ_or_organ_component that adjacent_to instances of type physical_object ?"
  },
  {
    "query": "entities connected by treats relationship?",
    "predicate": "treats",
    "tipsubiect": "health_care_activity",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type health_care_activity that treats instances of type physical_object ?"
  },
  {
    "query": "entities connected by treats relationship?",
    "predicate": "treats",
    "tipsubiect": "occupational_activity",
    "tipobiect": "anatomical_structure",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type occupational_activity that treats instances of type anatomical_structure ?"
  },
  {
    "query": "entities connected by ingredient_of relationship?",
    "predicate": "ingredient_of",
    "tipsubiect": "enzyme",
    "tipobiect": "substance",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type enzyme that ingredient_of instances of type substance ?"
  },
  {
    "query": "entities connected by ingredient_of relationship?",
    "predicate": "ingredient_of",
    "tipsubiect": "food",
    "tipobiect": "substance",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type food that ingredient_of instances of type substance ?"
  },
  {
    "query": "entities connected by ingredient_of relationship?",
    "predicate": "ingredient_of",
    "tipsubiect": "physical_object",
    "tipobiect": "inorganic_chemical",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type physical_object that ingredient_of instances of type inorganic_chemical ?"
  },
  {
    "query": "entities connected by occurs_in relationship?",
    "predicate": "occurs_in",
    "tipsubiect": "cell_component",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type cell_component that occurs_in instances of type physical_object ?"
  },
  {
    "query": "entities connected by occurs_in relationship?",
    "predicate": "occurs_in",
    "tipsubiect": "genetic_function",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type genetic_function that occurs_in instances of type physical_object ?"
  },
  {
    "query": "entities connected by affects relationship?",
    "predicate": "affects",
    "tipsubiect": "health_care_activity",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type health_care_activity that affects instances of type physical_object ?"
  },
  {
    "query": "entities connected by affects relationship?",
    "predicate": "affects",
    "tipsubiect": "occupational_activity",
    "tipobiect": "anatomical_structure",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type occupational_activity that affects instances of type anatomical_structure ?"
  },
  {
    "query": "entities connected by location_of relationship?",
    "predicate": "location_of",
    "tipsubiect": "fully_formed_anatomical_structure",
    "tipobiect": "activity",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type fully_formed_anatomical_structure that location_of instances of type activity ?"
  },
  {
    "query": "entities connected by location_of relationship?",
    "predicate": "location_of",
    "tipsubiect": "body_location_or_region",
    "tipobiect": "event",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type body_location_or_region that location_of instances of type event ?"
  },
  {
    "query": "entities connected by method_of relationship?",
    "predicate": "method_of",
    "tipsubiect": "idea_or_concept",
    "tipobiect": "organism_attribute",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type idea_or_concept that method_of instances of type organism_attribute ?"
  },
  {
    "query": "entities connected by method_of relationship?",
    "predicate": "method_of",
    "tipsubiect": "occupational_activity",
    "tipobiect": "clinical_attribute",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type occupational_activity that method_of instances of type clinical_attribute ?"
  },
  {
    "query": "entities connected by evaluation_of relationship?",
    "predicate": "evaluation_of",
    "tipsubiect": "organism_attribute",
    "tipobiect": "pathologic_function",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type organism_attribute that evaluation_of instances of type pathologic_function ?"
  },
  {
    "query": "entities connected by evaluation_of relationship?",
    "predicate": "evaluation_of",
    "tipsubiect": "qualitative_concept",
    "tipobiect": "event",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type qualitative_concept that evaluation_of instances of type event ?"
  },
  {
    "query": "entities connected by connected_to relationship?",
    "predicate": "connected_to",
    "tipsubiect": "cell_component",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type cell_component that connected_to instances of type physical_object ?"
  },
  {
    "query": "entities connected by connected_to relationship?",
    "predicate": "connected_to",
    "tipsubiect": "body_location_or_region",
    "tipobiect": "body_space_or_junction",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type body_location_or_region that connected_to instances of type body_space_or_junction ?"
  },
  {
    "query": "entities connected by tributary_of relationship?",
    "predicate": "tributary_of",
    "tipsubiect": "body_space_or_junction",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type body_space_or_junction that tributary_of instances of type physical_object ?"
  },
  {
    "query": "entities connected by tributary_of relationship?",
    "predicate": "tributary_of",
    "tipsubiect": "physical_object",
    "tipobiect": "conceptual_entity",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type physical_object that tributary_of instances of type conceptual_entity ?"
  },
  {
    "query": "entities connected by branch_of relationship?",
    "predicate": "branch_of",
    "tipsubiect": "physical_object",
    "tipobiect": "body_system",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type physical_object that branch_of instances of type body_system ?"
  },
  {
    "query": "entities connected by branch_of relationship?",
    "predicate": "branch_of",
    "tipsubiect": "body_location_or_region",
    "tipobiect": "Resource",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type body_location_or_region that branch_of instances of type Resource ?"
  },
    {
    "query": "entities connected by location_of relationship?",
    "predicate": "location_of",
    "tipsubiect": "anatomical_structure",
    "tipobiect": "activity",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type anatomical_structure that location_of instances of type activity ?"
  },
   {
    "query": "entities connected by location_of relationship?",
    "predicate": "location_of",
    "tipsubiect": "physical_object",
    "tipobiect": "health_care_activity",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type physical_object that location_of instances of type health_care_activity ?"
  }

]


In [ ]:
async def query_sparql_type24(predicate, tipsubiect, tipobiect):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT DISTINCT ?subject ?predicate ?object
    WHERE {{
      ?subject umls:{predicate} ?object .
      ?subject rdf:type/rdfs:subClassOf* umls:{tipsubiect} .
      ?object rdf:type/rdfs:subClassOf* umls:{tipobiect}.
      BIND(umls:{predicate} AS ?predicate)
      }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print("ret is")
        print(ret)
        print()
        extr = extract_triples(ret)
        print("extr")
        print(extr)
        print("final sp")
        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type24(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']
        llm = query_data['llm']
        tipsubiect = query_data['tipsubiect']
        tipobiect = query_data['tipobiect']

        predicate = query_data['predicate']


        ground_truth_triples = await query_sparql_type24(predicate, tipsubiect, tipobiect)

        print(query)
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print("resp etxt")
        print(response_text)


        clean_text1 = strip_prefixes(response_text)
         # Extract triples from model response
        print("strip1")
        print(clean_text1)
        print("extr trip1")
        model_response_triples1 = extract_triplesf7(clean_text1)
        print("model triple is:")
        print(model_response_triples1)

        clean_triples = await search_by_subjects_location_of_clean_triplesfinal(
            triple_strings=model_response_triples1,
            local_search=local_search,
            prompt_level=prompt_level,
            predicate = predicate
        )
        print("clean triples are")
        print(clean_triples)




        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=llm, response_text=clean_triples)

        print("dupppp")
        print(response_text2)
        clean_text = strip_prefixes(response_text2)

        print("strip")
        print(clean_text)
        print("extr trip")
        model_response_triples = extract_triplesf7(clean_text)

        print(model_response_triples)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(ground_truth_triples, model_response_triples)



        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }


    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results


    return accuracy_results, mean_metrics, response_results


In [ ]:
async def query_sparql_type25(subject, predicate, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT DISTINCT ?subject ?predicate ?object
    WHERE {{
      umls:{subject} umls:{predicate} umls:{objectf}.
      umls:{predicate} rdfs:range ?object.
           BIND(umls:{subject} AS ?subject)
      BIND(rdfs:range AS ?predicate)

      }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print(ret)
        print()
        extr = extract_triples(ret)
        print(extr)
        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type25(queries, prompt_level,  local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']
        llm = query_data['llm']

        predicate = query_data['predicate']
        subject = query_data['subject']
        objectf = query_data['object']
        ground_truth_triples = await query_sparql_type25(subject,predicate, objectf)

        response = await local_search.asearch(query=llm, search_prompt=prompt_level)
        response_text = response.response
        print("resp etxt")
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=query, response_text=response_text)


        print("dupppp")
        print(response_text2)
        clean_text = strip_prefixes(response_text2)

        print("strip")
        print(clean_text)
        print("extr trip")
        model_response_triples = extract_triplesf7(clean_text)


        print(model_response_triples)


        print("ground_truth_triples")
        print(ground_truth_triples)
        print("model_response_triples")
        print(model_response_triples)
        common, extra, precision, recall, f1_score = find_common_and_extra_elements_object_only_strings(ground_truth_triples, model_response_triples)

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)

        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }


    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results


    return accuracy_results, mean_metrics, response_results


In [ ]:
queries_type_12b_r = [
  {
    "query": "posterior_mediastinum contains t8_segment_of_esophagus. What is the most general class of t8_segment_of_esophagus?",
    "subject": "posterior_mediastinum",
    "predicate": "contains",
    "object": "t8_segment_of_esophagus",
    "llm": "entries connected by contains?"
  },
  {
    "query": "right_submandibular_triangle contains right_hyoglossus. What is the most general class of right_hyoglossus?",
    "subject": "right_submandibular_triangle",
    "predicate": "contains",
    "object": "right_hyoglossus",
    "llm": "entries connected by contains?"
  },
  {
    "query": "plasma_membranes surrounds cytoplasms. What is the most general class of cytoplasms?",
    "subject": "plasma_membranes",
    "predicate": "surrounds",
    "object": "cytoplasms",
    "llm": "entries connected by surrounds?"
  },
  {
    "query": "nucleoplasm surrounds nucleolus_cell. What is the most general class of nucleolus_cell?",
    "subject": "nucleoplasm",
    "predicate": "surrounds",
    "object": "nucleolus_cell",
    "llm": "entries connected by surrounds?"
  },
  {
    "query": "sioux_county_ne conceptual_part_of nebraska_geographic_location. What is the most general class of nebraska_geographic_location?",
    "subject": "sioux_county_ne",
    "predicate": "conceptual_part_of",
    "object": "nebraska_geographic_location",
    "llm": "entries connected by conceptual_part_of?"
  },
  {
    "query": "dilation_size conceptual_part_of dilate. What is the most general class of dilate?",
    "subject": "dilation_size",
    "predicate": "conceptual_part_of",
    "object": "dilate",
    "llm": "entries connected by conceptual_part_of?"
  },
  {
    "query": "complications result_of gastrointestinal_endoscopy. What is the most general class of gastrointestinal_endoscopy?",
    "subject": "complications",
    "predicate": "result_of",
    "object": "gastrointestinal_endoscopy",
    "llm": "entries connected by result_of?"
  },
  {
    "query": "absent_left_common_carotid_artery branch_of aortic_arch_structure. What is the most general class of aortic_arch_structure?",
    "subject": "absent_left_common_carotid_artery",
    "predicate": "branch_of",
    "object": "aortic_arch_structure",
    "llm": "entries connected by branch_of?"
  },
  {
    "query": "nerves_sacral branch_of root_of_uwda_hierarchy. What is the most general class of root_of_uwda_hierarchy?",
    "subject": "nerves_sacral",
    "predicate": "branch_of",
    "object": "root_of_uwda_hierarchy",
    "llm": "entries connected by branch_of?"
  },
  {
    "query": "synaptic_signaling occurs_in synapsis. What is the most general class of synapsis?",
    "subject": "synaptic_signaling",
    "predicate": "occurs_in",
    "object": "synapsis",
    "llm": "entries connected by occurs_in?"
  },
  {
    "query": "maintenance_of_protein_localisation_to_organelle occurs_in organelle. What is the most general class of organelle?",
    "subject": "maintenance_of_protein_localisation_to_organelle",
    "predicate": "occurs_in",
    "object": "organelle",
    "llm": "entries connected by occurs_in?"
  },
  {
    "query": "type_iii_no_bleeding_of_gastric_ulcer degree_of type_of_bleeding_of_gastric_ulcer. What is the most general class of type_of_bleeding_of_gastric_ulcer?",
    "subject": "type_iii_no_bleeding_of_gastric_ulcer",
    "predicate": "degree_of",
    "object": "type_of_bleeding_of_gastric_ulcer",
    "llm": "entries connected by degree_of?"
  },
  {
    "query": "type_iii_no_bleeding_of_duodenal_ulcer degree_of type_of_bleeding_of_duodenal_ulcer. What is the most general class of type_of_bleeding_of_duodenal_ulcer?",
    "subject": "type_iii_no_bleeding_of_duodenal_ulcer",
    "predicate": "degree_of",
    "object": "type_of_bleeding_of_duodenal_ulcer",
    "llm": "entries connected by degree_of?"
  },
  {
    "query": "polypectomy uses polypectomy_forceps. What is the most general class of polypectomy_forceps?",
    "subject": "polypectomy",
    "predicate": "uses",
    "object": "polypectomy_forceps",
    "llm": "entries connected by uses?"
  },
  {
    "query": "biopsy_nos uses biopsy_instruments. What is the most general class of biopsy_instruments?",
    "subject": "biopsy_nos",
    "predicate": "uses",
    "object": "biopsy_instruments",
    "llm": "entries connected by uses?"
  },
  {
    "query": "hypertrophic_cicatrix manifestation_of epidermolysis_bullosa_pretibial. What is the most general class of epidermolysis_bullosa_pretibial?",
    "subject": "hypertrophic_cicatrix",
    "predicate": "manifestation_of",
    "object": "epidermolysis_bullosa_pretibial",
    "llm": "entries connected by manifestation_of?"
  },
  {
    "query": "cheloid manifestation_of coli_adenomatous_polyposis. What is the most general class of coli_adenomatous_polyposis?",
    "subject": "cheloid",
    "predicate": "manifestation_of",
    "object": "coli_adenomatous_polyposis",
    "llm": "entries connected by manifestation_of?"
  },
  {
    "query": "howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light measures bodies_howell_jolly. What is the most general class of bodies_howell_jolly?",
    "subject": "howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light",
    "predicate": "measures",
    "object": "bodies_howell_jolly",
    "llm": "entries connected by measures?"
  },
  {
    "query": "chromatins part_of nucleoplasm. What is the most general class of nucleoplasm?",
    "subject": "chromatins",
    "predicate": "part_of",
    "object": "nucleoplasm",
    "llm": "entries connected by part_of?"
  },
  {
    "query": "tracheas adjacent_to oesophagus. What is the most general class of oesophagus?",
    "subject": "tracheas",
    "predicate": "adjacent_to",
    "object": "oesophagus",
    "llm": "entries connected by adjacent_to?"
  },
  {
    "query": "imaging_study_imp_pt_xxx_nar_radiology associated_with imaging_study. What is the most general class of imaging_study?",
    "subject": "imaging_study_imp_pt_xxx_nar_radiology",
    "predicate": "associated_with",
    "object": "imaging_study",
    "llm": "entries connected by associated_with?"
  },
  {
    "query": "sulcal_segment_of_precentral_gyrus adjacent_to structure_of_precentral_sulcus. What is the most general class of structure_of_precentral_sulcus?",
    "subject": "sulcal_segment_of_precentral_gyrus",
    "predicate": "adjacent_to",
    "object": "structure_of_precentral_sulcus",
    "llm": "entries connected by adjacent_to?"
  },
  {
    "query": "lactobacillales ingredient_of lactic_acid_bacteria_oral_pwd. What is the most general class of lactic_acid_bacteria_oral_pwd?",
    "subject": "lactobacillales",
    "predicate": "ingredient_of",
    "object": "lactic_acid_bacteria_oral_pwd",
    "llm": "entries connected by ingredient_of?"
  },
  {
    "query": "chemotactic_factor_inactivator exhibits inactivator_chemotaxis_function. What is the most general class of inactivator_chemotaxis_function?",
    "subject": "chemotactic_factor_inactivator",
    "predicate": "exhibits",
    "object": "inactivator_chemotaxis_function",
    "llm": "entries connected by exhibits?"
  },
  {
    "query": "ampicillins ingredient_of ampicillin_ophthalmic_ointment. What is the most general class of ampicillin_ophthalmic_ointment?",
    "subject": "ampicillins",
    "predicate": "ingredient_of",
    "object": "ampicillin_ophthalmic_ointment",
    "llm": "entries connected by ingredient_of?"
  },
    {
    "query": "polypectomy uses polypectomy_forceps. What is the most general class of polypectomy_forceps?",
    "subject": "polypectomy",
    "predicate": "uses",
    "object": "polypectomy_forceps",
    "llm": "entries connected by uses?"
  },
  {
    "query": "biopsy_nos uses biopsy_instruments. What is the most general class of biopsy_instruments?",
    "subject": "biopsy_nos",
    "predicate": "uses",
    "object": "biopsy_instruments",
    "llm": "entries connected by uses?"
  },
]

In [ ]:
queries_type_12_r = [
  {
    "query": "posterior_mediastinum contains t8_segment_of_esophagus. What is posterior_mediastinum?",
    "subject": "posterior_mediastinum",
    "predicate": "contains",
    "object": "t8_segment_of_esophagus",
    "llm": "entries connected by contains?"
  },
  {
    "query": "right_submandibular_triangle contains right_hyoglossus. What is right_submandibular_triangle?",
    "subject": "right_submandibular_triangle",
    "predicate": "contains",
    "object": "right_hyoglossus",
    "llm": "entries connected by contains?"
  },
  {
    "query": "plasma_membranes surrounds cytoplasms. What is plasma_membranes?",
    "subject": "plasma_membranes",
    "predicate": "surrounds",
    "object": "cytoplasms",
    "llm": "entries connected by surrounds?"
  },
  {
    "query": "nucleoplasm surrounds nucleolus_cell. What is nucleoplasm?",
    "subject": "nucleoplasm",
    "predicate": "surrounds",
    "object": "nucleolus_cell",
    "llm": "entries connected by surrounds?"
  },
  {
    "query": "sioux_county_ne conceptual_part_of nebraska_geographic_location. What is sioux_county_ne?",
    "subject": "sioux_county_ne",
    "predicate": "conceptual_part_of",
    "object": "nebraska_geographic_location",
    "llm": "entries connected by conceptual_part_of?"
  },
  {
    "query": "dilation_size conceptual_part_of dilate. What is dilation_size?",
    "subject": "dilation_size",
    "predicate": "conceptual_part_of",
    "object": "dilate",
    "llm": "entries connected by conceptual_part_of?"
  },
  {
    "query": "complications result_of gastrointestinal_endoscopy. What is complications?",
    "subject": "complications",
    "predicate": "result_of",
    "object": "gastrointestinal_endoscopy",
    "llm": "entries connected by result_of?"
  },
  {
    "query": "absent_left_common_carotid_artery branch_of aortic_arch_structure. What is absent_left_common_carotid_artery?",
    "subject": "absent_left_common_carotid_artery",
    "predicate": "branch_of",
    "object": "aortic_arch_structure",
    "llm": "entries connected by branch_of?"
  },
  {
    "query": "nerves_sacral branch_of root_of_uwda_hierarchy. What is nerves_sacral?",
    "subject": "nerves_sacral",
    "predicate": "branch_of",
    "object": "root_of_uwda_hierarchy",
    "llm": "entries connected by branch_of?"
  },
  {
    "query": "synaptic_signaling occurs_in synapsis. What is synaptic_signaling?",
    "subject": "synaptic_signaling",
    "predicate": "occurs_in",
    "object": "synapsis",
    "llm": "entries connected by occurs_in?"
  },
  {
    "query": "maintenance_of_protein_localisation_to_organelle occurs_in organelle. What is maintenance_of_protein_localisation_to_organelle?",
    "subject": "maintenance_of_protein_localisation_to_organelle",
    "predicate": "occurs_in",
    "object": "organelle",
    "llm": "entries connected by occurs_in?"
  },
  {
    "query": "type_iii_no_bleeding_of_gastric_ulcer degree_of type_of_bleeding_of_gastric_ulcer. What is type_iii_no_bleeding_of_gastric_ulcer?",
    "subject": "type_iii_no_bleeding_of_gastric_ulcer",
    "predicate": "degree_of",
    "object": "type_of_bleeding_of_gastric_ulcer",
    "llm": "entries connected by degree_of?"
  },
  {
    "query": "type_iii_no_bleeding_of_duodenal_ulcer degree_of type_of_bleeding_of_duodenal_ulcer. What is type_iii_no_bleeding_of_duodenal_ulcer?",
    "subject": "type_iii_no_bleeding_of_duodenal_ulcer",
    "predicate": "degree_of",
    "object": "type_of_bleeding_of_duodenal_ulcer",
    "llm": "entries connected by degree_of?"
  },
  {
    "query": "polypectomy uses polypectomy_forceps. What is polypectomy?",
    "subject": "polypectomy",
    "predicate": "uses",
    "object": "polypectomy_forceps",
    "llm": "entries connected by uses?"
  },
  {
    "query": "biopsy_nos uses biopsy_instruments. What is biopsy_nos?",
    "subject": "biopsy_nos",
    "predicate": "uses",
    "object": "biopsy_instruments",
    "llm": "entries connected by uses?"
  },
  {
    "query": "hypertrophic_cicatrix manifestation_of epidermolysis_bullosa_pretibial. What is hypertrophic_cicatrix?",
    "subject": "hypertrophic_cicatrix",
    "predicate": "manifestation_of",
    "object": "epidermolysis_bullosa_pretibial",
    "llm": "entries connected by manifestation_of?"
  },
  {
    "query": "cheloid manifestation_of coli_adenomatous_polyposis. What is cheloid?",
    "subject": "cheloid",
    "predicate": "manifestation_of",
    "object": "coli_adenomatous_polyposis",
    "llm": "entries connected by manifestation_of?"
  },
  {
    "query": "howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light measures bodies_howell_jolly. What is howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light?",
    "subject": "howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light",
    "predicate": "measures",
    "object": "bodies_howell_jolly",
    "llm": "entries connected by measures?"
  },
  {
    "query": "chromatins part_of nucleoplasm. What is chromatins?",
    "subject": "chromatins",
    "predicate": "part_of",
    "object": "nucleoplasm",
    "llm": "entries connected by part_of?"
  },
  {
    "query": "tracheas adjacent_to oesophagus. What is tracheas?",
    "subject": "tracheas",
    "predicate": "adjacent_to",
    "object": "oesophagus",
    "llm": "entries connected by adjacent_to?"
  },
  {
    "query": "imaging_study_imp_pt_xxx_nar_radiology associated_with imaging_study. What is imaging_study_imp_pt_xxx_nar_radiology?",
    "subject": "imaging_study_imp_pt_xxx_nar_radiology",
    "predicate": "associated_with",
    "object": "imaging_study",
    "llm": "entries connected by associated_with?"
  },
  {
    "query": "sulcal_segment_of_precentral_gyrus adjacent_to structure_of_precentral_sulcus. What is sulcal_segment_of_precentral_gyrus?",
    "subject": "sulcal_segment_of_precentral_gyrus",
    "predicate": "adjacent_to",
    "object": "structure_of_precentral_sulcus",
    "llm": "entries connected by adjacent_to?"
  },
  {
    "query": "lactobacillales ingredient_of lactic_acid_bacteria_oral_pwd. What is lactobacillales?",
    "subject": "lactobacillales",
    "predicate": "ingredient_of",
    "object": "lactic_acid_bacteria_oral_pwd",
    "llm": "entries connected by ingredient_of?"
  },
  {
    "query": "chemotactic_factor_inactivator exhibits inactivator_chemotaxis_function. What is chemotactic_factor_inactivator?",
    "subject": "chemotactic_factor_inactivator",
    "predicate": "exhibits",
    "object": "inactivator_chemotaxis_function",
    "llm": "entries connected by exhibits?"
  },
  {
    "query": "ampicillins ingredient_of ampicillin_ophthalmic_ointment. What is ampicillins?",
    "subject": "ampicillins",
    "predicate": "ingredient_of",
    "object": "ampicillin_ophthalmic_ointment",
    "llm": "entries connected by ingredient_of?"
  }
]


In [ ]:

async def query_sparql_type26(subject, predicate, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT DISTINCT ?subject ?predicate ?object
    WHERE {{
      umls:{subject} umls:{predicate} umls:{objectf}.
      umls:{objectf} rdf:type ?object.
           BIND(umls:{objectf} AS ?subject)
      BIND(rdf:type AS ?predicate)

      }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print("ret is")
        print(ret)
        print()
        extr = extract_triples(ret)
        print("extr")
        print(extr)
        print("final sp")
        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type26(queries, prompt_level,  local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']

        llm = query_data['llm']
        predicate = query_data['predicate']
        subject = query_data['subject']
        objectf = query_data['object']
        ground_truth_triples = await query_sparql_type26(subject,predicate, objectf)

        response = await local_search.asearch(query=llm, search_prompt=prompt_level)
        response_text = response.response
        print("resp etxt")
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=query, response_text=response_text)


        print("dupppp")
        print(response_text2)
        clean_text = strip_prefixes(response_text2)
        print("strip")
        print(clean_text)
        print("extr trip")
        model_response_triples = extract_triplesfsl(clean_text)

        print(model_response_triples)


        print("ground_truth_triples")
        print(ground_truth_triples)
        print("model_response_triples")
        print(model_response_triples)
        common, extra, precision, recall, f1_score = find_common_and_extra_objects(ground_truth_triples, model_response_triples)



        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results


    return accuracy_results, mean_metrics, response_results


In [ ]:

async def query_sparql_type27(subject, predicate, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT DISTINCT ?subject ?predicate ?object
    WHERE {{
      umls:{subject} umls:{predicate} umls:{objectf}.
      umls:{predicate} rdfs:domain ?object.
           BIND(umls:{subject} AS ?subject)

      BIND(rdfs:domain AS ?predicate)

      }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print(ret)
        print()
        extr = extract_triples(ret)
        print(extr)
        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type27(queries, prompt_level,  local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']
        llm = query_data['llm']

        predicate = query_data['predicate']
        subject = query_data['subject']
        objectf = query_data['object']
        ground_truth_triples = await query_sparql_type27(subject,predicate, objectf)

        response = await local_search.asearch(query=llm, search_prompt=prompt_level)
        response_text = response.response
        print("resp etxt")
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=query, response_text=response_text)


        print("dupppp")
        print(response_text2)
        clean_text = strip_prefixes(response_text2)

        print("strip")
        print(clean_text)
        print("extr trip")
        model_response_triples = extract_triplesf7(clean_text)



        print(model_response_triples)


        print("ground_truth_triples")
        print(ground_truth_triples)
        print("model_response_triples")
        print(model_response_triples)
        common, extra, precision, recall, f1_score = find_common_and_extra_elements_object_only_strings(ground_truth_triples, model_response_triples)

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }


    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results


    return accuracy_results, mean_metrics, response_results


In [ ]:
queries_type_13b_r = [
  {
    "query": "tracheas adjacent_to oesophagus. What is the most general class of tracheas?",
    "subject": "tracheas",
    "predicate": "adjacent_to",
    "object": "oesophagus",
    "llm": "entries connected by adjacent_to?"
  },
  {
    "query": "sulcal_segment_of_precentral_gyrus adjacent_to structure_of_precentral_sulcus. What is the most general class of sulcal_segment_of_precentral_gyrus?",
    "subject": "sulcal_segment_of_precentral_gyrus",
    "predicate": "adjacent_to",
    "object": "structure_of_precentral_sulcus",
    "llm": "entries connected by adjacent_to?"
  },
  {
    "query": "plasma_membranes surrounds cytoplasms. What is the most general class of plasma_membranes?",
    "subject": "plasma_membranes",
    "predicate": "surrounds",
    "object": "cytoplasms",
    "llm": "entries connected by surrounds?"
  },
  {
    "query": "nucleoplasm surrounds nucleolus_cell. What is the most general class of nucleoplasm?",
    "subject": "nucleoplasm",
    "predicate": "surrounds",
    "object": "nucleolus_cell",
    "llm": "entries connected by surrounds?"
  },
  {
    "query": "chemotactic_factor_inactivator exhibits inactivator_chemotaxis_function. What is the most general class of chemotactic_factor_inactivator?",
    "subject": "chemotactic_factor_inactivator",
    "predicate": "exhibits",
    "object": "inactivator_chemotaxis_function",
    "llm": "entries connected by exhibits?"
  },
  {
    "query": "synaptic_signaling occurs_in synapsis. What is the most general class of synaptic_signaling?",
    "subject": "synaptic_signaling",
    "predicate": "occurs_in",
    "object": "synapsis",
    "llm": "entries connected by occurs_in?"
  },
  {
    "query": "posterior_mediastinum contains t8_segment_of_esophagus. What is the most general class of posterior_mediastinum?",
    "subject": "posterior_mediastinum",
    "predicate": "contains",
    "object": "t8_segment_of_esophagus",
    "llm": "entries connected by contains?"
  },
  {
    "query": "right_submandibular_triangle contains right_hyoglossus. What is the most general class of right_submandibular_triangle?",
    "subject": "right_submandibular_triangle",
    "predicate": "contains",
    "object": "right_hyoglossus",
    "llm": "entries connected by contains?"
  },

  {
    "query": "complications result_of gastrointestinal_endoscopy. What is the most general class of complications?",
    "subject": "complications",
    "predicate": "result_of",
    "object": "gastrointestinal_endoscopy",
    "llm": "entries connected by result_of?"
  },
  {
    "query": "ampicillins ingredient_of ampicillin_ophthalmic_ointment. What is the most general class of ampicillins?",
    "subject": "ampicillins",
    "predicate": "ingredient_of",
    "object": "ampicillin_ophthalmic_ointment",
    "llm": "entries connected by ingredient_of?"
  },
  {
    "query": "lactobacillales ingredient_of lactic_acid_bacteria_oral_pwd. What is the most general class of lactobacillales?",
    "subject": "lactobacillales",
    "predicate": "ingredient_of",
    "object": "lactic_acid_bacteria_oral_pwd",
    "llm": "entries connected by ingredient_of?"
  },
    {
    "query": "left_temporal_part_of_frontal_bone connected_to left_sphenofrontal_suture. What is the most general class of left_temporal_part_of_frontal_bone?",
    "subject": "left_temporal_part_of_frontal_bone",
    "predicate": "connected_to",
    "object": "left_sphenofrontal_suture",
    "llm": "entries connected by connected_to?"
  },
  {
    "query": "maxillary_process_of_zygomatic_bone connected_to structure_of_zygomaticomaxillary_suture_of_skull. What is the most general class of maxillary_process_of_zygomatic_bone?",
    "subject": "maxillary_process_of_zygomatic_bone",
    "predicate": "connected_to",
    "object": "structure_of_zygomaticomaxillary_suture_of_skull",
    "llm": "entries connected by connected_to?"
  },
  {
    "query": "temporal_part_of_frontal_bone connected_to structure_of_sphenofrontal_suture_of_skull. What is the most general class of temporal_part_of_frontal_bone?",
    "subject": "temporal_part_of_frontal_bone",
    "predicate": "connected_to",
    "object": "structure_of_sphenofrontal_suture_of_skull",
    "llm": "entries connected by connected_to?"
  },
  {
    "query": "type_ib_oozing_arterial_bleeding_of_gastric_ulcer degree_of type_of_bleeding_of_gastric_ulcer. What is the most general class of type_ib_oozing_arterial_bleeding_of_gastric_ulcer?",
    "subject": "type_ib_oozing_arterial_bleeding_of_gastric_ulcer",
    "predicate": "degree_of",
    "object": "type_of_bleeding_of_gastric_ulcer",
    "llm": "entries connected by degree_of?"
  },
  {
    "query": "type_ib_oozing_arterial_bleeding_of_duodenal_ulcer degree_of type_of_bleeding_of_duodenal_ulcer. What is the most general class of type_ib_oozing_arterial_bleeding_of_duodenal_ulcer?",
    "subject": "type_ib_oozing_arterial_bleeding_of_duodenal_ulcer",
    "predicate": "degree_of",
    "object": "type_of_bleeding_of_duodenal_ulcer",
    "llm": "entries connected by degree_of?"
  },
  {
    "query": "escambia_county_al conceptual_part_of alabama. What is the most general class of escambia_county_al?",
    "subject": "escambia_county_al",
    "predicate": "conceptual_part_of",
    "object": "alabama",
    "llm": "entries connected by conceptual_part_of?"
  },
  {
    "query": "marshall_islands conceptual_part_of oceania. What is the most general class of marshall_islands?",
    "subject": "marshall_islands",
    "predicate": "conceptual_part_of",
    "object": "oceania",
    "llm": "entries connected by conceptual_part_of?"
  },
  {
    "query": "complications result_of gastrointestinal_endoscopy. What is the most general class of complications?",
    "subject": "complications",
    "predicate": "result_of",
    "object": "gastrointestinal_endoscopy",
    "llm": "entries connected by result_of?"
  },
  {
    "query": "biopsy_nos uses biopsy_snare. What is the most general class of biopsy_nos?",
    "subject": "biopsy_nos",
    "predicate": "uses",
    "object": "biopsy_snare",
    "llm": "entries connected by uses?"
  },
  {
    "query": "dilate uses non_guided_bougie. What is the most general class of dilate?",
    "subject": "dilate",
    "predicate": "uses",
    "object": "non_guided_bougie",
    "llm": "entries connected by uses?"
  },
  {
    "query": "dilate treats dilated_lesion. What is the most general class of dilate?",
    "subject": "dilate",
    "predicate": "treats",
    "object": "dilated_lesion",
    "llm": "entries connected by treats?"
  },
  {
    "query": "mucosal_resection treats lesion_resected. What is the most general class of mucosal_resection?",
    "subject": "mucosal_resection",
    "predicate": "treats",
    "object": "lesion_resected",
    "llm": "entries connected by treats?"
  },
  {
    "query": "cavity_abdominal contains livers. What is the most general class of cavity_abdominal?",
    "subject": "cavity_abdominal",
    "predicate": "contains",
    "object": "livers",
    "llm": "entries connected by contains?"
  },
  {
    "query": "superior_mediastinum contains t1_segment_of_esophagus. What is the most general class of superior_mediastinum?",
    "subject": "superior_mediastinum",
    "predicate": "contains",
    "object": "t1_segment_of_esophagus",
    "llm": "entries connected by contains?"
  }
]

In [ ]:
queries_type_13r = [
  {
    "query": "tracheas adjacent_to oesophagus. What is tracheas?",
    "subject": "tracheas",
    "predicate": "adjacent_to",
    "object": "oesophagus",
    "llm": "entries connected by adjacent_to?"
  },
  {
    "query": "sulcal_segment_of_precentral_gyrus adjacent_to structure_of_precentral_sulcus. What is sulcal_segment_of_precentral_gyrus?",
    "subject": "sulcal_segment_of_precentral_gyrus",
    "predicate": "adjacent_to",
    "object": "structure_of_precentral_sulcus",
    "llm": "entries connected by adjacent_to?"
  },
  {
    "query": "plasma_membranes surrounds cytoplasms. What is plasma_membranes?",
    "subject": "plasma_membranes",
    "predicate": "surrounds",
    "object": "cytoplasms",
    "llm": "entries connected by surrounds?"
  },
  {
    "query": "nucleoplasm surrounds nucleolus_cell. What is nucleoplasm?",
    "subject": "nucleoplasm",
    "predicate": "surrounds",
    "object": "nucleolus_cell",
    "llm": "entries connected by surrounds?"
  },
  {
    "query": "chemotactic_factor_inactivator exhibits inactivator_chemotaxis_function. What is chemotactic_factor_inactivator?",
    "subject": "chemotactic_factor_inactivator",
    "predicate": "exhibits",
    "object": "inactivator_chemotaxis_function",
    "llm": "entries connected by exhibits?"
  },
  {
    "query": "synaptic_signaling occurs_in synapsis. What is synaptic_signaling?",
    "subject": "synaptic_signaling",
    "predicate": "occurs_in",
    "object": "synapsis",
    "llm": "entries connected by occurs_in?"
  },
  {
    "query": "posterior_mediastinum contains t8_segment_of_esophagus. What is posterior_mediastinum?",
    "subject": "posterior_mediastinum",
    "predicate": "contains",
    "object": "t8_segment_of_esophagus",
    "llm": "entries connected by contains?"
  },
  {
    "query": "right_submandibular_triangle contains right_hyoglossus. What is right_submandibular_triangle?",
    "subject": "right_submandibular_triangle",
    "predicate": "contains",
    "object": "right_hyoglossus",
    "llm": "entries connected by contains?"
  },
  {
    "query": "complications result_of gastrointestinal_endoscopy. What is complications?",
    "subject": "complications",
    "predicate": "result_of",
    "object": "gastrointestinal_endoscopy",
    "llm": "entries connected by result_of?"
  },
  {
    "query": "ampicillins ingredient_of ampicillin_ophthalmic_ointment. What is ampicillins?",
    "subject": "ampicillins",
    "predicate": "ingredient_of",
    "object": "ampicillin_ophthalmic_ointment",
    "llm": "entries connected by ingredient_of?"
  },
  {
    "query": "lactobacillales ingredient_of lactic_acid_bacteria_oral_pwd. What is lactobacillales?",
    "subject": "lactobacillales",
    "predicate": "ingredient_of",
    "object": "lactic_acid_bacteria_oral_pwd",
    "llm": "entries connected by ingredient_of?"
  },
  {
    "query": "left_temporal_part_of_frontal_bone connected_to left_sphenofrontal_suture. What is left_temporal_part_of_frontal_bone?",
    "subject": "left_temporal_part_of_frontal_bone",
    "predicate": "connected_to",
    "object": "left_sphenofrontal_suture",
    "llm": "entries connected by connected_to?"
  },
  {
    "query": "maxillary_process_of_zygomatic_bone connected_to structure_of_zygomaticomaxillary_suture_of_skull. What is maxillary_process_of_zygomatic_bone?",
    "subject": "maxillary_process_of_zygomatic_bone",
    "predicate": "connected_to",
    "object": "structure_of_zygomaticomaxillary_suture_of_skull",
    "llm": "entries connected by connected_to?"
  },
  {
    "query": "temporal_part_of_frontal_bone connected_to structure_of_sphenofrontal_suture_of_skull. What is temporal_part_of_frontal_bone?",
    "subject": "temporal_part_of_frontal_bone",
    "predicate": "connected_to",
    "object": "structure_of_sphenofrontal_suture_of_skull",
    "llm": "entries connected by connected_to?"
  },
  {
    "query": "type_ib_oozing_arterial_bleeding_of_gastric_ulcer degree_of type_of_bleeding_of_gastric_ulcer. What is type_ib_oozing_arterial_bleeding_of_gastric_ulcer?",
    "subject": "type_ib_oozing_arterial_bleeding_of_gastric_ulcer",
    "predicate": "degree_of",
    "object": "type_of_bleeding_of_gastric_ulcer",
    "llm": "entries connected by degree_of?"
  },
  {
    "query": "type_ib_oozing_arterial_bleeding_of_duodenal_ulcer degree_of type_of_bleeding_of_duodenal_ulcer. What is type_ib_oozing_arterial_bleeding_of_duodenal_ulcer?",
    "subject": "type_ib_oozing_arterial_bleeding_of_duodenal_ulcer",
    "predicate": "degree_of",
    "object": "type_of_bleeding_of_duodenal_ulcer",
    "llm": "entries connected by degree_of?"
  },
  {
    "query": "escambia_county_al conceptual_part_of alabama. What is escambia_county_al?",
    "subject": "escambia_county_al",
    "predicate": "conceptual_part_of",
    "object": "alabama",
    "llm": "entries connected by conceptual_part_of?"
  },
  {
    "query": "marshall_islands conceptual_part_of oceania. What is marshall_islands?",
    "subject": "marshall_islands",
    "predicate": "conceptual_part_of",
    "object": "oceania",
    "llm": "entries connected by conceptual_part_of?"
  },
  {
    "query": "biopsy_nos uses biopsy_snare. What is biopsy_nos?",
    "subject": "biopsy_nos",
    "predicate": "uses",
    "object": "biopsy_snare",
    "llm": "entries connected by uses?"
  },
  {
    "query": "dilate uses non_guided_bougie. What is dilate?",
    "subject": "dilate",
    "predicate": "uses",
    "object": "non_guided_bougie",
    "llm": "entries connected by uses?"
  },
  {
    "query": "dilate treats dilated_lesion. What is dilate?",
    "subject": "dilate",
    "predicate": "treats",
    "object": "dilated_lesion",
    "llm": "entries connected by treats?"
  },
  {
    "query": "mucosal_resection treats lesion_resected. What is mucosal_resection?",
    "subject": "mucosal_resection",
    "predicate": "treats",
    "object": "lesion_resected",
    "llm": "entries connected by treats?"
  },
  {
    "query": "cavity_abdominal contains livers. What is cavity_abdominal?",
    "subject": "cavity_abdominal",
    "predicate": "contains",
    "object": "livers",
    "llm": "entries connected by contains?"
  },
  {
    "query": "superior_mediastinum contains t1_segment_of_esophagus. What is superior_mediastinum?",
    "subject": "superior_mediastinum",
    "predicate": "contains",
    "object": "t1_segment_of_esophagus",
    "llm": "entries connected by contains?"
  }
]


In [ ]:

async def query_sparql_type28(subject, predicate, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT DISTINCT ?subject ?predicate ?object
    WHERE {{
      umls:{subject} umls:{predicate} umls:{objectf}.
      umls:{subject} rdf:type ?object.
           BIND(umls:{subject} AS ?subject)
      BIND(rdf:type AS ?predicate)

      }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print(ret)
        print()
        extr = extract_triples(ret)
        print(extr)
        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type28(queries, prompt_level,  local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']
        llm = query_data['llm']

        predicate = query_data['predicate']
        subject = query_data['subject']
        objectf = query_data['object']
        ground_truth_triples = await query_sparql_type28(subject,predicate, objectf)

        response = await local_search.asearch(query=llm, search_prompt=prompt_level)
        response_text = response.response

        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=query, response_text=response_text)


        print(response_text2)
        clean_text = strip_prefixes(response_text2)

        print(clean_text)
        model_response_triples = extract_triplesf7(clean_text)


        print(model_response_triples)


        print("ground_truth_triples")
        print(ground_truth_triples)
        print("model_response_triples")
        print(model_response_triples)
        common, extra, precision, recall, f1_score = find_common_and_extra_objects(ground_truth_triples, model_response_triples)

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }


    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results


    return accuracy_results, mean_metrics, response_results


In [ ]:

#REASONING ASK

In [ ]:


queries_type_A3 = [
    {
        "query": "*Is it true that* the target of diagnoses be pathologic_function?",
        "predicate": "diagnoses",
        "subject": "pathologic_function",
        "llm": "entries connected by diagnoses?"
    },
        {
        "query": "*Is it true that* the target of diagnoses be body_substance?",
        "predicate": "diagnoses",
        "subject": "body_substance",
        "llm": "entries connected by diagnoses?"
    },
      {
        "query": "*Is it true that* the target of adjacent_to be tissue?",
        "predicate": "adjacent_to",
        "subject": "tissue",
        "llm": "entries connected by adjacent_to?"
    },
       {
        "query": "*Is it true that* the target of branch_of be classification?",
        "predicate": "branch_of",
        "subject": "classification",
        "llm": "entries connected by branch_of?"
    },
    {
        "query": "*Is it true that* the target of adjacent_to be body_space_or_junction?",
        "predicate": "adjacent_to",
        "subject": "body_space_or_junction",
        "llm": "entries connected by adjacent_to?"
    },
    {
        "query": "*Is it true that* the target of branch_of be body_location_or_region?",
        "predicate": "branch_of",
        "subject": "body_location_or_region",
        "llm": "entries connected by branch_of?"
    },
    {
        "query": "*Is it true that* the target of conceptual_part_of be nucleotide_sequence?",
        "predicate": "conceptual_part_of",
        "subject": "nucleotide_sequence",
        "llm": "entries connected by conceptual_part_of?"
    },
    {
        "query": "*Is it true that* the target of contains be body_system?",
        "predicate": "contains",
        "subject": "body_system",
        "llm": "entries connected by contains?"
    },
   {
        "query": "*Is it true that* the target of part_of be physical_object?",
        "predicate": "part_of",
        "subject": "physical_object",
        "llm": "entries connected by part_of?"
    },
    {
        "query": "*Is it true that* the target of part_of be activity?",
        "predicate": "part_of",
        "subject": "activity",
        "llm": "entries connected by part_of?"
    },
    {
        "query": "*Is it true that* the target of conceptual_part_of be geographic_area?",
        "predicate": "conceptual_part_of",
        "subject": "geographic_area",
        "llm": "entries connected by conceptual_part_of?"
    },
    {
        "query": "*Is it true that* the target of conceptual_part_of be temporal_concept?",
        "predicate": "conceptual_part_of",
        "subject": "temporal_concept",
        "llm": "entries connected by conceptual_part_of?"
    },
    {
        "query": "*Is it true that* the target of branch_of be fully_formed_anatomical_structure?",
        "predicate": "branch_of",
        "subject": "fully_formed_anatomical_structure",
        "llm": "entries connected by branch_of?"
    },
     {
        "query": "*Is it true that* the target of branch_of be fungus?",
        "predicate": "branch_of",
        "subject": "fungus",
        "llm": "entries connected by branch_of?"
    },
    {
        "query": "*Is it true that* the target of part_of be cell_component?",
        "predicate": "part_of",
        "subject": "cell_component",
        "llm": "entries connected by part_of?"
    },
    {
        "query": "*Is it true that* the target of surrounds be body_system?",
        "predicate": "surrounds",
        "subject": "body_system",
        "llm": "entries connected by surrounds?"
    },

{
        "query": "*Is it true that* the target of exhibits be plant?",
        "predicate": "exhibits",
        "subject": "plant",
        "llm": "entries connected by exhibits?"
    },
        {
        "query": "*Is it true that* the target of surrounds be body_location_or_region?",
        "predicate": "surrounds",
        "subject": "body_location_or_region",
        "llm": "entries connected by surrounds?"
    },

    {
        "query": "*Is it true that* the target of tributary_of be body_part_organ_or_organ_component?",
        "predicate": "tributary_of",
        "subject": "body_part_organ_or_organ_component",
        "llm": "entries connected by tributary_of?"
    },

        {
        "query": "*Is it true that* the target of exhibits be molecular_function?",
        "predicate": "exhibits",
        "subject": "molecular_function",
        "llm": "entries connected by exhibits?"
    },
        {
        "query": "*Is it true that* the target of exhibits be bacterium?",
        "predicate": "exhibits",
        "subject": "bacterium",
        "llm": "entries connected by exhibits?"
    },
    {
        "query": "*Is it true that* the target of exhibits be molecular_function?",
        "predicate": "exhibits",
        "subject": "molecular_function",
        "llm": "entries connected by exhibits?"
    },
    {
        "query": "*Is it true that* the target of ingredient_of be medical_device?",
        "predicate": "ingredient_of",
        "subject": "medical_device",
        "llm": "entries connected by ingredient_of?"
    },
    {
        "query": "*Is it true that* the target of location_of be biomedical_occupation_or_discipline?",
        "predicate": "location_of",
        "subject": "biomedical_occupation_or_discipline",
        "llm": "entries connected by location_of?"
    },
    {
        "query": "*Is it true that* the target of occurs_in be cell_component?",
        "predicate": "occurs_in",
        "subject": "cell_component",
        "llm": "entries connected by occurs_in?"
    }
]


In [ ]:

async def query_sparql_type29(subject, predicate):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>

      ASK {{
        umls:{predicate} rdfs:range ?r .
        umls:{subject} rdfs:subClassOf ?r .
      }}

    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type29(queries, prompt_level,  local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        queryl = query_data['llm']



        subject = query_data['subject']
        predicate = query_data['predicate']
        ground_truth_triples = await query_sparql_type29(subject, predicate)

        response = await local_search.asearch(query=queryl, search_prompt=prompt_level)
        response_text = response.response
        print("resp etxt")
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query, response_text=response_text)


        print("dupppp")
        print(response_text2)

        clean_text = strip_prefixes(response_text2)

        print("strip")
        print(clean_text)
        print("extr trip")


        print("ground_truth_triples")
        print(ground_truth_triples)
        print("model_response_triples")
        print(clean_text)

        gt_bool = normalize_to_bool(ground_truth_triples)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1


        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)




    return x


In [ ]:

queries_type_A4 = [

  {
    "query": "*Is it true that* the source of location_of can be body_substance?",
    "predicate": "location_of",
    "subject": "body_substance",
    "llm": "entries connected by location_of?"
  },
  {
    "query": "*Is it true that* the source of evaluation_of can be pharmacologic_substance?",
    "predicate": "evaluation_of",
    "subject": "pharmacologic_substance",
    "llm": "entries connected by evaluation_of?"
  },
  {
    "query": "*Is it true that* the source of location_of can be medical_device?",
    "predicate": "location_of",
    "subject": "medical_device",
    "llm": "entries connected by location_of?"
  },
  {
    "query": "*Is it true that* the source of manifestation_of can be congenital_abnormality?",
    "predicate": "manifestation_of",
    "subject": "congenital_abnormality",
    "llm": "entries connected by manifestation_of?"
  },
  {
    "query": "*Is it true that* the source of manifestation_of can be disease_or_syndrome?",
    "predicate": "manifestation_of",
    "subject": "disease_or_syndrome",
    "llm": "entries connected by manifestation_of?"
  },
  {
    "query": "*Is it true that* the source of measures can be amino_acid_peptide_or_protein?",
    "predicate": "measures",
    "subject": "amino_acid_peptide_or_protein",
    "llm": "entries connected by measures?"
  },
  {
    "query": "*Is it true that* the source of part_of can be pharmacologic_substance?",
    "predicate": "part_of",
    "subject": "pharmacologic_substance",
    "llm": "entries connected by part_of?"
  },
  {
    "query": "*Is it true that* the source of exhibits can be body_system?",
    "predicate": "exhibits",
    "subject": "body_system",
    "llm": "entries connected by exhibits?"
  },
  {
    "query": "*Is it true that* the source of part_of can be organ_or_tissue_function?",
    "predicate": "part_of",
    "subject": "organ_or_tissue_function",
    "llm": "entries connected by part_of?"
  },
   {
    "query": "*Is it true that* the source of part_of can be plant?",
    "predicate": "part_of",
    "subject": "plant",
    "llm": "entries connected by part_of?"
  },
  {
    "query": "*Is it true that* the source of ingredient_of can be clinical_drug?",
    "predicate": "ingredient_of",
    "subject": "clinical_drug",
    "llm": "entries connected by ingredient_of?"
  },
  {
    "query": "*Is it true that* the source of adjacent_to can be body_space_or_junction?",
    "predicate": "adjacent_to",
    "subject": "body_space_or_junction",
    "llm": "entries connected by adjacent_to?"
  },
    {
    "query": "*Is it true that* the source of adjacent_to can be cell_component?",
    "predicate": "adjacent_to",
    "subject": "cell_component",
    "llm": "entries connected by adjacent_to?"
  },

  {
    "query": "*Is it true that* the source of measures can be biomedical_or_dental_material?",
    "predicate": "measures",
    "subject": "biomedical_or_dental_material",
    "llm": "entries connected by measures?"
  },
  {
    "query": "*Is it true that* the source of surrounds can be body_location_or_region?",
    "predicate": "surrounds",
    "subject": "body_location_or_region",
    "llm": "entries connected by surrounds?"
  },
  {
    "query": "*Is it true that* the source of ingredient_of can be drug_delivery_device?",
    "predicate": "ingredient_of",
    "subject": "drug_delivery_device",
    "llm": "entries connected by ingredient_of?"
  },
  {
    "query": "*Is it true that* the source of ingredient_of can be biomedical_or_dental_material?",
    "predicate": "ingredient_of",
    "subject": "biomedical_or_dental_material",
    "llm": "entries connected by ingredient_of?"
  },
  {
    "query": "*Is it true that* the source of associated_with can be biomedical_occupation_or_discipline?",
    "predicate": "associated_with",
    "subject": "biomedical_occupation_or_discipline",
    "llm": "entries connected by associated_with?"
  },
  {
    "query": "*Is it true that* the source of manifestation_of can be neoplastic_process?",
    "predicate": "manifestation_of",
    "subject": "neoplastic_process",
    "llm": "entries connected by manifestation_of?"
  },
  {
    "query": "*Is it true that* the source of measures can be organism?",
    "predicate": "measures",
    "subject": "organism",
    "llm": "entries connected by measures?"
  },
    {
    "query": "*Is it true that* the source of measures can be event?",
    "predicate": "measures",
    "subject": "event",
    "llm": "entries connected by measures?"
  },
  {
    "query": "*Is it true that* the source of ingredient_of can be manufactured_object?",
    "predicate": "ingredient_of",
    "subject": "manufactured_object",
    "llm": "entries connected by ingredient_of?"
  },
  {
    "query": "*Is it true that* the source of location_of can be idea_or_concept?",
    "predicate": "location_of",
    "subject": "idea_or_concept",
    "llm": "entries connected by location_of?"
  },
  {
    "query": "*Is it true that* the source of location_of can be therapeutic_or_preventive_procedure?",
    "predicate": "location_of",
    "subject": "therapeutic_or_preventive_procedure",
    "llm": "entries connected by location_of?"
  },
  {
    "query": "*Is it true that* the source of location_of can be eukaryote?",
    "predicate": "location_of",
    "subject": "eukaryote",
    "llm": "entries connected by location_of?"
  }
]


In [ ]:

async def query_sparql_type30(subject, predicate):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>

      ASK {{
        umls:{predicate} rdfs:domain ?r .
        umls:{subject} rdfs:subClassOf ?r .
      }}

    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type30(queries, prompt_level,  local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        queryl = query_data['llm']



        subject = query_data['subject']
        predicate = query_data['predicate']
        ground_truth_triples = await query_sparql_type30(subject, predicate)

        response = await local_search.asearch(query=queryl, search_prompt=prompt_level)
        response_text = response.response
        print("resp etxt")
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query, response_text=response_text)
        print(response_text2)
        clean_text = strip_prefixes(response_text2)
        print(clean_text)
        print("ground_truth_triples")
        print(ground_truth_triples)
        print("model_response_triples")
        print(clean_text)

        gt_bool = normalize_to_bool(ground_truth_triples)

        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")




    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)

    return x


In [ ]:
queries_type_A5 = [
    {"query": "*Is it true that* abnormal_toe_morphology is a anatomical_structure?", "subject": "abnormal_toe_morphology", "object": "anatomical_structure"},
    {"query": "*Is it true that* exudate_in_the_colonic_lumen is a substance?", "subject": "exudate_in_the_colonic_lumen", "object": "substance"},
    {"query": "*Is it true that* cerebellar_cysts is a anatomical_abnormality?", "subject": "cerebellar_cysts", "object": "anatomical_abnormality"},
    {"query": "*Is it true that* sulphadiazine is a antibiotic?", "subject": "sulphadiazine", "object": "antibiotic"},
    {"query": "*Is it true that* colonic_stents is a physical_object?", "subject": "colonic_stents", "object": "physical_object"},
    {"query": "*Is it true that* epidermolysis_bullosa_pretibial is a anatomical_structure?", "subject": "epidermolysis_bullosa_pretibial", "object": "anatomical_structure"},
    {"query": "*Is it true that* muscular_dystrophy_congenital_1c is a disease_or_syndrome?", "subject": "muscular_dystrophy_congenital_1c", "object": "disease_or_syndrome"},
    {"query": "*Is it true that* nilotinib is a anatomical_structure?", "subject": "nilotinib", "object": "anatomical_structure"},
    {"query": "*Is it true that* ascorbic_acid_collagen_hydrolyzed is a chemical_viewed_functionally?", "subject": "ascorbic_acid_collagen_hydrolyzed", "object": "chemical_viewed_functionally"},
    {"query": "*Is it true that* locomotion_involved_in_locomotory_behavior is a event?", "subject": "locomotion_involved_in_locomotory_behavior", "object": "event"},
    {"query": "*Is it true that* urogenital_system_development is a biologic_function?", "subject": "urogenital_system_development", "object": "biologic_function"},
    {"query": "*Is it true that* hydrocerin_topical_cream is a physical_object?", "subject": "hydrocerin_topical_cream", "object": "physical_object"},
    {"query": "*Is it true that* superior_frontal_sulcus_human_only is a anatomical_structure?", "subject": "superior_frontal_sulcus_human_only", "object": "anatomical_structure"},
    {"query": "*Is it true that* polyvinyl_alcohol is a substance?", "subject": "polyvinyl_alcohol", "object": "substance"},
    {"query": "*Is it true that* sulcal_segment_of_inferior_frontal_gyrus is a entity?", "subject": "sulcal_segment_of_inferior_frontal_gyrus", "object": "entity"},
    {"query": "*Is it true that* carrington_incontinence_skin_kit is a manufactured_object?", "subject": "carrington_incontinence_skin_kit", "object": "manufactured_object"},
    {"query": "*Is it true that* cleansers_skin is a physical_object?", "subject": "cleansers_skin", "object": "physical_object"},
    {"query": "*Is it true that* durapatite is a chemical?", "subject": "durapatite", "object": "chemical"},
    {"query": "*Is it true that* science_of_anatomy is a occupation_or_discipline?", "subject": "science_of_anatomy", "object": "occupation_or_discipline"},
    {"query": "*Is it true that* escherichia_coli is a bacterium?", "subject": "escherichia_coli", "object": "bacterium"},
    {"query": "*Is it true that* lactobacillus_cap_oral_14_lactobacillus_cap_vag_14 is a drug_delivery_device?", "subject": "lactobacillus_cap_oral_14_lactobacillus_cap_vag_14", "object": "drug_delivery_device"},
    {"query": "*Is it true that* quality_of_preparation is a conceptual_entity?", "subject": "quality_of_preparation", "object": "conceptual_entity"},
    {"query": "*Is it true that* polypectomy is a health_care_activity?", "subject": "polypectomy", "object": "health_care_activity"},
    {"query": "*Is it true that* colonic_parasites_diagnosis is a organism?", "subject": "colonic_parasites_diagnosis", "object": "organism"}
]


In [ ]:

async def query_sparql_type31(subject, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>


      ASK {{
      umls:{subject} rdf:type  umls:{objectf}.
      }}

    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type31(queries, prompt_level,  local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0
    for query_data in queries:
        query = query_data['query']

        subject = query_data['subject']
        objectf = query_data['object']
        ground_truth_triples = await query_sparql_type31(subject, objectf)

        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query, response_text=response_text)
        print(response_text2)

        clean_text = strip_prefixes(response_text2)
        print(clean_text)

        print(ground_truth_triples)
        print(clean_text)

        gt_bool = normalize_to_bool(ground_truth_triples)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")




    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)



    return x


In [ ]:
queries_type_A6 = [
    {"query": "*Is it true that* anatomical_abnormality is a subclass of anatomical_structure?", "subject": "anatomical_abnormality", "object": "anatomical_structure", "llm": "anatomical_abnormality subClassOf ?"},
    {"query": "*Is it true that* body_substance is a subclass of substance?", "subject": "body_substance", "object": "substance", "llm": "body_substance subClassOf ?"},
    {"query": "*Is it true that* acquired_abnormality is a subclass of anatomical_abnormality?", "subject": "acquired_abnormality", "object": "anatomical_abnormality", "llm": "acquired_abnormality subClassOf ?"},
    {"query": "*Is it true that* pharmacologic_substance is a subclass of antibiotic?", "subject": "pharmacologic_substance", "object": "antibiotic", "llm": "pharmacologic_substance subClassOf ?"},
    {"query": "*Is it true that* medical_device is a subclass of physical_object?", "subject": "medical_device", "object": "physical_object", "llm": "medical_device subClassOf ?"},
    {"query": "*Is it true that* congenital_abnormality is a subclass of anatomical_structure?", "subject": "congenital_abnormality", "object": "anatomical_structure", "llm": "congenital_abnormality subClassOf ?"},
    {"query": "*Is it true that* disease_or_syndrome is a subclass of disease_or_syndrome?", "subject": "disease_or_syndrome", "object": "disease_or_syndrome", "llm": "disease_or_syndrome subClassOf ?"},
    {"query": "*Is it true that* amino_acid_peptide_or_protein is a subclass of anatomical_structure?", "subject": "amino_acid_peptide_or_protein", "object": "anatomical_structure", "llm": "amino_acid_peptide_or_protein subClassOf ?"},
    {"query": "*Is it true that* pharmacologic_substance is a subclass of chemical_viewed_functionally?", "subject": "pharmacologic_substance", "object": "chemical_viewed_functionally", "llm": "pharmacologic_substance subClassOf ?"},
    {"query": "*Is it true that* biologic_function is a subclass of event?", "subject": "biologic_function", "object": "event", "llm": "biologic_function subClassOf ?"},
    {"query": "*Is it true that* organ_or_tissue_function is a subclass of biologic_function?", "subject": "organ_or_tissue_function", "object": "biologic_function", "llm": "organ_or_tissue_function subClassOf ?"},
    {"query": "*Is it true that* clinical_drug is a subclass of physical_object?", "subject": "clinical_drug", "object": "physical_object", "llm": "clinical_drug subClassOf ?"},
    {"query": "*Is it true that* body_space_or_junction is a subclass of anatomical_structure?", "subject": "body_space_or_junction", "object": "anatomical_structure", "llm": "body_space_or_junction subClassOf ?"},
    {"query": "*Is it true that* biomedical_or_dental_material is a subclass of substance?", "subject": "biomedical_or_dental_material", "object": "substance", "llm": "biomedical_or_dental_material subClassOf ?"},
    {"query": "*Is it true that* mls:body_location_or_region is a subclass of Enitity?", "subject": "mls:body_location_or_region", "object": "Enitity", "llm": "mls:body_location_or_region subClassOf ?"},
    {"query": "*Is it true that* drug_delivery_device is a subclass of manufactured_object?", "subject": "drug_delivery_device", "object": "manufactured_object", "llm": "drug_delivery_device subClassOf ?"},
    {"query": "*Is it true that* biomedical_or_dental_material is a subclass of physical_object?", "subject": "biomedical_or_dental_material", "object": "physical_object", "llm": "biomedical_or_dental_material subClassOf ?"},
    {"query": "*Is it true that* biomedical_occupation_or_discipline is a subclass of occupation_or_discipline?", "subject": "biomedical_occupation_or_discipline", "object": "occupation_or_discipline", "llm": "biomedical_occupation_or_discipline subClassOf ?"},
    {"query": "*Is it true that* neoplastic_process is a subclass of pathologic_function?", "subject": "neoplastic_process", "object": "Umls:pathologic_function", "llm": "neoplastic_process subClassOf ?"},
    {"query": "*Is it true that* organism is a subclass of bacterium?", "subject": "organism", "object": "Umls:bacterium", "llm": "organism subClassOf ?"},
    {"query": "*Is it true that* manufactured_object is a subclass of drug_delivery_device?", "subject": "manufactured_object", "object": "drug_delivery_device", "llm": "manufactured_object subClassOf ?"},
    {"query": "*Is it true that* idea_or_concept is a subclass of conceptual_entity?", "subject": "idea_or_concept", "object": "conceptual_entity", "llm": "idea_or_concept subClassOf ?"},
    {"query": "*Is it true that* therapeutic_or_preventive_procedure is a subclass of health_care_activity?", "subject": "therapeutic_or_preventive_procedure", "object": "health_care_activity", "llm": "therapeutic_or_preventive_procedure subClassOf ?"},
    {"query": "*Is it true that* eukaryote is a subclass of organism?", "subject": "eukaryote", "object": "organism", "llm": "eukaryote subClassOf ?"}
]


In [ ]:

async def query_sparql_type32(subject, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>


      ASK {{
      umls:{subject} rdfs:subClassOf  umls:{objectf}.
      }}

    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type32(queries, prompt_level,  local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        llm = query_data['llm']



        subject = query_data['subject']
        objectf = query_data['object']
        ground_truth_triples = await query_sparql_type32(subject, objectf)

        response = await local_search.asearch(query=llm, search_prompt=prompt_level)
        response_text = response.response
        print("resp etxt")
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query, response_text=response_text)
        print(response_text2)
        clean_text = strip_prefixes(response_text2)
        print(clean_text)
        print(ground_truth_triples)
        print(clean_text)

        gt_bool = normalize_to_bool(ground_truth_triples)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")



    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)



    return x


In [ ]:
queries_type_A7 = [
    {
        "query": "*Is it true that* cytology diagnoses something that is a anatomical_structure?",
        "subject": "cytology",
        "object": "anatomical_structure",
        "predicate": "diagnoses",
        "llm": "cytology diagnoses ?"
    },
    {
        "query": "*Is it true that* sigmoid_colon location_of something that is a substance?",
        "subject": "sigmoid_colon",
        "object": "substance",
        "predicate": "location_of",
        "llm": "sigmoid_colon location_of ?"
    },
    {
        "query": "*Is it true that* deprecated_us_guidance_for_repair_of_pseudoaneurysm_av_fistula Analyses something that is a anatomical_abnormality?",
        "subject": "deprecated_us_guidance_for_repair_of_pseudoaneurysm_av_fistula",
        "object": "anatomical_abnormality",
        "predicate": "analyses",
        "llm": "deprecated_us_guidance_for_repair_of_pseudoaneurysm_av_fistula analyses ?"
    },
    {
        "query": "*Is it true that* medications_pt_patient evaluation_of something that is a antibiotic?",
        "subject": "medications_pt_patient",
        "object": "antibiotic",
        "predicate": "evaluation_of",
        "llm": "medications_pt_patient evaluation_of ?"
    },
    {
        "query": "*Is it true that* whole_examined_colon location_of something that is a physical_object?",
        "subject": "whole_examined_colon",
        "object": "physical_object",
        "predicate": "location_of",
        "llm": "whole_examined_colon location_of ?"
    },
    {
        "query": "*Is it true that* hypertrophic_cicatrix manifestation_of something that is a anatomical_structure?",
        "subject": "hypertrophic_cicatrix",
        "object": "anatomical_structure",
        "predicate": "manifestation_of",
        "llm": "hypertrophic_cicatrix manifestation_of ?"
    },
    {
        "query": "*Is it true that* hypotonia_neonatal manifestation_of something that is a disease_or_syndrome?",
        "subject": "hypotonia_neonatal",
        "object": "disease_or_syndrome",
        "predicate": "manifestation_of",
        "llm": "hypotonia_neonatal manifestation_of ?"
    },
    {
        "query": "*Is it true that* transferrin_carbohydrate_deficient_asialo_prthr_pt_csf_ord measures something that is a anatomical_structure?",
        "subject": "transferrin_carbohydrate_deficient_asialo_prthr_pt_csf_ord",
        "object": "anatomical_structure",
        "predicate": "measures",
        "llm": "transferrin_carbohydrate_deficient_asialo_prthr_pt_csf_ord measures ?"
    },
    {
        "query": "*Is it true that* collagen_hydrolyzed part_of something that is a chemical_viewed_functionally?",
        "subject": "collagen_hydrolyzed",
        "object": "chemical_viewed_functionally",
        "predicate": "part_of",
        "llm": "collagen_hydrolyzed part_of ?"
    },
    {
        "query": "*Is it true that* chemotactic_factor_inactivator exhibits something that is a event?",
        "subject": "chemotactic_factor_inactivator",
        "object": "event",
        "predicate": "exhibits",
        "llm": "chemotactic_factor_inactivator exhibits ?"
    },
    {
        "query": "*Is it true that* renal_system_development part_of something that is a biologic_function?",
        "subject": "renal_system_development",
        "object": "biologic_function",
        "predicate": "part_of",
        "llm": "renal_system_development part_of ?"
    },
    {
        "query": "*Is it true that* quaternium_15 ingredient_of something that is a physical_object?",
        "subject": "quaternium_15",
        "object": "physical_object",
        "predicate": "ingredient_of",
        "llm": "quaternium_15 ingredient_of ?"
    },
    {
        "query": "*Is it true that* sulcal_segment_of_superior_frontal_gyrus adjacent_to something that is a anatomical_structure?",
        "subject": "sulcal_segment_of_superior_frontal_gyrus",
        "object": "anatomical_structure",
        "predicate": "adjacent_to",
        "llm": "sulcal_segment_of_superior_frontal_gyrus adjacent_to ?"
    },
    {
        "query": "*Is it true that* di_2_ethylhexyl_phthalate_mcnc_pt_water_qn measures something that is a substance?",
        "subject": "di_2_ethylhexyl_phthalate_mcnc_pt_water_qn",
        "object": "substance",
        "predicate": "measures",
        "llm": "di_2_ethylhexyl_phthalate_mcnc_pt_water_qn measures ?"
    },
    {
        "query": "*Is it true that* oesophageal_nerve_plexus surrounds something that is a body_location_or_region?",
        "subject": "oesophageal_nerve_plexus",
        "object": "body_location_or_region",
        "predicate": "surrounds",
        "llm": "oesophageal_nerve_plexus surrounds ?"
    },
    {
        "query": "*Is it true that* imidazolidinyl_urea ingredient_of something that is a manufactured_object?",
        "subject": "imidazolidinyl_urea",
        "object": "manufactured_object",
        "predicate": "ingredient_of",
        "llm": "imidazolidinyl_urea ingredient_of ?"
    },
    {
        "query": "*Is it true that* anise_oil ingredient_of something that is a physical_object?",
        "subject": "anise_oil",
        "object": "physical_object",
        "predicate": "ingredient_of",
        "llm": "anise_oil ingredient_of ?"
    },
    {
        "query": "*Is it true that* cause_of_graft_failure_find_pt_graft_nom Analyses something that is a biomedical_or_dental_material?",
        "subject": "cause_of_graft_failure_find_pt_graft_nom",
        "object": "biomedical_or_dental_material",
        "predicate": "analyses",
        "llm": "cause_of_graft_failure_find_pt_graft_nom analyses ?"
    },
    {
        "query": "*Is it true that* deprecated_clinical_discipline_speech_therapy_treatment_plan associated_with something that is a biomedical_occupation_or_discipline?",
        "subject": "deprecated_clinical_discipline_speech_therapy_treatment_plan",
        "object": "biomedical_occupation_or_discipline",
        "predicate": "associated_with",
        "llm": "deprecated_clinical_discipline_speech_therapy_treatment_plan associated_with ?"
    },
    {
        "query": "*Is it true that* type_of_colonic_tumor is a manifestation_of something that is a pathologic_function?",
        "subject": "type_of_colonic_tumor",
        "object": "pathologic_function",
        "predicate": "manifestation_of",
        "llm": "type_of_colonic_tumor manifestation_of ?"
    },
    {
        "query": "*Is it true that* multiple_drug_resistant_organism_prid_pt_sputum_nom measures something that is a organism?",
        "subject": "multiple_drug_resistant_organism_prid_pt_sputum_nom",
        "object": "organism",
        "predicate": "measures",
        "llm": "multiple_drug_resistant_organism_prid_pt_sputum_nom measures ?"
    },
    {
        "query": "*Is it true that* lactobacillus ingredient_of something that is a manufactured_object?",
        "subject": "lactobacillus",
        "object": "manufactured_object",
        "predicate": "ingredient_of",
        "llm": "lactobacillus ingredient_of ?"
    },
    {
        "query": "*Is it true that* left_colic_flexure location_of something that is a conceptual_entity?",
        "subject": "left_colic_flexure",
        "object": "conceptual_entity",
        "predicate": "location_of",
        "llm": "left_colic_flexure location_of ?"
    },
    {
        "query": "*Is it true that* area_of_papilla location_of something that is a therapeutic_or_preventive_procedure?",
        "subject": "area_of_papilla",
        "object": "therapeutic_or_preventive_procedure",
        "predicate": "location_of",
        "llm": "area_of_papilla location_of ?"
    },
    {
        "query": "*Is it true that* caecum location_of something that is a organism?",
        "subject": "caecum",
        "object": "organism",
        "predicate": "location_of",
        "llm": "caecum location_of ?"
    }
]


In [ ]:

async def query_sparql_type33(subject, predicate, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>


      ASK {{
      umls:{subject} umls:{predicate}  ?x .
      ?x rdf:type umls:{objectf}.}}

    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type33(queries, prompt_level,  local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        llm = query_data['llm']

        subject = query_data['subject']
        objectf = query_data['object']
        predicate = query_data['predicate']
        ground_truth_triples = await query_sparql_type33(subject, predicate, objectf)
        response = await local_search.asearch(query=llm, search_prompt=prompt_level)
        response_text = response.response
        print("resp etxt")
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query, response_text=response_text)
        print(response_text2)
        clean_text = strip_prefixes(response_text2)
        print(clean_text)
        print(ground_truth_triples)
        print(clean_text)

        gt_bool = normalize_to_bool(ground_truth_triples)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")



    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)



    return x


In [ ]:
queries_type_A8 = [
    {"query": "*Is it true that* adjacent_to a kind of spatially_related_to?", "subject": "adjacent_to", "object": "spatially_related_to", "llm": "adjacent_to subpropertyof ?"},
    {"query": "*Is it true that* adjacent_to a kind of spatially_related_to?", "subject": "adjacent_to", "object": "spatially_related_to", "llm": "adjacent_to subpropertyof ?"},

    {"query": "*Is it true that* affects a kind of functionally_related_to?", "subject": "affects", "object": "functionally_related_to", "llm": "affects subpropertyof ?"},
    {"query": "*Is it true that* affects a kind of functionally_related_to?", "subject": "affects", "object": "functionally_related_to", "llm": "affects subpropertyof ?"},

    {"query": "*Is it true that* branch_of a kind of physically_related_to?", "subject": "branch_of", "object": "physically_related_to", "llm": "branch_of subpropertyof ?"},
    {"query": "*Is it true that* branch_of a kind of physically_related_to?", "subject": "branch_of", "object": "physically_related_to", "llm": "branch_of subpropertyof ?"},

    {"query": "*Is it true that* conceptual_part_of a kind of conceptually_related_to?", "subject": "conceptual_part_of", "object": "conceptually_related_to", "llm": "conceptual_part_of subpropertyof ?"},
    {"query": "*Is it true that* conceptual_part_of a kind of conceptually_related_to?", "subject": "conceptual_part_of", "object": "conceptually_related_to", "llm": "conceptual_part_of subpropertyof ?"},

    {"query": "*Is it true that* conceptually_related_to a kind of associated_with?", "subject": "conceptually_related_to", "object": "associated_with", "llm": "conceptually_related_to subpropertyof ?"},

    {"query": "*Is it true that* connected_to a kind of physically_related_to?", "subject": "connected_to", "object": "physically_related_to", "llm": "connected_to subpropertyof ?"},

    {"query": "*Is it true that* consists_of a kind of physically_related_to?", "subject": "consists_of", "object": "physically_related_to", "llm": "consists_of subpropertyof ?"},

    {"query": "*Is it true that* contains a kind of physically_related_to?", "subject": "contains", "object": "physically_related_to", "llm": "contains subpropertyof ?"},

    {"query": "*Is it true that* is part_of a kind of physically_related_to?", "subject": "part_of", "object": "physically_related_to", "llm": "part_of subpropertyof ?"},
    {"query": "*Is it true that* part_of a kind of physically_related_to?", "subject": "part_of", "object": "physically_related_to", "llm": "part_of subpropertyof ?"},

    {"query": "*Is it true that* physically_related_to a kind of physically_related_to?", "subject": "physically_related_to", "object": "physically_related_to", "llm": "physically_related_to subpropertyof ?"},
    {"query": "*Is it true that* physically_related_to a kind of associated_with?", "subject": "physically_related_to", "object": "associated_with", "llm": "physically_related_to subpropertyof ?"},

    {"query": "*Is it true that* spatially_related_to a kind of associated_with?", "subject": "spatially_related_to", "object": "associated_with", "llm": "spatially_related_to subpropertyof ?"},

    {"query": "*Is it true that* surrounds a kind of spatially_related_to?", "subject": "surrounds", "object": "spatially_related_to", "llm": "surrounds subpropertyof ?"},

    {"query": "*Is it true that* treats a kind of affects?", "subject": "treats", "object": "affects", "llm": "treats subpropertyof ?"},
    {"query": "*Is it true that* treats a kind of affects?", "subject": "treats", "object": "affects", "llm": "treats subpropertyof ?"},

    {"query": "*Is it true that* tributary_of a kind of physically_related_to?", "subject": "tributary_of", "object": "physically_related_to", "llm": "tributary_of subpropertyof ?"},

    {"query": "*Is it true that* exhibits a kind of functionally_related_to?", "subject": "exhibits", "object": "functionally_related_to", "llm": "exhibits subpropertyof ?"},

    {"query": "*Is it true that* functionally_related_to a kind of associated_with?", "subject": "functionally_related_to", "object": "associated_with", "llm": "functionally_related_to subpropertyof ?"},

    {"query": "*Is it true that* ingredient_of a kind of physically_related_to?", "subject": "ingredient_of", "object": "physically_related_to", "llm": "ingredient_of subpropertyof ?"},

    {"query": "*Is it true that* location_of a kind of spatially_related_to?", "subject": "location_of", "object": "spatially_related_to", "llm": "location_of subpropertyof ?"},

    {"query": "*Is it true that* occurs_in a kind of spatially_related_to?", "subject": "occurs_in", "object": "spatially_related_to", "llm": "occurs_in subpropertyof ?"}
]


In [ ]:

async def query_sparql_type34(subject, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
      ASK {{
      umls:{subject} rdfs:subPropertyOf umls:{objectf}  .}}

    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type34(queries, prompt_level,  local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        llm = query_data['llm']



        subject = query_data['subject']
        objectf = query_data['object']
        ground_truth_triples = await query_sparql_type34(subject, objectf)
        response = await local_search.asearch(query=llm, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        response_text2 = await infer_rdfs_triples_simple_modifiedr(query, response_text=response_text)

        print(response_text2)

        clean_text = strip_prefixes(response_text2)
        print(clean_text)

        print(ground_truth_triples)
        print(clean_text)

        gt_bool = normalize_to_bool(ground_truth_triples)

        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)



    return x


In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type1(subject, predicate):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT ?subject ?predicate ?object
    WHERE {{
      umls:{subject} umls:{predicate} ?object.
      BIND(umls:{predicate} AS ?predicate)
      BIND(umls:{subject} AS ?subject)
    }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print("ret is")
        print(ret)
        print()
        extr = extract_triples(ret)
        print("extr")
        print(extr)
        print("final sp")
        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type1(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']
        predicate = query_data['predicate']

        ground_truth_triples = await query_sparql_type1(subject, predicate)
        print("ground")
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)

        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(ground_truth_triples, model_response_triples)

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }
    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }


    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results








In [ ]:
async def compute_accuracy_for_query_type1(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):

    accuracy_results = {'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    all_query_results = []

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']
        predicate = query_data['predicate']

        ground_truth_triples = await query_sparql_type1(subject, predicate)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)

        common, extra, precision, recall, f1_score = \
            find_common_and_extra_elements2(
                ground_truth_triples,
                model_response_triples
            )


        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)

        response_results['results'].append(response_text)


        all_query_results.append({
            "query": query,
            "benchmark_triples": list(ground_truth_triples),
            "llm_triples": list(model_response_triples),
            "common_triples": common,
            "extra_triples": extra,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score
        })


    mean_metrics = {
        'precision': float(np.mean(accuracy_results['precision'])) if accuracy_results['precision'] else 0,
        'recall': float(np.mean(accuracy_results['recall'])) if accuracy_results['recall'] else 0,
        'f1_score': float(np.mean(accuracy_results['f1_score'])) if accuracy_results['f1_score'] else 0,
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")

    return accuracy_results, mean_metrics, response_results


In [ ]:
queries_type_1 = [
    {"query": "colostomy_stoma location_of?", "subject": "colostomy_stoma", "predicate": "location_of"},
    {"query": "rectal_pouch location_of?", "subject": "rectal_pouch", "predicate": "location_of"},
    {"query": "sulcal_segment_of_middle_frontal_gyrus adjacent_to?", "subject": "sulcal_segment_of_middle_frontal_gyrus", "predicate": "adjacent_to"},
    {"query": "wall_of_biventricular_part_of_heart surrounds?", "subject": "wall_of_biventricular_part_of_heart", "predicate": "surrounds"},
    {"query": "duct_cannulation uses?", "subject": "duct_cannulation", "predicate": "uses"},
    {"query": "oesophagogastroduodenoscopy diagnoses?", "subject": "oesophagogastroduodenoscopy", "predicate": "diagnoses"},
    {"query": "cytochrome_b5 exhibits?", "subject": "cytochrome_b5", "predicate": "exhibits"},
    {"query": "ascorbic_acid ingredient_of?", "subject": "ascorbic_acid", "predicate": "ingredient_of"},
    {"query": "retrograde_cholangiopancreatography_endoscopic diagnoses?", "subject": "retrograde_cholangiopancreatography_endoscopic", "predicate": "diagnoses"},
    {"query": "perfumes ingredient_of?", "subject": "perfumes", "predicate": "ingredient_of"},
    {"query": "aggressive_behaviors manifestation_of?", "subject": "aggressive_behaviors", "predicate": "manifestation_of"},
    {"query": "minimum_data_set_mds_version_3 method_of?", "subject": "minimum_data_set_mds_version_3", "predicate": "method_of"},
    {"query": "structure_of_carotid_sheath surrounds?", "subject": "structure_of_carotid_sheath", "predicate": "surrounds"},
    {"query": "polypectomy uses?", "subject": "polypectomy", "predicate": "uses"},
    {"query": "thermal_therapy uses?", "subject": "thermal_therapy", "predicate": "uses"},
    {"query": "cavity_abdominal contains?", "subject": "cavity_abdominal", "predicate": "contains"},
    {"query": "left_nasal_bone connected_to?", "subject": "left_nasal_bone", "predicate": "connected_to"},
    {"query": "structure_of_umbilical_artery branch_of?", "subject": "structure_of_umbilical_artery", "predicate": "branch_of"},
    {"query": "routes_drug_administration associated_with?", "subject": "routes_drug_administration", "predicate": "associated_with"},
    {"query": "aloe_vera_gel ingredient_of?", "subject": "aloe_vera_gel", "predicate": "ingredient_of"},
    {"query": "posterior_mediastinum contains?", "subject": "posterior_mediastinum", "predicate": "contains"},
    {"query": "descending_part_of_duodenum location_of?", "subject": "descending_part_of_duodenum", "predicate": "location_of"},
    {"query": "caecum location_of?", "subject": "caecum", "predicate": "location_of"},
    {"query": "ataxia manifestation_of?", "subject": "ataxia", "predicate": "manifestation_of"},
    {"query": "assays_immunofluorescence method_of?", "subject": "assays_immunofluorescence", "predicate": "method_of"}
]


In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np

from SPARQLWrapper import SPARQLWrapper, JSON

async def query_sparql_type2(object2, predicate):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT ?subject ?predicate ?object
    WHERE {{
        {{ umls:{object2} umls:{predicate} ?object .
           BIND(umls:{object2} AS ?subject)
           BIND(umls:{predicate} AS ?predicate)
        }}
        UNION
        {{ ?subject umls:{predicate} umls:{object2} .
           BIND(umls:{object2} AS ?object)
           BIND(umls:{predicate} AS ?predicate)
        }}
    }}
    """

    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        extr = extract_triples(ret)
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []


async def compute_accuracy_for_query2(queries, prompt_level, local_search, query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']
        object2 = query_data['object']
        predicate = query_data['predicate']

        ground_truth_triples = await query_sparql_type2(object2, predicate)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)

        print(model_response_triples)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(ground_truth_triples, model_response_triples)

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }


    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results





In [ ]:
queries_type_2 = [
      {
        "query": "Retrieve all triples where the predicate is 'adjacent_to', and the subject OR the object is 'oesophagus'.",
        "predicate": "adjacent_to",
        "object": "oesophagus"
    },
    {"query": "Retrieve all triples where the predicate is 'tributary_of', and the subject or object is 'structure_of_inferior_petrosal_sinus'.",
     "predicate": "tributary_of",
     "object": "structure_of_inferior_petrosal_sinus"},

    {"query": "Retrieve all triples where the predicate is 'associated_with', and the subject or object is 'subsequent_visit_evaluation_note'.",
     "predicate": "associated_with",
     "object": "subsequent_visit_evaluation_note"},

    {"query": "Retrieve all triples where the predicate is 'associated_with', and the subject or object is 'principal_diagnosis'.",
     "predicate": "associated_with",
     "object": "principal_diagnosis"},

    {"query": "Retrieve all triples where the predicate is 'branch_of', and the subject or object is 'nerves_sacral'.",
     "predicate": "branch_of",
     "object": "nerves_sacral"},

    {"query": "Retrieve all triples where the predicate is 'branch_of', and the subject or object is 'structure_of_umbilical_artery'.",
     "predicate": "branch_of",
     "object": "structure_of_umbilical_artery"},

    {"query": "Retrieve all triples where the predicate is 'exhibits', and the subject or object is 'dihydrolipoyl_dehydrogenase_complex_location'.",
     "predicate": "exhibits",
     "object": "dihydrolipoyl_dehydrogenase_complex_location"},

    {"query": "Retrieve all triples where the predicate is 'part_of', and the subject or object is 'skins'.",
     "predicate": "part_of",
     "object": "skins"},

    {"query": "Retrieve all triples where the predicate is 'manifestation_of', and the subject or object is 'biliary_calculus'.",
     "predicate": "manifestation_of",
     "object": "biliary_calculus"},

    {"query": "Retrieve all triples where the predicate is 'part_of', and the subject or object is 'fibrillar_component_of_intercellular_matrix'.",
     "predicate": "part_of",
     "object": "fibrillar_component_of_intercellular_matrix"},

    {"query": "Retrieve all triples where the predicate is 'part_of', and the subject or object is 'meiotic_cell_cycle'.",
     "predicate": "part_of",
     "object": "meiotic_cell_cycle"},

    {"query": "Retrieve all triples where the predicate is 'part_of', and the subject or object is 'hepatic_lamina'.",
     "predicate": "part_of",
     "object": "hepatic_lamina"},

    {"query": "Retrieve all triples where the predicate is 'surrounds', and the subject or object is 'nucleoplasm'.",
     "predicate": "surrounds",
     "object": "nucleoplasm"},

    {"query": "Retrieve all triples where the predicate is 'surrounds', and the subject or object is 'cytoplasms'.",
     "predicate": "surrounds",
     "object": "cytoplasms"},

    {"query": "Retrieve all triples where the predicate is 'part_of', and the subject or object is 'midgut'.",
     "predicate": "part_of",
     "object": "midgut"},

    {"query": "Retrieve all triples where the predicate is 'associated_with', and the subject or object is 'umls:medication_administered'.",
     "predicate": "associated_with",
     "object": "umls:medication_administered"},

    {"query": "Retrieve all triples where the predicate is 'associated_with', and the subject or object is 'date_onset_or_exacerbation'.",
     "predicate": "associated_with",
     "object": "date_onset_or_exacerbation"},

    {"query": "Retrieve all triples where the predicate is 'branch_of', and the subject or object is 'thoracic_aorta'.",
     "predicate": "branch_of",
     "object": "thoracic_aorta"},

    {"query": "Retrieve all triples where the predicate is 'associated_with', and the subject or object is 'routes'.",
     "predicate": "associated_with",
     "object": "routes"},

    {"query": "Retrieve all triples where the predicate is 'connected_to', and the subject or object is 'left_sphenoparietal_suture'.",
     "predicate": "connected_to",
     "object": "left_sphenoparietal_suture"},

    {"query": "Retrieve all triples where the predicate is 'connected_to', and the subject or object is 'left_nasal_bone'.",
     "predicate": "connected_to",
     "object": "left_nasal_bone"},

    {"query": "Retrieve all triples where the predicate is 'connected_to', and the subject or object is 'temporal_part_of_left_greater_wing_of_sphenoid'.",
     "predicate": "connected_to",
     "object": "temporal_part_of_left_greater_wing_of_sphenoid"},

    {"query": "Retrieve all triples where the predicate is 'part_of', and the subject or object is 'blood'.",
     "predicate": "part_of",
     "object": "blood"},

    {"query": "Retrieve all triples where the predicate is 'ingredient_of', and the subject or object is 'aloe_vera_gel'.",
     "predicate": "ingredient_of",
     "object": "aloe_vera_gel"},

    {"query": "Retrieve all triples where the predicate is 'connected_to', and the subject or object is 'left_zygomaticomaxillary_suture'.",
     "predicate": "connected_to",
     "object": "left_zygomaticomaxillary_suture"}

]


In [ ]:
async def query_sparql_type3(subject, obj):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>

    SELECT ?subject ?predicate ?object
    WHERE {{ umls:{subject} ?predicate umls:{obj}.
        BIND(umls:{subject} AS ?subject)
        BIND(umls:{obj} AS ?object)
      }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        extr = extract_triples(ret)
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query3(queries, prompt_level, local_search, query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']
        object2 = query_data['subject']
        object1 = query_data['object']

        ground_truth_triples = await query_sparql_type3(object2, object1)
        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")

        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(ground_truth_triples, model_response_triples)

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")

    mean_metrics = {
          'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
          'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
          'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
      }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }


    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results






In [ ]:
queries_type3 = [
    {
        "query": "Retrieve the relationships that connect posterior_mediastinum and t8_segment_of_esophagus",
        "subject": "posterior_mediastinum",
        "object": "t8_segment_of_esophagus"
    },
    {
        "query": "Retrieve the relationships that connect right_submandibular_triangle and right_hyoglossus",
        "subject": "right_submandibular_triangle",
        "object": "right_hyoglossus"
    },
    {
        "query": "Retrieve the relationships that connect plasma_membranes and cytoplasms",
        "subject": "plasma_membranes",
        "object": "cytoplasms"
    },
    {
        "query": "Retrieve the relationships that connect nucleoplasm and nucleolus_cell",
        "subject": "nucleoplasm",
        "object": "nucleolus_cell"
    },
    {
        "query": "Retrieve the relationships that connect sioux_county_ne and nebraska_geographic_location",
        "subject": "sioux_county_ne",
        "object": "nebraska_geographic_location"
    },
    {
        "query": "Retrieve the relationships that connect dilation_size and dilate",
        "subject": "dilation_size",
        "object": "dilate"
    },
    {
        "query": "Retrieve the relationships that connect complications and gastrointestinal_endoscopy",
        "subject": "complications",
        "object": "gastrointestinal_endoscopy"
    },
    {
        "query": "Retrieve the relationships that connect absent_left_common_carotid_artery and aortic_arch_structure",
        "subject": "absent_left_common_carotid_artery",
        "object": "aortic_arch_structure"
    },
    {
        "query": "Retrieve the relationships that connect nerves_sacral and root_of_uwda_hierarchy",
        "subject": "nerves_sacral",
        "object": "root_of_uwda_hierarchy"
    },
    {
        "query": "Retrieve the relationships that connect synaptic_signaling and synapsis",
        "subject": "synaptic_signaling",
        "object": "synapsis"
    },
    {
        "query": "Retrieve the relationships that connect maintenance_of_protein_localisation_to_organelle and organelle",
        "subject": "maintenance_of_protein_localisation_to_organelle",
        "object": "organelle"
    },
    {
        "query": "Retrieve the relationships that connect type_iii_no_bleeding_of_gastric_ulcer and type_of_bleeding_of_gastric_ulcer",
        "subject": "type_iii_no_bleeding_of_gastric_ulcer",
        "object": "type_of_bleeding_of_gastric_ulcer"
    },
    {
        "query": "Retrieve the relationships that connect type_iii_no_bleeding_of_duodenal_ulcer and type_of_bleeding_of_duodenal_ulcer",
        "subject": "type_iii_no_bleeding_of_duodenal_ulcer",
        "object": "type_of_bleeding_of_duodenal_ulcer"
    },
    {
        "query": "Retrieve the relationships that connect polypectomy and polypectomy_forceps",
        "subject": "polypectomy",
        "object": "polypectomy_forceps"
    },
    {
        "query": "Retrieve the relationships that connect biopsy_nos and biopsy_instruments",
        "subject": "biopsy_nos",
        "object": "biopsy_instruments"
    },
    {
        "query": "Retrieve the relationships that connect hypertrophic_cicatrix and epidermolysis_bullosa_pretibial",
        "subject": "hypertrophic_cicatrix",
        "object": "epidermolysis_bullosa_pretibial"
    },
    {
        "query": "Retrieve the relationships that connect cheloid and coli_adenomatous_polyposis",
        "subject": "cheloid",
        "object": "coli_adenomatous_polyposis"
    },
    {
        "query": "Retrieve the relationships that connect howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light and bodies_howell_jolly",
        "subject": "howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light",
        "object": "bodies_howell_jolly"
    },
    {
        "query": "Retrieve the relationships that connect chromatins and nucleoplasm",
        "subject": "chromatins",
        "object": "nucleoplasm"
    },
    {
        "query": "Retrieve the relationships that connect tracheas and oesophagus",
        "subject": "tracheas",
        "object": "oesophagus"
    },
    {
        "query": "Retrieve the relationships that connect imaging_study_imp_pt_xxx_nar_radiology and imaging_study",
        "subject": "imaging_study_imp_pt_xxx_nar_radiology",
        "object": "imaging_study"
    },
    {
        "query": "Retrieve the relationships that connect sulcal_segment_of_precentral_gyrus and structure_of_precentral_sulcus",
        "subject": "sulcal_segment_of_precentral_gyrus",
        "object": "structure_of_precentral_sulcus"
    },
    {
        "query": "Retrieve the relationships that connect lactobacillales and lactic_acid_bacteria_oral_pwd",
        "subject": "lactobacillales",
        "object": "lactic_acid_bacteria_oral_pwd"
    },
    {
        "query": "Retrieve the relationships that connect chemotactic_factor_inactivator and inactivator_chemotaxis_function",
        "subject": "chemotactic_factor_inactivator",
        "object": "inactivator_chemotaxis_function"
    },
    {
        "query": "Retrieve the relationships that connect ampicillins and ampicillin_ophthalmic_ointment",
        "subject": "ampicillins",
        "object": "ampicillin_ophthalmic_ointment"
    }
]


In [ ]:
async def query_sparql_type4(predicate, obj):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT ?subject ?predicate ?object
    WHERE {{ ?subject umls:{predicate} umls:{obj}.
        BIND(umls:{obj} AS ?object)
        BIND(umls:{predicate} AS ?predicate)
      }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        extr = extract_triples(ret)
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_query_type4(queries, prompt_level, local_search, query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}


    for query_data in queries:
        query = query_data['query']
        predicate = query_data['predicate']
        object1 = query_data['object']

        ground_truth_triples = await query_sparql_type4(predicate, object1)
        print(f"\nQuery: {query}:")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(ground_truth_triples, model_response_triples)

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }


    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results





In [ ]:
queries_type4 = [
    {
        "query": "Find the subject where the predicate is 'contains' and the object is 't8_segment_of_esophagus'.",
        "predicate": "contains",
        "object": "t8_segment_of_esophagus"
    },
    {
        "query": "Find the subject where the predicate is 'contains' and the object is 'right_hyoglossus'.",
        "predicate": "contains",
        "object": "right_hyoglossus"
    },
    {
        "query": "Find the subject where the predicate is 'surrounds' and the object is 'cytoplasms'.",
        "predicate": "surrounds",
        "object": "cytoplasms"
    },
    {
        "query": "Find the subject where the predicate is 'surrounds' and the object is 'nucleolus_cell'.",
        "predicate": "surrounds",
        "object": "nucleolus_cell"
    },
    {
        "query": "Find the subject where the predicate is 'conceptual_part_of' and the object is 'nebraska_geographic_location'.",
        "predicate": "conceptual_part_of",
        "object": "nebraska_geographic_location"
    },
    {
        "query": "Find the subject where the predicate is 'conceptual_part_of' and the object is 'dilate'.",
        "predicate": "conceptual_part_of",
        "object": "dilate"
    },
    {
        "query": "Find the subject where the predicate is 'result_of' and the object is 'gastrointestinal_endoscopy'.",
        "predicate": "result_of",
        "object": "gastrointestinal_endoscopy"
    },
    {
        "query": "Find the subject where the predicate is 'branch_of' and the object is 'aortic_arch_structure'.",
        "predicate": "branch_of",
        "object": "aortic_arch_structure"
    },
    {
        "query": "Find the subject where the predicate is 'branch_of' and the object is 'root_of_uwda_hierarchy'.",
        "predicate": "branch_of",
        "object": "root_of_uwda_hierarchy"
    },
    {
        "query": "Find the subject where the predicate is 'occurs_in' and the object is 'synapsis'.",
        "predicate": "occurs_in",
        "object": "synapsis"
    },
    {
        "query": "Find the subject where the predicate is 'occurs_in' and the object is 'organelle'.",
        "predicate": "occurs_in",
        "object": "organelle"
    },
    {
        "query": "Find the subject where the predicate is 'degree_of' and the object is 'type_of_bleeding_of_gastric_ulcer'.",
        "predicate": "degree_of",
        "object": "type_of_bleeding_of_gastric_ulcer"
    },
    {
        "query": "Find the subject where the predicate is 'degree_of' and the object is 'type_of_bleeding_of_duodenal_ulcer'.",
        "predicate": "degree_of",
        "object": "type_of_bleeding_of_duodenal_ulcer"
    },
    {
        "query": "Find the subject where the predicate is 'uses' and the object is 'polypectomy_forceps'.",
        "predicate": "uses",
        "object": "polypectomy_forceps"
    },
    {
        "query": "Find the subject where the predicate is 'uses' and the object is 'biopsy_instruments'.",
        "predicate": "uses",
        "object": "biopsy_instruments"
    },
    {
        "query": "Find the subject where the predicate is 'manifestation_of' and the object is 'epidermolysis_bullosa_pretibial'.",
        "predicate": "manifestation_of",
        "object": "epidermolysis_bullosa_pretibial"
    },
    {
        "query": "Find the subject where the predicate is 'manifestation_of' and the object is 'coli_adenomatous_polyposis'.",
        "predicate": "manifestation_of",
        "object": "coli_adenomatous_polyposis"
    },
    {
        "query": "Find the subject where the predicate is 'measures' and the object is 'bodies_howell_jolly'.",
        "predicate": "measures",
        "object": "bodies_howell_jolly"
    },
    {
        "query": "Find the subject where the predicate is 'part_of' and the object is 'nucleoplasm'.",
        "predicate": "part_of",
        "object": "nucleoplasm"
    },
    {
        "query": "Find the subject where the predicate is 'adjacent_to' and the object is 'oesophagus'.",
        "predicate": "adjacent_to",
        "object": "oesophagus"
    },
    {
        "query": "Find the subject where the predicate is 'associated_with' and the object is 'imaging_study'.",
        "predicate": "associated_with",
        "object": "imaging_study"
    },
    {
        "query": "Find the subject where the predicate is 'adjacent_to' and the object is 'structure_of_precentral_sulcus'.",
        "predicate": "adjacent_to",
        "object": "structure_of_precentral_sulcus"
    },
    {
        "query": "Find the subject where the predicate is 'ingredient_of' and the object is 'lactic_acid_bacteria_oral_pwd'.",
        "predicate": "ingredient_of",
        "object": "lactic_acid_bacteria_oral_pwd"
    },
    {
        "query": "Find the subject where the predicate is 'exhibits' and the object is 'inactivator_chemotaxis_function'.",
        "predicate": "exhibits",
        "object": "inactivator_chemotaxis_function"
    },
    {
        "query": "Find the subject where the predicate is 'ingredient_of' and the object is 'ampicillin_ophthalmic_ointment'.",
        "predicate": "ingredient_of",
        "object": "ampicillin_ophthalmic_ointment"
    }
]


In [ ]:
async def query_sparql_type5(subject):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>

    SELECT ?subject ?predicate ?object
    WHERE {{umls:{subject} ?predicate ?object.
        BIND(umls:{subject} AS ?subject)
      }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        extr = extract_triples(ret)
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []



async def compute_accuracy_for_query_type5(queries, prompt_level, local_search, query_name, model_name, prompt_name):
    """
    Computes accuracy, precision, recall, and F1-score for query results.
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']

        ground_truth_triples = await query_sparql_type5(subject)

        print(f"\nQuery: {query}:")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)
        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(ground_truth_triples, model_response_triples)
        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)
        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")

    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results




In [ ]:
queries_type5 = [
    {
        "query": "Find the relationships and attributes for 'posterior_mediastinum'.",
        "subject": "posterior_mediastinum"
    },
    {
        "query": "Find the relationships and attributes for 'right_submandibular_triangle'.",
        "subject": "right_submandibular_triangle"
    },
    {
        "query": "Find the relationships and attributes for 'plasma_membranes'.",
        "subject": "plasma_membranes"
    },
    {
        "query": "Find the relationships and attributes for 'nucleoplasm'.",
        "subject": "nucleoplasm"
    },
    {
        "query": "Find the relationships and attributes for 'sioux_county_ne'.",
        "subject": "sioux_county_ne"
    },
    {
        "query": "Find the relationships and attributes for 'dilation_size'.",
        "subject": "dilation_size"
    },
    {
        "query": "Find the relationships and attributes for 'complications'.",
        "subject": "complications"
    },
    {
        "query": "Find the relationships and attributes for 'absent_left_common_carotid_artery'.",
        "subject": "absent_left_common_carotid_artery"
    },
    {
        "query": "Find the relationships and attributes for 'nerves_sacral'.",
        "subject": "nerves_sacral"
    },
    {
        "query": "Find the relationships and attributes for 'synaptic_signaling'.",
        "subject": "synaptic_signaling"
    },
    {
        "query": "Find the relationships and attributes for 'maintenance_of_protein_localisation_to_organelle'.",
        "subject": "maintenance_of_protein_localisation_to_organelle"
    },
    {
        "query": "Find the relationships and attributes for 'type_iii_no_bleeding_of_gastric_ulcer'.",
        "subject": "type_iii_no_bleeding_of_gastric_ulcer"
    },
    {
        "query": "Find the relationships and attributes for 'type_iii_no_bleeding_of_duodenal_ulcer'.",
        "subject": "type_iii_no_bleeding_of_duodenal_ulcer"
    },
    {
        "query": "Find the relationships and attributes for 'polypectomy'.",
        "subject": "polypectomy"
    },
    {
        "query": "Find the relationships and attributes for 'biopsy_nos'.",
        "subject": "biopsy_nos"
    },
    {
        "query": "Find the relationships and attributes for 'hypertrophic_cicatrix'.",
        "subject": "hypertrophic_cicatrix"
    },
    {
        "query": "Find the relationships and attributes for 'cheloid'.",
        "subject": "cheloid"
    },
    {
        "query": "Find the relationships and attributes for 'howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light'.",
        "subject": "howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light"
    },
    {
        "query": "Find the relationships and attributes for 'chromatins'.",
        "subject": "chromatins"
    },
    {
        "query": "Find the relationships and attributes for 'tracheas'.",
        "subject": "tracheas"
    },
    {
        "query": "Find the relationships and attributes for 'imaging_study_imp_pt_xxx_nar_radiology'.",
        "subject": "imaging_study_imp_pt_xxx_nar_radiology"
    },
    {
        "query": "Find the relationships and attributes for 'sulcal_segment_of_precentral_gyrus'.",
        "subject": "sulcal_segment_of_precentral_gyrus"
    },
    {
        "query": "Find the relationships and attributes for 'lactobacillales'.",
        "subject": "lactobacillales"
    },
    {
        "query": "Find the relationships and attributes for 'chemotactic_factor_inactivator'.",
        "subject": "chemotactic_factor_inactivator"
    },
    {
        "query": "Find the relationships and attributes for 'ampicillins'.",
        "subject": "ampicillins"
    }
]


In [ ]:
async def query_sparql_type6(predicate):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)
    query = f"""
      PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
      PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
      PREFIX umls: <http://www.nlm.nih.gov/research/umls>


      SELECT ?subject ?predicate ?object
      WHERE {{
          {{ ?subject umls:{predicate} ?object. }}
          UNION
          {{ ?object umls:{predicate} ?subject. }}
          BIND(umls:{predicate} AS ?predicate)
      }}
      """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        extr = extract_triples(ret)
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []


async def compute_accuracy_for_type6(queries, prompt_level, local_search, query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']
        predicate = query_data['predicate']

        ground_truth_triples = await query_sparql_type6(predicate)
        print(f"\nQuery: {query}:")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(ground_truth_triples, model_response_triples)

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")
    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results




In [ ]:
queries_type6 = [
    {
        "query": "For the predicate 'location_of', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "location_of"
    },
    {
        "query": "For the predicate 'manifestation_of', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "manifestation_of"
    },
    {
        "query": "For the predicate 'part_of', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "part_of"
    },
    {
        "query": "For the predicate 'associated_with', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "associated_with"
    },
    {
        "query": "For the predicate 'evaluation_of', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "evaluation_of"
    },
    {
        "query": "For the predicate 'method_of', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "method_of"
    },
    {
        "query": "For the predicate 'ingredient_of', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "ingredient_of"
    },
    {
        "query": "For the predicate 'branch_of', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "branch_of"
    },
    {
        "query": "For the predicate 'occurs_in', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "occurs_in"
    },
    {
        "query": "For the predicate 'exhibits', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "exhibits"
    },
    {
        "query": "For the predicate 'adjacent_to', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "adjacent_to"
    },
    {
        "query": "For the predicate 'connected_to', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "connected_to"
    },
    {
        "query": "For the predicate 'contains', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "contains"
    },
    {
        "query": "For the predicate 'surrounds', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "surrounds"
    },
    {
        "query": "For the predicate 'tributary_of', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "tributary_of"
    },
    {
        "query": "For the predicate 'analyzes', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "analyzes"
    },
    {
        "query": "For the predicate 'measures', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "measures"
    },
    {
        "query": "For the predicate 'consists_of', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "consists_of"
    },
    {
        "query": "For the predicate 'diagnoses', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "diagnoses"
    },
    {
        "query": "For the predicate 'uses', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "uses"
    },
    {
        "query": "For the predicate 'degree_of', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "degree_of"
    },
    {
        "query": "For the predicate 'conceptual_part_of', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "conceptual_part_of"
    },
    {
        "query": "For the predicate 'result_of', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "result_of"
    },
    {
        "query": "For the predicate 'treats', return all entities that are directly connected by it, even if they are subject or object.",
        "predicate": "treats"
    }
]


In [ ]:
def extract_triples_type_describe(sparql_output):
    triples = []
    lines = sparql_output.split("\n")

    for line in lines:
        match = re.findall(r'<http://www.nlm.nih.gov/research/umls([^>]*)>', line)
        if len(match) == 3:
            triples.append(f"{match[0]} {match[1]} {match[2]}")

    return triples

async def query_sparql_type7(subject):
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    DESCRIBE umls:{subject}
    """

    sparql.setQuery(query)

    try:
        ret = sparql.query().convert()
        if isinstance(ret, bytes):
            ret = ret.decode('utf-8')

            extr = (extract_triples_type_describe(ret))
            print(extr)
            print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []


async def compute_accuracy_type7(queries, prompt_level, local_search, query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']

        ground_truth_triples = await query_sparql_type7(subject)
        print(f"\nQuery: {query}:")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)
        common, extra, precision, recall, f1_score = find_common_and_extra_elements2(ground_truth_triples, model_response_triples)
        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)

        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")

    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }


    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results





In [ ]:
queries_type7 = [
    {
        "query": "What do you know about colostomy_stoma? Return all triples even when it is subject or object.",
        "subject": "colostomy_stoma"
    },
    {
        "query": "What do you know about rectal_pouch? Return all triples even when it is subject or object.",
        "subject": "rectal_pouch"
    },
    {
        "query": "What do you know about sulcal_segment_of_middle_frontal_gyrus? Return all triples even when it is subject or object.",
        "subject": "sulcal_segment_of_middle_frontal_gyrus"
    },
    {
        "query": "What do you know about wall_of_biventricular_part_of_heart? Return all triples even when it is subject or object.",
        "subject": "wall_of_biventricular_part_of_heart"
    },
    {
        "query": "What do you know about duct_cannulation? Return all triples even when it is subject or object.",
        "subject": "duct_cannulation"
    },
    {
        "query": "What do you know about oesophagogastroduodenoscopy? Return all triples even when it is subject or object.",
        "subject": "oesophagogastroduodenoscopy"
    },
    {
        "query": "What do you know about cytochrome_b5? Return all triples even when it is subject or object.",
        "subject": "cytochrome_b5"
    },
    {
        "query": "What do you know about ascorbic_acid? Return all triples even when it is subject or object.",
        "subject": "ascorbic_acid"
    },
    {
        "query": "What do you know about retrograde_cholangiopancreatography_endoscopic? Return all triples even when it is subject or object.",
        "subject": "retrograde_cholangiopancreatography_endoscopic"
    },
    {
        "query": "What do you know about perfumes? Return all triples even when it is subject or object.",
        "subject": "perfumes"
    },
    {
        "query": "What do you know about aggressive_behaviors? Return all triples even when it is subject or object.",
        "subject": "aggressive_behaviors"
    },
    {
        "query": "What do you know about minimum_data_set_mds_version_3? Return all triples even when it is subject or object.",
        "subject": "minimum_data_set_mds_version_3"
    },
    {
        "query": "What do you know about structure_of_carotid_sheath? Return all triples even when it is subject or object.",
        "subject": "structure_of_carotid_sheath"
    },
    {
        "query": "What do you know about polypectomy? Return all triples even when it is subject or object.",
        "subject": "polypectomy"
    },
    {
        "query": "What do you know about thermal_therapy? Return all triples even when it is subject or object.",
        "subject": "thermal_therapy"
    },
    {
        "query": "What do you know about cavity_abdominal? Return all triples even when it is subject or object.",
        "subject": "cavity_abdominal"
    },
    {
        "query": "What do you know about left_nasal_bone? Return all triples even when it is subject or object.",
        "subject": "left_nasal_bone"
    },
    {
        "query": "What do you know about structure_of_umbilical_artery? Return all triples even when it is subject or object.",
        "subject": "structure_of_umbilical_artery"
    },
    {
        "query": "What do you know about routes_drug_administration? Return all triples even when it is subject or object.",
        "subject": "routes_drug_administration"
    },
    {
        "query": "What do you know about aloe_vera_gel? Return all triples even when it is subject or object.",
        "subject": "aloe_vera_gel"
    },
    {
        "query": "What do you know about posterior_mediastinum? Return all triples even when it is subject or object.",
        "subject": "posterior_mediastinum"
    },
    {
        "query": "What do you know about descending_part_of_duodenum? Return all triples even when it is subject or object.",
        "subject": "descending_part_of_duodenum"
    },
    {
        "query": "What do you know about caecum? Return all triples even when it is subject or object.",
        "subject": "caecum"
    },
    {
        "query": "What do you know about ataxia? Return all triples even when it is subject or object.",
        "subject": "ataxia"
    },
    {
        "query": "What do you know about assays_immunofluorescence? Return all triples even when it is subject or object.",
        "subject": "assays_immunofluorescence"
    }
]


In [ ]:
#STANDARD ASK QUERIES

In [ ]:

import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type8(subject, predicate, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    ASK
    WHERE {{
      umls:{subject} umls:{predicate} umls:{objectf}.
      BIND(umls:{predicate} AS ?predicate)
      BIND(umls:{subject} AS ?subject)
       BIND(umls:{objectf} AS ?object)
    }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type8(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0
    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']
        predicate = query_data['predicate']
        objectf = query_data['object']
        ground_truth_triples = await query_sparql_type8(subject, predicate, objectf)
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)
        gt_bool = normalize_to_bool(ground_truth_triples)
        print(clean_text)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        print("model bool")
        print(model_bool)
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)
    return x

In [ ]:
async def compute_accuracy_for_query_type8(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']
        predicate = query_data['predicate']
        objectf = query_data['object']

        ground_truth_bool = await query_sparql_type8(subject, predicate, objectf)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)
        gt_bool = normalize_to_bool(ground_truth_bool)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "subject": subject,
            "predicate": predicate,
            "object": objectf,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
queries_type_a1 = [
    {
        "query": "*Is it true that* posterior_mediastinum contains t8_segment_of_esophagus?",
        "subject": "posterior_mediastinum",
        "predicate": "contains",
        "object": "t8_segment_of_esophagus"
    },
    {
        "query": "*Is it true that* right_submandibular_triangle contains right_hyoglossus?",
        "subject": "right_submandibular_triangle",
        "predicate": "contains",
        "object": "right_hyoglossus"
    },
    {
        "query": "*Is it true that* plasma_membranes surrounds cytoplasms?",
        "subject": "plasma_membranes",
        "predicate": "surrounds",
        "object": "cytoplasms"
    },
    {
        "query": "*Is it true that* nucleoplasm surrounds nucleolus_cell?",
        "subject": "nucleoplasm",
        "predicate": "surrounds",
        "object": "nucleolus_cell"
    },
    {
        "query": "*Is it true that* sioux_county_ne conceptual_part_of nebraska_geographic_location?",
        "subject": "sioux_county_ne",
        "predicate": "conceptual_part_of",
        "object": "nebraska_geographic_location"
    },
    {
        "query": "*Is it true that* dilation_size conceptual_part_of dilate?",
        "subject": "dilation_size",
        "predicate": "conceptual_part_of",
        "object": "dilate"
    },
    {
        "query": "*Is it true that* complications result_of gastrointestinal_endoscopy?",
        "subject": "complications",
        "predicate": "result_of",
        "object": "gastrointestinal_endoscopy"
    },
    {
        "query": "*Is it true that* absent_left_common_carotid_artery branch_of aortic_arch_structure?",
        "subject": "absent_left_common_carotid_artery",
        "predicate": "branch_of",
        "object": "aortic_arch_structure"
    },
    {
        "query": "*Is it true that* nerves_sacral branch_of root_of_uwda_hierarchy?",
        "subject": "nerves_sacral",
        "predicate": "branch_of",
        "object": "root_of_uwda_hierarchy"
    },
    {
        "query": "*Is it true that* synaptic_signaling occurs_in synapsis?",
        "subject": "synaptic_signaling",
        "predicate": "occurs_in",
        "object": "synapsis"
    },
    {
        "query": "*Is it true that* maintenance_of_protein_localisation_to_organelle occurs_in organelle?",
        "subject": "maintenance_of_protein_localisation_to_organelle",
        "predicate": "occurs_in",
        "object": "organelle"
    },
    {
        "query": "*Is it true that* type_iii_no_bleeding_of_gastric_ulcer degree_of type_of_bleeding_of_gastric_ulcer?",
        "subject": "type_iii_no_bleeding_of_gastric_ulcer",
        "predicate": "degree_of",
        "object": "type_of_bleeding_of_gastric_ulcer"
    },
    {
        "query": "*Is it true that* type_iii_no_bleeding_of_duodenal_ulcer degree_of type_of_bleeding_of_duodenal_ulcer?",
        "subject": "type_iii_no_bleeding_of_duodenal_ulcer",
        "predicate": "degree_of",
        "object": "type_of_bleeding_of_duodenal_ulcer"
    },
    {
        "query": "*Is it true that* polypectomy uses polypectomy_forceps?",
        "subject": "polypectomy",
        "predicate": "uses",
        "object": "polypectomy_forceps"
    },
    {
        "query": "*Is it true that* biopsy_nos uses biopsy_instruments?",
        "subject": "biopsy_nos",
        "predicate": "uses",
        "object": "biopsy_instruments"
    },
    {
        "query": "*Is it true that* hypertrophic_cicatrix manifestation_of epidermolysis_bullosa_pretibial?",
        "subject": "hypertrophic_cicatrix",
        "predicate": "manifestation_of",
        "object": "epidermolysis_bullosa_pretibial"
    },
    {
        "query": "*Is it true that* cheloid manifestation_of coli_adenomatous_polyposis?",
        "subject": "cheloid",
        "predicate": "manifestation_of",
        "object": "coli_adenomatous_polyposis"
    },
    {
        "query": "*Is it true that* howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light measures bodies_howell_jolly?",
        "subject": "howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light",
        "predicate": "measures",
        "object": "bodies_howell_jolly"
    },
    {
        "query": "*Is it true that* chromatins part_of nucleoplasm?",
        "subject": "chromatins",
        "predicate": "part_of",
        "object": "nucleoplasm"
    },
    {
        "query": "*Is it true that* tracheas adjacent_to oesophagus?",
        "subject": "tracheas",
        "predicate": "adjacent_to",
        "object": "oesophagus"
    },
    {
        "query": "*Is it true that* imaging_study_imp_pt_xxx_nar_radiology associated_with imaging_study?",
        "subject": "imaging_study_imp_pt_xxx_nar_radiology",
        "predicate": "associated_with",
        "object": "imaging_study"
    },
    {
        "query": "*Is it true that* sulcal_segment_of_precentral_gyrus adjacent_to structure_of_precentral_sulcus?",
        "subject": "sulcal_segment_of_precentral_gyrus",
        "predicate": "adjacent_to",
        "object": "structure_of_precentral_sulcus"
    },
    {
        "query": "*Is it true that* lactobacillales ingredient_of lactic_acid_bacteria_oral_pwd?",
        "subject": "lactobacillales",
        "predicate": "ingredient_of",
        "object": "lactic_acid_bacteria_oral_pwd"
    },
    {
        "query": "*Is it true that* chemotactic_factor_inactivator exhibits inactivator_chemotaxis_function?",
        "subject": "chemotactic_factor_inactivator",
        "predicate": "exhibits",
        "object": "inactivator_chemotaxis_function"
    },
    {
        "query": "*Is it true that* ampicillins ingredient_of ampicillin_ophthalmic_ointment?",
        "subject": "ampicillins",
        "predicate": "ingredient_of",
        "object": "ampicillin_ophthalmic_ointment"
    },
    {
        "query": "*Is it true that* polypectomy uses polypectomy_forceps?",
        "subject": "polypectomy",
        "predicate": "uses",
        "object": "polypectomy_forceps"
    },
    {
        "query": "*Is it true that* biopsy_nos uses biopsy_instruments?",
        "subject": "biopsy_nos",
        "predicate": "uses",
        "object": "biopsy_instruments"
    },
]


In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type9(subject, predicate):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    ASK
    WHERE {{
      umls:{subject} umls:{predicate} ?object.
      BIND(umls:{predicate} AS ?predicate)
      BIND(umls:{subject} AS ?subject)

    }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type9(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0
    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']
        predicate = query_data['predicate']

        ground_truth_triples = await query_sparql_type9(subject, predicate)
        print("ground")
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        clean_text = strip_prefixes(response_text)
        print(model_response_triples)
        gt_bool = normalize_to_bool(ground_truth_triples)
        print(clean_text)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        print(model_bool)
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)
    return x



In [ ]:
async def compute_accuracy_for_query_type9(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']
        predicate = query_data['predicate']

        ground_truth_bool = await query_sparql_type9(subject, predicate)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)

        gt_bool = normalize_to_bool(ground_truth_bool)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "subject": subject,
            "predicate": predicate,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
queries_type_2a = [
    {
        "query": "*Is it true that* colostomy_stoma appears in relation with location_of?",
        "subject": "colostomy_stoma",
        "predicate": "location_of"
    },
    {
        "query": "*Is it true that* rectal_pouch appears in relation with location_of?",
        "subject": "rectal_pouch",
        "predicate": "location_of"
    },
    {
        "query": "*Is it true that* sulcal_segment_of_middle_frontal_gyrus appears in relation with adjacent_to?",
        "subject": "sulcal_segment_of_middle_frontal_gyrus",
        "predicate": "adjacent_to"
    },
    {
        "query": "*Is it true that* wall_of_biventricular_part_of_heart appears in relation with surrounds?",
        "subject": "wall_of_biventricular_part_of_heart",
        "predicate": "surrounds"
    },
    {
        "query": "*Is it true that* duct_cannulation appears in relation with uses?",
        "subject": "duct_cannulation",
        "predicate": "uses"
    },
    {
        "query": "*Is it true that* oesophagogastroduodenoscopy appears in relation with diagnoses?",
        "subject": "oesophagogastroduodenoscopy",
        "predicate": "diagnoses"
    },
    {
        "query": "*Is it true that* cytochrome_b5 appears in relation with exhibits?",
        "subject": "cytochrome_b5",
        "predicate": "exhibits"
    },
    {
        "query": "*Is it true that* ascorbic_acid appears in relation with ingredient_of?",
        "subject": "ascorbic_acid",
        "predicate": "ingredient_of"
    },
    {
        "query": "*Is it true that* retrograde_cholangiopancreatography_endoscopic appears in relation with diagnoses?",
        "subject": "retrograde_cholangiopancreatography_endoscopic",
        "predicate": "diagnoses"
    },
    {
        "query": "*Is it true that* perfumes appears in relation with ingredient_of?",
        "subject": "perfumes",
        "predicate": "ingredient_of"
    },
    {
        "query": "*Is it true that* aggressive_behaviors appears in relation with manifestation_of?",
        "subject": "aggressive_behaviors",
        "predicate": "manifestation_of"
    },
    {
        "query": "*Is it true that* minimum_data_set_mds_version_3 appears in relation with method_of?",
        "subject": "minimum_data_set_mds_version_3",
        "predicate": "method_of"
    },
    {
        "query": "*Is it true that* structure_of_carotid_sheath appears in relation with surrounds?",
        "subject": "structure_of_carotid_sheath",
        "predicate": "surrounds"
    },
    {
        "query": "*Is it true that* polypectomy appears in relation with uses?",
        "subject": "polypectomy",
        "predicate": "uses"
    },
    {
        "query": "*Is it true that* thermal_therapy appears in relation with uses?",
        "subject": "thermal_therapy",
        "predicate": "uses"
    },
    {
        "query": "*Is it true that* cavity_abdominal appears in relation with contains?",
        "subject": "cavity_abdominal",
        "predicate": "contains"
    },
    {
        "query": "*Is it true that* left_nasal_bone appears in relation with connected_to?",
        "subject": "left_nasal_bone",
        "predicate": "connected_to"
    },
    {
        "query": "*Is it true that* structure_of_umbilical_artery appears in relation with branch_of?",
        "subject": "structure_of_umbilical_artery",
        "predicate": "branch_of"
    },
    {
        "query": "*Is it true that* routes_drug_administration appears in relation with associated_with?",
        "subject": "routes_drug_administration",
        "predicate": "associated_with"
    },
    {
        "query": "*Is it true that* aloe_vera_gel appears in relation with ingredient_of?",
        "subject": "aloe_vera_gel",
        "predicate": "ingredient_of"
    },
    {
        "query": "*Is it true that* posterior_mediastinum appears in relation with contains?",
        "subject": "posterior_mediastinum",
        "predicate": "contains"
    },
    {
        "query": "*Is it true that* descending_part_of_duodenum appears in relation with location_of?",
        "subject": "descending_part_of_duodenum",
        "predicate": "location_of"
    },
    {
        "query": "*Is it true that* caecum appears in relation with location_of?",
        "subject": "caecum",
        "predicate": "location_of"
    },
    {
        "query": "*Is it true that* ataxia appears in relation with manifestation_of?",
        "subject": "ataxia",
        "predicate": "manifestation_of"
    },
    {
        "query": "*Is it true that* assays_immunofluorescence appears in relation with method_of?",
        "subject": "assays_immunofluorescence",
        "predicate": "method_of"
    }
]



In [ ]:
queries_type_3a = [
  {
    "query": "*Is it true that* the contains appears with the value of t8_segment_of_esophagus?",
    "predicate": "contains",
    "object": "t8_segment_of_esophagus"
  },
  {
    "query": "*Is it true that* the contains appears with the value of right_hyoglossus?",
    "predicate": "contains",
    "object": "right_hyoglossus"
  },
  {
    "query": "*Is it true that* the surrounds appears with the value of cytoplasms?",
    "predicate": "surrounds",
    "object": "cytoplasms"
  },
  {
    "query": "*Is it true that* the surrounds appears with the value of nucleolus_cell?",
    "predicate": "surrounds",
    "object": "nucleolus_cell"
  },
  {
    "query": "*Is it true that* the conceptual_part_of appears with the value of nebraska_geographic_location?",
    "predicate": "conceptual_part_of",
    "object": "nebraska_geographic_location"
  },
  {
    "query": "*Is it true that* the conceptual_part_of appears with the value of dilate?",
    "predicate": "conceptual_part_of",
    "object": "dilate"
  },
  {
    "query": "*Is it true that* the result_of appears with the value of gastrointestinal_endoscopy?",
    "predicate": "result_of",
    "object": "gastrointestinal_endoscopy"
  },
  {
    "query": "*Is it true that* the branch_of appears with the value of aortic_arch_structure?",
    "predicate": "branch_of",
    "object": "aortic_arch_structure"
  },
  {
    "query": "*Is it true that* the branch_of appears with the value of root_of_uwda_hierarchy?",
    "predicate": "branch_of",
    "object": "root_of_uwda_hierarchy"
  },
  {
    "query": "*Is it true that* the occurs_in appears with the value of synapsis?",
    "predicate": "occurs_in",
    "object": "synapsis"
  },
  {
    "query": "*Is it true that* the occurs_in appears with the value of organelle?",
    "predicate": "occurs_in",
    "object": "organelle"
  },
  {
    "query": "*Is it true that* the degree_of appears with the value of type_of_bleeding_of_gastric_ulcer?",
    "predicate": "degree_of",
    "object": "type_of_bleeding_of_gastric_ulcer"
  },
  {
    "query": "*Is it true that* the degree_of appears with the value of type_of_bleeding_of_duodenal_ulcer?",
    "predicate": "degree_of",
    "object": "type_of_bleeding_of_duodenal_ulcer"
  },
  {
    "query": "*Is it true that* the uses appears with the value of polypectomy_forceps?",
    "predicate": "uses",
    "object": "polypectomy_forceps"
  },
  {
    "query": "*Is it true that* the uses appears with the value of biopsy_instruments?",
    "predicate": "uses",
    "object": "biopsy_instruments"
  },
  {
    "query": "*Is it true that* the manifestation_of appears with the value of epidermolysis_bullosa_pretibial?",
    "predicate": "manifestation_of",
    "object": "epidermolysis_bullosa_pretibial"
  },
  {
    "query": "*Is it true that* the manifestation_of appears with the value of coli_adenomatous_polyposis?",
    "predicate": "manifestation_of",
    "object": "coli_adenomatous_polyposis"
  },
  {
    "query": "*Is it true that* the measures appears with the value of bodies_howell_jolly?",
    "predicate": "measures",
    "object": "bodies_howell_jolly"
  },
  {
    "query": "*Is it true that* the part_of appears with the value of nucleoplasm?",
    "predicate": "part_of",
    "object": "nucleoplasm"
  },
  {
    "query": "*Is it true that* the adjacent_to appears with the value of oesophagus?",
    "predicate": "adjacent_to",
    "object": "oesophagus"
  },
  {
    "query": "*Is it true that* the associated_with appears with the value of imaging_study?",
    "predicate": "associated_with",
    "object": "imaging_study"
  },
  {
    "query": "*Is it true that* the adjacent_to appears with the value of structure_of_precentral_sulcus?",
    "predicate": "adjacent_to",
    "object": "structure_of_precentral_sulcus"
  },
  {
    "query": "*Is it true that* the ingredient_of appears with the value of lactic_acid_bacteria_oral_pwd?",
    "predicate": "ingredient_of",
    "object": "lactic_acid_bacteria_oral_pwd"
  },
  {
    "query": "*Is it true that* the exhibits appears with the value of inactivator_chemotaxis_function?",
    "predicate": "exhibits",
    "object": "inactivator_chemotaxis_function"
  },
  {
    "query": "*Is it true that* the ingredient_of appears with the value of ampicillin_ophthalmic_ointment?",
    "predicate": "ingredient_of",
    "object": "ampicillin_ophthalmic_ointment"
  }
]


In [ ]:
#1bun
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type10(predicate, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    ASK
    WHERE {{
       ?subject umls:{predicate} umls:{objectf}.
      BIND(umls:{predicate} AS ?predicate)
      BIND(umls:{objectf} AS ?object)

    }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type10(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0
    for query_data in queries:
        query = query_data['query']
        objectf = query_data['object']
        predicate = query_data['predicate']

        ground_truth_triples = await query_sparql_type10(predicate, objectf)
        print("ground")
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)
        gt_bool = normalize_to_bool(ground_truth_triples)
        print(clean_text)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        print("model bool")
        print(model_bool)
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)
    return x




In [ ]:
async def compute_accuracy_for_query_type10(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        objectf = query_data['object']
        predicate = query_data['predicate']

        ground_truth_bool = await query_sparql_type10(predicate, objectf)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)


        gt_bool = normalize_to_bool(ground_truth_bool)

        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "predicate": predicate,
            "object": objectf,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
queries_type_4a = [
  {
    "query": "*Is it true that* the contains appears near t8_segment_of_esophagus?",
    "predicate": "contains",
    "object": "t8_segment_of_esophagus"
  },
  {
    "query": "*Is it true that* the contains appears near right_hyoglossus?",
    "predicate": "contains",
    "object": "right_hyoglossus"
  },
  {
    "query": "*Is it true that* the surrounds appears near cytoplasms?",
    "predicate": "surrounds",
    "object": "cytoplasms"
  },
  {
    "query": "*Is it true that* the surrounds appears near nucleolus_cell?",
    "predicate": "surrounds",
    "object": "nucleolus_cell"
  },
  {
    "query": "*Is it true that* the conceptual_part_of appears near nebraska_geographic_location?",
    "predicate": "conceptual_part_of",
    "object": "nebraska_geographic_location"
  },
  {
    "query": "*Is it true that* the conceptual_part_of appears near dilate?",
    "predicate": "conceptual_part_of",
    "object": "dilate"
  },
  {
    "query": "*Is it true that* the result_of appears near gastrointestinal_endoscopy?",
    "predicate": "result_of",
    "object": "gastrointestinal_endoscopy"
  },
  {
    "query": "*Is it true that* the branch_of appears near aortic_arch_structure?",
    "predicate": "branch_of",
    "object": "aortic_arch_structure"
  },
  {
    "query": "*Is it true that* the branch_of appears near root_of_uwda_hierarchy?",
    "predicate": "branch_of",
    "object": "root_of_uwda_hierarchy"
  },
  {
    "query": "*Is it true that* the occurs_in appears near synapsis?",
    "predicate": "occurs_in",
    "object": "synapsis"
  },
  {
    "query": "*Is it true that* the occurs_in appears near organelle?",
    "predicate": "occurs_in",
    "object": "organelle"
  },
  {
    "query": "*Is it true that* the degree_of appears near type_of_bleeding_of_gastric_ulcer?",
    "predicate": "degree_of",
    "object": "type_of_bleeding_of_gastric_ulcer"
  },
  {
    "query": "*Is it true that* the degree_of appears near type_of_bleeding_of_duodenal_ulcer?",
    "predicate": "degree_of",
    "object": "type_of_bleeding_of_duodenal_ulcer"
  },
  {
    "query": "*Is it true that* the uses appears near polypectomy_forceps?",
    "predicate": "uses",
    "object": "polypectomy_forceps"
  },
  {
    "query": "*Is it true that* the uses appears near biopsy_instruments?",
    "predicate": "uses",
    "object": "biopsy_instruments"
  },
  {
    "query": "*Is it true that* the manifestation_of appears near epidermolysis_bullosa_pretibial?",
    "predicate": "manifestation_of",
    "object": "epidermolysis_bullosa_pretibial"
  },
  {
    "query": "*Is it true that* the manifestation_of appears near coli_adenomatous_polyposis?",
    "predicate": "manifestation_of",
    "object": "coli_adenomatous_polyposis"
  },
  {
    "query": "*Is it true that* the measures appears near bodies_howell_jolly?",
    "predicate": "measures",
    "object": "bodies_howell_jolly"
  },
  {
    "query": "*Is it true that* the part_of appears near nucleoplasm?",
    "predicate": "part_of",
    "object": "nucleoplasm"
  },
  {
    "query": "*Is it true that* the adjacent_to appears near oesophagus?",
    "predicate": "adjacent_to",
    "object": "oesophagus"
  },
  {
    "query": "*Is it true that* the associated_with appears near imaging_study?",
    "predicate": "associated_with",
    "object": "imaging_study"
  },
  {
    "query": "*Is it true that* the adjacent_to appears near structure_of_precentral_sulcus?",
    "predicate": "adjacent_to",
    "object": "structure_of_precentral_sulcus"
  },
  {
    "query": "*Is it true that* the ingredient_of appears near lactic_acid_bacteria_oral_pwd?",
    "predicate": "ingredient_of",
    "object": "lactic_acid_bacteria_oral_pwd"
  },
  {
    "query": "*Is it true that* the exhibits appears near inactivator_chemotaxis_function?",
    "predicate": "exhibits",
    "object": "inactivator_chemotaxis_function"
  },
  {
    "query": "*Is it true that* the ingredient_of appears near ampicillin_ophthalmic_ointment?",
    "predicate": "ingredient_of",
    "object": "ampicillin_ophthalmic_ointment"
  }
]


In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type11(predicate, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
ASK
WHERE {{
    {{ umls:{objectf} umls:{predicate} ?ent . }}
    UNION
    {{
        ?end umls:{predicate} umls:{objectf}.
    }}
}}
"""
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print("ret is")
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type11(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0
    for query_data in queries:
        query = query_data['query']
        objectf = query_data['object']
        predicate = query_data['predicate']

        ground_truth_triples = await query_sparql_type11(predicate, objectf)
        print("ground")
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)
        gt_bool = normalize_to_bool(ground_truth_triples)
        print(clean_text)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        print("model bool")
        print(model_bool)
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)
    return x


In [ ]:
async def compute_accuracy_for_query_type11(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        objectf = query_data['object']
        predicate = query_data['predicate']

        ground_truth_bool = await query_sparql_type11(predicate, objectf)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)

        gt_bool = normalize_to_bool(ground_truth_bool)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "predicate": predicate,
            "object": objectf,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
async def compute_accuracy_for_query_type11(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        objectf = query_data['object']
        predicate = query_data['predicate']

        ground_truth_bool = await query_sparql_type11(predicate, objectf)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)

        # Ground truth -> bool
        gt_bool = normalize_to_bool(ground_truth_bool)

        # Model -> bool
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "predicate": predicate,
            "object": objectf,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
queries_type_5a = [
  {
    "query": "*Is it true that* there is a relationship that connects posterior_mediastinum and t8_segment_of_esophagus?",
    "subject": "posterior_mediastinum",
    "object": "t8_segment_of_esophagus"
  },
  {
    "query": "*Is it true that* there is a relationship that connects right_submandibular_triangle and right_hyoglossus?",
    "subject": "right_submandibular_triangle",
    "object": "right_hyoglossus"
  },
  {
    "query": "*Is it true that* there is a relationship that connects plasma_membranes and cytoplasms?",
    "subject": "plasma_membranes",
    "object": "cytoplasms"
  },
  {
    "query": "*Is it true that* there is a relationship that connects nucleoplasm and nucleolus_cell?",
    "subject": "nucleoplasm",
    "object": "nucleolus_cell"
  },
  {
    "query": "*Is it true that* there is a relationship that connects sioux_county_ne and nebraska_geographic_location?",
    "subject": "sioux_county_ne",
    "object": "nebraska_geographic_location"
  },
  {
    "query": "*Is it true that* there is a relationship that connects dilation_size and dilate?",
    "subject": "dilation_size",
    "object": "dilate"
  },
  {
    "query": "*Is it true that* there is a relationship that connects complications and gastrointestinal_endoscopy?",
    "subject": "complications",
    "object": "gastrointestinal_endoscopy"
  },
  {
    "query": "*Is it true that* there is a relationship that connects absent_left_common_carotid_artery and aortic_arch_structure?",
    "subject": "absent_left_common_carotid_artery",
    "object": "aortic_arch_structure"
  },
  {
    "query": "*Is it true that* there is a relationship that connects nerves_sacral and root_of_uwda_hierarchy?",
    "subject": "nerves_sacral",
    "object": "root_of_uwda_hierarchy"
  },
  {
    "query": "*Is it true that* there is a relationship that connects synaptic_signaling and synapsis?",
    "subject": "synaptic_signaling",
    "object": "synapsis"
  },
  {
    "query": "*Is it true that* there is a relationship that connects maintenance_of_protein_localisation_to_organelle and organelle?",
    "subject": "maintenance_of_protein_localisation_to_organelle",
    "object": "organelle"
  },
  {
    "query": "*Is it true that* there is a relationship that connects type_iii_no_bleeding_of_gastric_ulcer and type_of_bleeding_of_gastric_ulcer?",
    "subject": "type_iii_no_bleeding_of_gastric_ulcer",
    "object": "type_of_bleeding_of_gastric_ulcer"
  },
  {
    "query": "*Is it true that* there is a relationship that connects type_iii_no_bleeding_of_duodenal_ulcer and type_of_bleeding_of_duodenal_ulcer?",
    "subject": "type_iii_no_bleeding_of_duodenal_ulcer",
    "object": "type_of_bleeding_of_duodenal_ulcer"
  },
  {
    "query": "*Is it true that* there is a relationship that connects polypectomy and polypectomy_forceps?",
    "subject": "polypectomy",
    "object": "polypectomy_forceps"
  },
  {
    "query": "*Is it true that* there is a relationship that connects biopsy_nos and biopsy_instruments?",
    "subject": "biopsy_nos",
    "object": "biopsy_instruments"
  },
  {
    "query": "*Is it true that* there is a relationship that connects hypertrophic_cicatrix and epidermolysis_bullosa_pretibial?",
    "subject": "hypertrophic_cicatrix",
    "object": "epidermolysis_bullosa_pretibial"
  },
  {
    "query": "*Is it true that* there is a relationship that connects cheloid and coli_adenomatous_polyposis?",
    "subject": "cheloid",
    "object": "coli_adenomatous_polyposis"
  },
  {
    "query": "*Is it true that* there is a relationship that connects howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light and bodies_howell_jolly?",
    "subject": "howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light",
    "object": "bodies_howell_jolly"
  },
  {
    "query": "*Is it true that* there is a relationship that connects chromatins and nucleoplasm?",
    "subject": "chromatins",
    "object": "nucleoplasm"
  },
  {
    "query": "*Is it true that* there is a relationship that connects tracheas and oesophagus?",
    "subject": "tracheas",
    "object": "oesophagus"
  },
  {
    "query": "*Is it true that* there is a relationship that connects imaging_study_imp_pt_xxx_nar_radiology and imaging_study?",
    "subject": "imaging_study_imp_pt_xxx_nar_radiology",
    "object": "imaging_study"
  },
  {
    "query": "*Is it true that* there is a relationship that connects sulcal_segment_of_precentral_gyrus and structure_of_precentral_sulcus?",
    "subject": "sulcal_segment_of_precentral_gyrus",
    "object": "structure_of_precentral_sulcus"
  },
  {
    "query": "*Is it true that* there is a relationship that connects lactobacillales and lactic_acid_bacteria_oral_pwd?",
    "subject": "lactobacillales",
    "object": "lactic_acid_bacteria_oral_pwd"
  },
  {
    "query": "*Is it true that* there is a relationship that connects chemotactic_factor_inactivator and inactivator_chemotaxis_function?",
    "subject": "chemotactic_factor_inactivator",
    "object": "inactivator_chemotaxis_function"
  },
  {
    "query": "*Is it true that* there is a relationship that connects ampicillins and ampicillin_ophthalmic_ointment?",
    "subject": "ampicillins",
    "object": "ampicillin_ophthalmic_ointment"
  }
]


In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type12(subject, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
     PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    ASK
    WHERE {{
       umls:{subject} ?predicate umls:{objectf}.



}}
"""
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type12(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0
    for query_data in queries:
        query = query_data['query']
        objectf = query_data['object']
        subject = query_data['subject']

        ground_truth_triples = await query_sparql_type12(subject, objectf)
        print("ground")
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)
        gt_bool = normalize_to_bool(ground_truth_triples)
        print(clean_text)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        print(model_bool)
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)
    return x







In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
async def compute_accuracy_for_query_type12(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        objectf = query_data['object']
        subject = query_data['subject']

        ground_truth_bool = await query_sparql_type12(subject, objectf)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)

        # Ground truth -> bool
        gt_bool = normalize_to_bool(ground_truth_bool)

        # Model -> bool
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "subject": subject,
            "object": objectf,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
async def compute_accuracy_for_query_type12(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        objectf = query_data['object']
        subject = query_data['subject']

        ground_truth_bool = await query_sparql_type12(subject, objectf)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)

        gt_bool = normalize_to_bool(ground_truth_bool)

        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "subject": subject,
            "object": objectf,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
queries_type_6a = [
  {
    "query": "*Is it true that* posterior_mediastinum and t8_segment_of_esophagus are connected in any way?",
    "subject": "posterior_mediastinum",
    "object": "t8_segment_of_esophagus"
  },
  {
    "query": "*Is it true that* right_submandibular_triangle and right_hyoglossus are connected in any way?",
    "subject": "right_submandibular_triangle",
    "object": "right_hyoglossus"
  },
  {
    "query": "*Is it true that* plasma_membranes and cytoplasms are connected in any way?",
    "subject": "plasma_membranes",
    "object": "cytoplasms"
  },
  {
    "query": "*Is it true that* nucleoplasm and nucleolus_cell are connected in any way?",
    "subject": "nucleoplasm",
    "object": "nucleolus_cell"
  },
  {
    "query": "*Is it true that* sioux_county_ne and nebraska_geographic_location are connected in any way?",
    "subject": "sioux_county_ne",
    "object": "nebraska_geographic_location"
  },
  {
    "query": "*Is it true that* dilation_size and dilate are connected in any way?",
    "subject": "dilation_size",
    "object": "dilate"
  },
  {
    "query": "*Is it true that* complications and gastrointestinal_endoscopy are connected in any way?",
    "subject": "complications",
    "object": "gastrointestinal_endoscopy"
  },
  {
    "query": "*Is it true that* absent_left_common_carotid_artery and aortic_arch_structure are connected in any way?",
    "subject": "absent_left_common_carotid_artery",
    "object": "aortic_arch_structure"
  },
  {
    "query": "*Is it true that* nerves_sacral and root_of_uwda_hierarchy are connected in any way?",
    "subject": "nerves_sacral",
    "object": "root_of_uwda_hierarchy"
  },
  {
    "query": "*Is it true that* synaptic_signaling and synapsis are connected in any way?",
    "subject": "synaptic_signaling",
    "object": "synapsis"
  },
  {
    "query": "*Is it true that* maintenance_of_protein_localisation_to_organelle and organelle are connected in any way?",
    "subject": "maintenance_of_protein_localisation_to_organelle",
    "object": "organelle"
  },
  {
    "query": "*Is it true that* type_iii_no_bleeding_of_gastric_ulcer and type_of_bleeding_of_gastric_ulcer are connected in any way?",
    "subject": "type_iii_no_bleeding_of_gastric_ulcer",
    "object": "type_of_bleeding_of_gastric_ulcer"
  },
  {
    "query": "*Is it true that* type_iii_no_bleeding_of_duodenal_ulcer and type_of_bleeding_of_duodenal_ulcer are connected in any way?",
    "subject": "type_iii_no_bleeding_of_duodenal_ulcer",
    "object": "type_of_bleeding_of_duodenal_ulcer"
  },
  {
    "query": "*Is it true that* polypectomy and polypectomy_forceps are connected in any way?",
    "subject": "polypectomy",
    "object": "polypectomy_forceps"
  },
  {
    "query": "*Is it true that* biopsy_nos and biopsy_instruments are connected in any way?",
    "subject": "biopsy_nos",
    "object": "biopsy_instruments"
  },
  {
    "query": "*Is it true that* hypertrophic_cicatrix and epidermolysis_bullosa_pretibial are connected in any way?",
    "subject": "hypertrophic_cicatrix",
    "object": "epidermolysis_bullosa_pretibial"
  },
  {
    "query": "*Is it true that* cheloid and coli_adenomatous_polyposis are connected in any way?",
    "subject": "cheloid",
    "object": "coli_adenomatous_polyposis"
  },
  {
    "query": "*Is it true that* howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light and bodies_howell_jolly are connected in any way?",
    "subject": "howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light",
    "object": "bodies_howell_jolly"
  },
  {
    "query": "*Is it true that* chromatins and nucleoplasm are connected in any way?",
    "subject": "chromatins",
    "object": "nucleoplasm"
  },
  {
    "query": "*Is it true that* tracheas and oesophagus are connected in any way?",
    "subject": "tracheas",
    "object": "oesophagus"
  },
  {
    "query": "*Is it true that* imaging_study_imp_pt_xxx_nar_radiology and imaging_study are connected in any way?",
    "subject": "imaging_study_imp_pt_xxx_nar_radiology",
    "object": "imaging_study"
  },
  {
    "query": "*Is it true that* sulcal_segment_of_precentral_gyrus and structure_of_precentral_sulcus are connected in any way?",
    "subject": "sulcal_segment_of_precentral_gyrus",
    "object": "structure_of_precentral_sulcus"
  },
  {
    "query": "*Is it true that* lactobacillales and lactic_acid_bacteria_oral_pwd are connected in any way?",
    "subject": "lactobacillales",
    "object": "lactic_acid_bacteria_oral_pwd"
  },
  {
    "query": "*Is it true that* chemotactic_factor_inactivator and inactivator_chemotaxis_function are connected in any way?",
    "subject": "chemotactic_factor_inactivator",
    "object": "inactivator_chemotaxis_function"
  },
  {
    "query": "*Is it true that* ampicillins and ampicillin_ophthalmic_ointment are connected in any way?",
    "subject": "ampicillins",
    "object": "ampicillin_ophthalmic_ointment"
  }
]


In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type13(subject, objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
ASK
WHERE {{
    {{ umls:{subject} ?predicate umls:{objectf}  . }}
    UNION
    {{
         umls:{objectf} ?predicate umls:{subject}.
    }}
}}
"""
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result

    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type13(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    x = False
    total = 0
    correct = 0
    for query_data in queries:
        query = query_data['query']
        objectf = query_data['object']
        subject = query_data['subject']

        ground_truth_triples = await query_sparql_type13(subject, objectf)
        print("ground")
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)


        print(model_response_triples)
        gt_bool = normalize_to_bool(ground_truth_triples)

        print(clean_text)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        print("model bool")
        print(model_bool)
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)
    return x







In [ ]:
async def compute_accuracy_for_query_type13(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        objectf = query_data['object']
        subject = query_data['subject']

        ground_truth_bool = await query_sparql_type13(subject, objectf)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)
        gt_bool = normalize_to_bool(ground_truth_bool)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "subject": subject,
            "object": objectf,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type14(subject):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
ASK
WHERE {{
    umls:{subject} ?predicate ?object  .


}}
"""
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type14(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}


    x = False
    total = 0
    correct = 0
    for query_data in queries:
        query = query_data['query']

        subject = query_data['subject']

        ground_truth_triples = await query_sparql_type14(subject)
        print("ground")
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)
        gt_bool = normalize_to_bool(ground_truth_triples)
        print(clean_text)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        print(model_bool)
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)
    return x







In [ ]:
async def compute_accuracy_for_query_type14(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']

        ground_truth_bool = await query_sparql_type14(subject)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)

        gt_bool = normalize_to_bool(ground_truth_bool)

        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "subject": subject,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
queries_type_7a = [
  {
    "query": "*Is it true that* there are statements about posterior_mediastinum?",
    "subject": "posterior_mediastinum"
  },
  {
    "query": "*Is it true that* there are statements about right_submandibular_triangle?",
    "subject": "right_submandibular_triangle"
  },
  {
    "query": "*Is it true that* there are statements about plasma_membranes?",
    "subject": "plasma_membranes"
  },
  {
    "query": "*Is it true that* there are statements about nucleoplasm?",
    "subject": "nucleoplasm"
  },
  {
    "query": "*Is it true that* there are statements about sioux_county_ne?",
    "subject": "sioux_county_ne"
  },
  {
    "query": "*Is it true that* there are statements about dilation_size?",
    "subject": "dilation_size"
  },
  {
    "query": "*Is it true that* there are statements about complications?",
    "subject": "complications"
  },
  {
    "query": "*Is it true that* there are statements about absent_left_common_carotid_artery?",
    "subject": "absent_left_common_carotid_artery"
  },
  {
    "query": "*Is it true that* there are statements about nerves_sacral?",
    "subject": "nerves_sacral"
  },
  {
    "query": "*Is it true that* there are statements about synaptic_signaling?",
    "subject": "synaptic_signaling"
  },
  {
    "query": "*Is it true that* there are statements about maintenance_of_protein_localisation_to_organelle?",
    "subject": "maintenance_of_protein_localisation_to_organelle"
  },
  {
    "query": "*Is it true that* there are statements about type_iii_no_bleeding_of_gastric_ulcer?",
    "subject": "type_iii_no_bleeding_of_gastric_ulcer"
  },
  {
    "query": "*Is it true that* there are statements about type_iii_no_bleeding_of_duodenal_ulcer?",
    "subject": "type_iii_no_bleeding_of_duodenal_ulcer"
  },
  {
    "query": "*Is it true that* there are statements about polypectomy?",
    "subject": "polypectomy"
  },
  {
    "query": "*Is it true that* there are statements about biopsy_nos?",
    "subject": "biopsy_nos"
  },
  {
    "query": "*Is it true that* there are statements about hypertrophic_cicatrix?",
    "subject": "hypertrophic_cicatrix"
  },
  {
    "query": "*Is it true that* there are statements about cheloid?",
    "subject": "cheloid"
  },
  {
    "query": "*Is it true that* there are statements about howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light?",
    "subject": "howell_jolly_bodies_prthr_pt_bld_ord_microscopy_light"
  },
  {
    "query": "*Is it true that* there are statements about chromatins?",
    "subject": "chromatins"
  },
  {
    "query": "*Is it true that* there are statements about tracheas?",
    "subject": "tracheas"
  },
  {
    "query": "*Is it true that* there are statements about imaging_study_imp_pt_xxx_nar_radiology?",
    "subject": "imaging_study_imp_pt_xxx_nar_radiology"
  },
  {
    "query": "*Is it true that* there are statements about sulcal_segment_of_precentral_gyrus?",
    "subject": "sulcal_segment_of_precentral_gyrus"
  },
  {
    "query": "*Is it true that* there are statements about lactobacillales?",
    "subject": "lactobacillales"
  },
  {
    "query": "*Is it true that* there are statements about chemotactic_factor_inactivator?",
    "subject": "chemotactic_factor_inactivator"
  },
  {
    "query": "*Is it true that* there are statements about ampicillins?",
    "subject": "ampicillins"
  }
]


In [ ]:
queries_type_8a = [
  {
    "query": "*Is it true that* there are statements where t8_segment_of_esophagus is the object?",
    "object": "t8_segment_of_esophagus"
  },
  {
    "query": "*Is it true that* there are statements where right_hyoglossus is the object?",
    "object": "right_hyoglossus"
  },
  {
    "query": "*Is it true that* there are statements where cytoplasms is the object?",
    "object": "cytoplasms"
  },
  {
    "query": "*Is it true that* there are statements where nucleolus_cell is the object?",
    "object": "nucleolus_cell"
  },
  {
    "query": "*Is it true that* there are statements where nebraska_geographic_location is the object?",
    "object": "nebraska_geographic_location"
  },
  {
    "query": "*Is it true that* there are statements where dilate is the object?",
    "object": "dilate"
  },
  {
    "query": "*Is it true that* there are statements where gastrointestinal_endoscopy is the object?",
    "object": "gastrointestinal_endoscopy"
  },
  {
    "query": "*Is it true that* there are statements where aortic_arch_structure is the object?",
    "object": "aortic_arch_structure"
  },
  {
    "query": "*Is it true that* there are statements where root_of_uwda_hierarchy is the object?",
    "object": "root_of_uwda_hierarchy"
  },
  {
    "query": "*Is it true that* there are statements where synapsis is the object?",
    "object": "synapsis"
  },
  {
    "query": "*Is it true that* there are statements where organelle is the object?",
    "object": "organelle"
  },
  {
    "query": "*Is it true that* there are statements where type_of_bleeding_of_gastric_ulcer is the object?",
    "object": "type_of_bleeding_of_gastric_ulcer"
  },
  {
    "query": "*Is it true that* there are statements where type_of_bleeding_of_duodenal_ulcer is the object?",
    "object": "type_of_bleeding_of_duodenal_ulcer"
  },
  {
    "query": "*Is it true that* there are statements where polypectomy_forceps is the object?",
    "object": "polypectomy_forceps"
  },
  {
    "query": "*Is it true that* there are statements where biopsy_instruments is the object?",
    "object": "biopsy_instruments"
  },
  {
    "query": "*Is it true that* there are statements where epidermolysis_bullosa_pretibial is the object?",
    "object": "epidermolysis_bullosa_pretibial"
  },
  {
    "query": "*Is it true that* there are statements where coli_adenomatous_polyposis is the object?",
    "object": "coli_adenomatous_polyposis"
  },
  {
    "query": "*Is it true that* there are statements where bodies_howell_jolly is the object?",
    "object": "bodies_howell_jolly"
  },
  {
    "query": "*Is it true that* there are statements where nucleoplasm is the object?",
    "object": "nucleoplasm"
  },
  {
    "query": "*Is it true that* there are statements where oesophagus is the object?",
    "object": "oesophagus"
  },
  {
    "query": "*Is it true that* there are statements where imaging_study is the object?",
    "object": "imaging_study"
  },
  {
    "query": "*Is it true that* there are statements where structure_of_precentral_sulcus is the object?",
    "object": "structure_of_precentral_sulcus"
  },
  {
    "query": "*Is it true that* there are statements where lactic_acid_bacteria_oral_pwd is the object?",
    "object": "lactic_acid_bacteria_oral_pwd"
  },
  {
    "query": "*Is it true that* there are statements where inactivator_chemotaxis_function is the object?",
    "object": "inactivator_chemotaxis_function"
  },
  {
    "query": "*Is it true that* there are statements where ampicillin_ophthalmic_ointment is the object?",
    "object": "ampicillin_ophthalmic_ointment"
  }
]


In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type15(objectf):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
ASK
WHERE {{
    ?subject ?predicate umls:{objectf} .


}}
"""
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type15(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0


    for query_data in queries:
        query = query_data['query']

        objectf = query_data['object']

        ground_truth_triples = await query_sparql_type15(objectf)
        print("ground")
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)
        gt_bool = normalize_to_bool(ground_truth_triples)
        print(clean_text)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        print(model_bool)
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)
    return x




In [ ]:
async def compute_accuracy_for_query_type15(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        objectf = query_data['object']

        ground_truth_bool = await query_sparql_type15(objectf)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)

        gt_bool = normalize_to_bool(ground_truth_bool)

        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "object": objectf,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type16(predicate):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    ASK
    WHERE {{
        ?subject umls:{predicate} ?object .

    }}
"""
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print("ret is")
        print(ret)
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type16(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0
    for query_data in queries:
        query = query_data['query']

        predicate = query_data['predicate']

        ground_truth_triples = await query_sparql_type16(predicate)
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)
        gt_bool = normalize_to_bool(ground_truth_triples)
        print(clean_text)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        print(model_bool)
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)
    return x






In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type16(predicate):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
ASK
WHERE {{
    ?subject umls:{predicate} ?object .


}}
"""
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print("ret is")
        print(ret)
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type16(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0
    for query_data in queries:
        query = query_data['query']

        predicate = query_data['predicate']

        ground_truth_triples = await query_sparql_type16(predicate)
        print("ground")
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)
        gt_bool = normalize_to_bool(ground_truth_triples)
        print(clean_text)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        print(model_bool)
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)
    return x






In [ ]:
queries_type_9a = [
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "contains"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "contains"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "surrounds"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "surrounds"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "conceptual_part_of"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "conceptual_part_of"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "result_of"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "branch_of"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "branch_of"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "occurs_in"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "occurs_in"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "degree_of"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "degree_of"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "uses"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "uses"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "manifestation_of"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "manifestation_of"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "measures"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "part_of"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "adjacent_to"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "associated_with"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "adjacent_to"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "ingredient_of"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "exhibits"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "ingredient_of"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "uses"
  },
  {
    "query": "*Is it true that* there is a predicate that connects 2 entities?",
    "predicate": "uses"
  }
]


In [ ]:
#1bun
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type17(subject):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
ASK WHERE {{
 ?sub ?prop ?val.

FILTER (umls:{subject} = ?sub || umls:{subject} = ?prop || umls:{subject} = ?val)



}}
"""
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print("ret is")
        print(ret)
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result

    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type17(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0
    for query_data in queries:
        query = query_data['query']


        subject = query_data['subject']

        ground_truth_triples = await query_sparql_type17(subject)
        print("ground")
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print(response_text)
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)
        gt_bool = normalize_to_bool(ground_truth_triples)
        print(clean_text)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        print(model_bool)
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)
    return x


In [ ]:
async def compute_accuracy_for_query_type16(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        predicate = query_data['predicate']

        ground_truth_bool = await query_sparql_type16(predicate)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)
        gt_bool = normalize_to_bool(ground_truth_bool)

        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "predicate": predicate,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
queries_type_9a = [
    {"query": "*Is it true that* predicate called location_of connects 2 entities?*", "predicate": "location_of"},
    {"query": "*Is it true that* predicate called manifestation_of connects 2 entities?*", "predicate": "manifestation_of"},
    {"query": "*Is it true that* predicate called part_of connects 2 entities?*", "predicate": "part_of"},
    {"query": "*Is it true that* predicate called associated_with connects 2 entities?*", "predicate": "associated_with"},
    {"query": "*Is it true that* predicate called evaluation_of connects 2 entities?*", "predicate": "evaluation_of"},
    {"query": "*Is it true that* predicate called method_of connects 2 entities?*", "predicate": "method_of"},
    {"query": "*Is it true that* predicate called ingredient_of connects 2 entities?*", "predicate": "ingredient_of"},
    {"query": "*Is it true that* predicate called branch_of connects 2 entities?*", "predicate": "branch_of"},
    {"query": "*Is it true that* predicate called occurs_in connects 2 entities?*", "predicate": "occurs_in"},
    {"query": "*Is it true that* predicate called exhibits connects 2 entities?*", "predicate": "exhibits"},
    {"query": "*Is it true that* predicate called adjacent_to connects 2 entities?*", "predicate": "adjacent_to"},
    {"query": "*Is it true that* predicate called connected_to connects 2 entities?*", "predicate": "connected_to"},
    {"query": "*Is it true that* predicate called contains connects 2 entities?*", "predicate": "contains"},
    {"query": "*Is it true that* predicate called surrounds connects 2 entities?*", "predicate": "surrounds"},
    {"query": "*Is it true that* predicate called tributary_of connects 2 entities?*", "predicate": "tributary_of"},
    {"query": "*Is it true that* predicate called analyzes connects 2 entities?*", "predicate": "analyzes"},
    {"query": "*Is it true that* predicate called measures connects 2 entities?*", "predicate": "measures"},
    {"query": "*Is it true that* predicate called consists_of connects 2 entities?*", "predicate": "consists_of"},
    {"query": "*Is it true that* predicate called diagnoses connects 2 entities?*", "predicate": "diagnoses"},
    {"query": "*Is it true that* predicate called uses connects 2 entities?*", "predicate": "uses"},
    {"query": "*Is it true that* predicate called degree_of connects 2 entities?*", "predicate": "degree_of"},
    {"query": "*Is it true that* predicate called conceptual_part_of connects 2 entities?*", "predicate": "conceptual_part_of"},
    {"query": "*Is it true that* predicate called result_of connects 2 entities?*", "predicate": "result_of"},
    {"query": "*Is it true that* predicate called treats connects 2 entities?*", "predicate": "treats"},
        {"query": "*Is it true that* predicate called connected_to connects 2 entities?*", "predicate": "connected_to"}
]

In [ ]:
import asyncio
from SPARQLWrapper import SPARQLWrapper, JSON
import numpy as np


async def query_sparql_type17(subject):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
ASK WHERE {{
 ?sub ?prop ?val.

FILTER (umls:{subject} = ?sub || umls:{subject} = ?prop || umls:{subject} = ?val)



}}
"""
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print("ret is")
        print(ret)
        ask_result = ret.get("boolean", False)
        print("ASK result:", ask_result)
        return ask_result

    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type17(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}
    x = False
    total = 0
    correct = 0
    for query_data in queries:
        query = query_data['query']


        subject = query_data['subject']

        ground_truth_triples = await query_sparql_type17(subject)
        print("ground")
        print(ground_truth_triples)

        print(f"\nQuery: {query}:")
        print(f"\nQuery Type: {query_name}")
        print(f"\Model: {model_name}")
        print(f"\Prompt level: {prompt_name}")
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print("llm respinse is")
        print(response_text)
        clean_text = strip_prefixes(response_text)
        model_response_triples = extract_triplesf(clean_text)
        print(model_response_triples)
        gt_bool = normalize_to_bool(ground_truth_triples)
        print(clean_text)
        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))
        print("model bool")
        print(model_bool)
        x = gt_bool == model_bool
        total += 1
        if x:
          correct += 1

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", x)
        print("--------")

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)
    accuracy = correct / total if total else 0
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", accuracy)
    return x







In [ ]:
async def compute_accuracy_for_query_type17(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']

        ground_truth_bool = await query_sparql_type17(subject)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)

        gt_bool = normalize_to_bool(ground_truth_bool)

        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "subject": subject,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
async def compute_accuracy_for_query_type17(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']

        ground_truth_bool = await query_sparql_type17(subject)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)

        gt_bool = normalize_to_bool(ground_truth_bool)

        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "subject": subject,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
queries_type_10a = [
  {
    "query": "*Is it true that* there is posterior_mediastinum mentioned in the graph?",
    "subject": "posterior_mediastinum"
  },
  {
    "query": "*Is it true that* there is t8_segment_of_esophagus mentioned in the graph?",
    "subject": "t8_segment_of_esophagus"
  },
  {
    "query": "*Is it true that* there is right_submandibular_triangle mentioned in the graph?",
    "subject": "right_submandibular_triangle"
  },
  {
    "query": "*Is it true that* there is right_hyoglossus mentioned in the graph?",
    "subject": "right_hyoglossus"
  },
  {
    "query": "*Is it true that* there is plasma_membranes mentioned in the graph?",
    "subject": "plasma_membranes"
  },
  {
    "query": "*Is it true that* there is cytoplasms mentioned in the graph?",
    "subject": "cytoplasms"
  },
  {
    "query": "*Is it true that* there is nucleolus_cell mentioned in the graph?",
    "subject": "nucleolus_cell"
  },
  {
    "query": "*Is it true that* there is sioux_county_ne mentioned in the graph?",
    "subject": "sioux_county_ne"
  },
  {
    "query": "*Is it true that* there is nebraska_geographic_location mentioned in the graph?",
    "subject": "nebraska_geographic_location"
  },
  {
    "query": "*Is it true that* there is dilation_size mentioned in the graph?",
    "subject": "dilation_size"
  },
  {
    "query": "*Is it true that* there is gastrointestinal_endoscopy mentioned in the graph?",
    "subject": "gastrointestinal_endoscopy"
  },
  {
    "query": "*Is it true that* there is aortic_arch_structure mentioned in the graph?",
    "subject": "aortic_arch_structure"
  },
  {
    "query": "*Is it true that* there is synapsis mentioned in the graph?",
    "subject": "synapsis"
  },
  {
    "query": "*Is it true that* there is organelle mentioned in the graph?",
    "subject": "organelle"
  },
  {
    "query": "*Is it true that* there is biopsy_nos mentioned in the graph?",
    "subject": "biopsy_nos"
  },
  {
    "query": "*Is it true that* there is polypectomy_forceps mentioned in the graph?",
    "subject": "polypectomy_forceps"
  },
  {
    "query": "*Is it true that* there is epidermolysis_bullosa_pretibial mentioned in the graph?",
    "subject": "epidermolysis_bullosa_pretibial"
  },
  {
    "query": "*Is it true that* there is imaging_study mentioned in the graph?",
    "subject": "imaging_study"
  },
  {
    "query": "*Is it true that* there is lactic_acid_bacteria_oral_pwd mentioned in the graph?",
    "subject": "lactic_acid_bacteria_oral_pwd"
  },

  {
    "query": "*Is it true that* there is contains mentioned in the graph?",
    "subject": "contains"
  },
  {
    "query": "*Is it true that* there is surrounds mentioned in the graph?",
    "subject": "surrounds"
  },
  {
    "query": "*Is it true that* there is uses mentioned in the graph?",
    "subject": "uses"
  },
  {
    "query": "*Is it true that* there is manifestation_of mentioned in the graph?",
    "subject": "manifestation_of"
  },
  {
    "query": "*Is it true that* there is adjacent_to mentioned in the graph?",
    "subject": "adjacent_to"
  },
  {
    "query": "*Is it true that* there is ingredient_of mentioned in the graph?",
    "subject": "ingredient_of"
  }
]


In [ ]:
async def query_sparql_type7_reasoning(predicate, tipsubiect, tipobiect):
    """Queries the SPARQL endpoint with a dynamic subject and predicate and extracts triples."""
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setReturnFormat(JSON)

    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX umls: <http://www.nlm.nih.gov/research/umls>
    SELECT DISTINCT ?subject ?predicate ?object
    WHERE {{
      ?subject umls:{predicate} ?object .
      ?subject rdf:type/rdfs:subClassOf* umls:{tipsubiect} .
      ?object rdf:type/rdfs:subClassOf* umls:{tipobiect}.
      BIND(umls:{predicate} AS ?predicate)
      }}
    """
    sparql.setQuery(query)

    try:
        ret = sparql.queryAndConvert()
        print("ret is")
        print(ret)
        print()
        extr = extract_triples(ret)
        print("extr")
        print(extr)
        print("final sp")
        print(extract_triplessparql(extr))
        return extract_triplessparql(extr)
    except Exception as e:
        print(f"Error querying SPARQL: {e}")
        return []

async def compute_accuracy_for_query_type7_reasoning(queries, prompt_level, local_search,  query_name, model_name, prompt_name):
    """
    """
    accuracy_results = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': []}
    response_results = {'results': []}

    for query_data in queries:
        query = query_data['query']
        llm = query_data['llm']
        tipsubiect = query_data['tipsubiect']
        tipobiect = query_data['tipobiect']

        predicate = query_data['predicate']


        ground_truth_triples = await query_sparql_type7_reasoning(predicate, tipsubiect, tipobiect)

        print(query)
        response = await local_search.asearch(query=query, search_prompt=prompt_level)
        response_text = response.response
        print("resp etxt")
        print(response_text)


        clean_text1 = strip_prefixes(response_text)

        print("strip1")
        print(clean_text1)
        print("extr trip1")
        model_response_triples1 = extract_triplesf7(clean_text1)
        print("model triple is:")
        print(model_response_triples1)

        clean_triples = await search_by_subjects_location_of_clean_triplesfinal(
            triple_strings=model_response_triples1,
            local_search=local_search,
            prompt_level=prompt_level,
            predicate = predicate
        )

        print(clean_triples)


        response_text2 = await infer_rdfs_triples_simple_modifiedr(query=llm, response_text=clean_triples)


        print(response_text2)
        clean_text = strip_prefixes(response_text2)


        print(clean_text)

        model_response_triples = extract_triplesf7(clean_text)

        print(model_response_triples)

        common, extra, precision, recall, f1_score = find_common_and_extra_elements_subjectobj_any_order(ground_truth_triples, model_response_triples)

        accuracy_results['precision'].append(precision)
        accuracy_results['recall'].append(recall)
        accuracy_results['f1_score'].append(f1_score)
        response_results['results'].append(response_text)


        print("Common Triples:", common)
        print("Extra Triples:", extra)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1_score:.2f}")
        print(f"Response: {response_text}\n")


    mean_metrics = {
        'precision': np.mean(accuracy_results['precision']) if accuracy_results['precision'] else 0,
        'recall': np.mean(accuracy_results['recall']) if accuracy_results['recall'] else 0,
        'f1_score': np.mean(accuracy_results['f1_score']) if accuracy_results['f1_score'] else 0,
    }


    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")
    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)


    output_data = {
          "query": query,
          "query_name": query_name,
          "model_name": model_name,
          "prompt_name": prompt_name,
          "benchmark_triples": list(ground_truth_triples),
          "llm_triples": list(model_response_triples),
          "common_triples": common,
          "extra_triples": extra,
          "precision": precision,
          "recall": recall,
          "f1_score": f1_score,
          "accuracy_results": accuracy_results,
          "mean_metrics": mean_metrics,
          "responses": response_results
      }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    return accuracy_results, mean_metrics, response_results


    return accuracy_results, mean_metrics, response_results


In [ ]:
async def compute_accuracy_for_query_type17(
    queries, prompt_level, local_search,
    query_name, model_name, prompt_name
):
    accuracy_results = {'accuracy': [], 'gt_bool': [], 'model_bool': []}
    response_results = {'results': []}
    all_query_results = []

    total = 0
    correct = 0

    for query_data in queries:
        query = query_data['query']
        subject = query_data['subject']

        ground_truth_bool = await query_sparql_type17(subject)

        print("\nQuery:", query)
        print("Query Type:", query_name)
        print("Model:", model_name)
        print("Prompt level:", prompt_name)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        response_text = response.response
        response_results['results'].append(response_text)

        clean_text = strip_prefixes(response_text)

        gt_bool = normalize_to_bool(ground_truth_bool)

        model_bool = normalize_to_bool(extract_bool_from_llm(clean_text))

        match = (gt_bool == model_bool)

        total += 1
        if match:
            correct += 1

        accuracy_results['gt_bool'].append(gt_bool)
        accuracy_results['model_bool'].append(model_bool)
        accuracy_results['accuracy'].append(1 if match else 0)

        all_query_results.append({
            "query": query,
            "subject": subject,
            "ground_truth_bool": gt_bool,
            "model_bool": model_bool,
            "match": match,
            "response_text": response_text,
            "clean_text": clean_text
        })

        print("GT bool:", gt_bool)
        print("Model bool:", model_bool)
        print("MATCH:", match)
        print("--------")

    final_accuracy = correct / total if total else 0

    mean_metrics = {
        "accuracy": float(final_accuracy),
        "total": total,
        "correct": correct
    }

    safe_query_name = query_name.replace(" ", "_")
    safe_model_name = model_name.replace(" ", "_")
    safe_prompt_name = prompt_name.replace(" ", "_")

    filename = f"query{safe_query_name}-model{safe_model_name}-prompt{safe_prompt_name}.json"
    filepath = os.path.join(output_folder, filename)

    output_data = {
        "query_name": query_name,
        "model_name": model_name,
        "prompt_name": prompt_name,
        "all_queries": all_query_results,
        "accuracy_results": accuracy_results,
        "mean_metrics": mean_metrics,
        "responses": response_results
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print(f"Results saved to {filepath}")
    print("TOTAL:", total)
    print("CORRECT:", correct)
    print("ACCURACY:", final_accuracy)

    return accuracy_results, mean_metrics, response_results

In [ ]:
async def search_by_subjects_location_of_clean_triplesfinal(
    triple_strings,
    local_search,
    prompt_level,
    predicate
):
    EXCLUDED = {"type", "subClassOf", predicate}

    subjects = set()
    triples_with_predicate = []


    for s in triple_strings:
        if not s.strip():
            continue
        if predicate in s.split():
            triples_with_predicate.append(s)


    source_triples = (
        triples_with_predicate
        if triples_with_predicate
        else triple_strings
    )


    for s in source_triples:
        for token in s.split():
            if token in EXCLUDED:
                continue
            subjects.add(token)

    print("Subiecte unice (regulă finală):")
    print(subjects)

    all_triples = []


    for subject in subjects:
        query = f"{subject} {predicate} ?"
        print("query:", query)

        response = await local_search.asearch(
            query=query,
            search_prompt=prompt_level
        )

        clean_text = strip_prefixes(response.response)
        triples = extract_triplesfsl(clean_text)
        all_triples.extend(triples)

    return list(set(all_triples))


In [ ]:
queries_type_7_r = [

  {
    "query": "entities connected by diagnoses relationship?",
    "predicate": "diagnoses",
    "tipsubiect": "health_care_activity",
    "tipobiect": "disease_or_syndrome",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type health_care_activity that diagnoses instances of type disease_or_syndrome ?"
  },
  {
    "query": "entities connected by diagnoses relationship?",
    "predicate": "diagnoses",
    "tipsubiect": "diagnostic_procedure",
    "tipobiect": "biologic_function",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type diagnostic_procedure that diagnoses instances of type biologic_function ?"
  },
  {
    "query": "entities connected by adjacent_to relationship?",
    "predicate": "adjacent_to",
    "tipsubiect": "anatomical_structure",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type anatomical_structure that adjacent_to instances of type physical_object ?"
  },
  {
    "query": "entities connected by adjacent_to relationship?",
    "predicate": "adjacent_to",
    "tipsubiect": "body_part_organ_or_organ_component",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type body_part_organ_or_organ_component that adjacent_to instances of type physical_object ?"
  },
  {
    "query": "entities connected by treats relationship?",
    "predicate": "treats",
    "tipsubiect": "health_care_activity",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type health_care_activity that treats instances of type physical_object ?"
  },
  {
    "query": "entities connected by treats relationship?",
    "predicate": "treats",
    "tipsubiect": "occupational_activity",
    "tipobiect": "anatomical_structure",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type occupational_activity that treats instances of type anatomical_structure ?"
  },
  {
    "query": "entities connected by ingredient_of relationship?",
    "predicate": "ingredient_of",
    "tipsubiect": "enzyme",
    "tipobiect": "substance",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type enzyme that ingredient_of instances of type substance ?"
  },
  {
    "query": "entities connected by ingredient_of relationship?",
    "predicate": "ingredient_of",
    "tipsubiect": "food",
    "tipobiect": "substance",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type food that ingredient_of instances of type substance ?"
  },
  {
    "query": "entities connected by ingredient_of relationship?",
    "predicate": "ingredient_of",
    "tipsubiect": "physical_object",
    "tipobiect": "inorganic_chemical",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type physical_object that ingredient_of instances of type inorganic_chemical ?"
  },
  {
    "query": "entities connected by occurs_in relationship?",
    "predicate": "occurs_in",
    "tipsubiect": "cell_component",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type cell_component that occurs_in instances of type physical_object ?"
  },
  {
    "query": "entities connected by occurs_in relationship?",
    "predicate": "occurs_in",
    "tipsubiect": "genetic_function",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type genetic_function that occurs_in instances of type physical_object ?"
  },
  {
    "query": "entities connected by affects relationship?",
    "predicate": "affects",
    "tipsubiect": "health_care_activity",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type health_care_activity that affects instances of type physical_object ?"
  },
  {
    "query": "entities connected by affects relationship?",
    "predicate": "affects",
    "tipsubiect": "occupational_activity",
    "tipobiect": "anatomical_structure",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type occupational_activity that affects instances of type anatomical_structure ?"
  },
  {
    "query": "entities connected by location_of relationship?",
    "predicate": "location_of",
    "tipsubiect": "fully_formed_anatomical_structure",
    "tipobiect": "activity",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type fully_formed_anatomical_structure that location_of instances of type activity ?"
  },
  {
    "query": "entities connected by location_of relationship?",
    "predicate": "location_of",
    "tipsubiect": "body_location_or_region",
    "tipobiect": "event",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type body_location_or_region that location_of instances of type event ?"
  },
  {
    "query": "entities connected by method_of relationship?",
    "predicate": "method_of",
    "tipsubiect": "idea_or_concept",
    "tipobiect": "organism_attribute",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type idea_or_concept that method_of instances of type organism_attribute ?"
  },
  {
    "query": "entities connected by method_of relationship?",
    "predicate": "method_of",
    "tipsubiect": "occupational_activity",
    "tipobiect": "clinical_attribute",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type occupational_activity that method_of instances of type clinical_attribute ?"
  },
  {
    "query": "entities connected by evaluation_of relationship?",
    "predicate": "evaluation_of",
    "tipsubiect": "organism_attribute",
    "tipobiect": "pathologic_function",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type organism_attribute that evaluation_of instances of type pathologic_function ?"
  },
  {
    "query": "entities connected by evaluation_of relationship?",
    "predicate": "evaluation_of",
    "tipsubiect": "qualitative_concept",
    "tipobiect": "event",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type qualitative_concept that evaluation_of instances of type event ?"
  },
  {
    "query": "entities connected by connected_to relationship?",
    "predicate": "connected_to",
    "tipsubiect": "cell_component",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type cell_component that connected_to instances of type physical_object ?"
  },
  {
    "query": "entities connected by connected_to relationship?",
    "predicate": "connected_to",
    "tipsubiect": "body_location_or_region",
    "tipobiect": "body_space_or_junction",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type body_location_or_region that connected_to instances of type body_space_or_junction ?"
  },
  {
    "query": "entities connected by tributary_of relationship?",
    "predicate": "tributary_of",
    "tipsubiect": "body_space_or_junction",
    "tipobiect": "physical_object",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type body_space_or_junction that tributary_of instances of type physical_object ?"
  },
  {
    "query": "entities connected by tributary_of relationship?",
    "predicate": "tributary_of",
    "tipsubiect": "physical_object",
    "tipobiect": "conceptual_entity",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type physical_object that tributary_of instances of type conceptual_entity ?"
  },
  {
    "query": "entities connected by branch_of relationship?",
    "predicate": "branch_of",
    "tipsubiect": "physical_object",
    "tipobiect": "body_system",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type physical_object that branch_of instances of type body_system ?"
  },
  {
    "query": "entities connected by branch_of relationship?",
    "predicate": "branch_of",
    "tipsubiect": "body_location_or_region",
    "tipobiect": "Resource",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type body_location_or_region that branch_of instances of type Resource ?"
  },
    {
    "query": "entities connected by location_of relationship?",
    "predicate": "location_of",
    "tipsubiect": "anatomical_structure",
    "tipobiect": "activity",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type anatomical_structure that location_of instances of type activity ?"
  },
   {
    "query": "entities connected by location_of relationship?",
    "predicate": "location_of",
    "tipsubiect": "physical_object",
    "tipobiect": "health_care_activity",
    "llm": "from abox that you have, keep the instances as they are but return only instances of type physical_object that location_of instances of type health_care_activity ?"
  }

]
